# VRFSZ–GBM — Master Reproducibility Notebook

**Canonical code file for the GitHub repository.**

This notebook consolidates the final scientific workflow used for the manuscript:

1. acquire/audit paired DeltaDTM terrain inputs;
2. build the exact-grid terrain comparison and VRFSZ classes;
3. acquire Sentinel-1/JRC inputs;
4. audit acquisition readiness;
5. pilot and freeze the SAR flood classifier;
6. map all four retained events;
7. apply the corrected EVENT003 baseline branch;
8. run independent coarse GFDS corroboration;
9. rebuild corrected recurrence, relative-risk statistics, spatial bootstrap,
   robustness analysis, figures, provenance records, and final hashes.

## Final corrected scientific state

- Corrected EVENT003 flood fraction ≈ **0.205340**
- T=0.5 m: RR ≈ **1.0031**, 95% CI **0.9025–1.0904**
- T=1.0 m: RR ≈ **0.9681**, 95% CI **0.8702–1.0610**
- T=2.0 m: RR ≈ **0.9003**, 95% CI **0.7769–1.0383**
- Final interpretation: **NO SUPPORT** for the predeclared positive VRFSZ–SAR flood-detection association.

`NO SUPPORT` does not mean proof of no association and is not interpreted as a causal negative effect.

## Claim boundaries

- The DeltaDTM analysis is a **paired-product comparison** unless official native-mirror cell-value equivalence is independently demonstrated.
- GFDS is retained only as **coarse independent passive-microwave corroboration**, not 10 m ground truth.
- GloFAS, CYGNSS, gauge, and optical branches that did not yield analysis-ready event-matched references are not promoted to final quantitative validation.

## Execution rules

This is a reproducibility master notebook, not a one-click cloud pipeline. Acquisition
stages may require Google Earth Engine or provider authentication.

For a fresh reconstruction, run stages in order. If data already exist, skip acquisition
stages only after the corresponding readiness gates pass.

Do not substitute older EVENT003, recurrence, bootstrap, or robustness outputs after the
corrected branch has been run.

# STAGE 1 — Core terrain acquisition and source-pair audit

**Source provenance:** `01B_Audit_and_Acquire_Missing_Core_Terrain_Data.ipynb`.

Downloads/audits only the mandatory terrain inputs and preserves provider provenance.

# 01B — Audit and Acquire Missing Core Terrain Data

This notebook repairs the project chronology **without changing the scientific design**.

The Sentinel-1 branch has already been completed. This notebook therefore does **not** redownload or
retune Sentinel-1. It audits what already exists and acquires only the missing Phase-A terrain data.

---

# Correct core dataset design

The final terrain experiment must compare the **same DeltaDTM version** under two vertical references:

\[
Z_N = \text{DeltaDTM v1.1 referenced to EGM2008}
\]

\[
Z_M = \text{DeltaDTM v1.1 referenced to mean sea level via MDT HYBRID-CNES-CLS2022}
\]

This is cleaner than mixing native DeltaDTM v1.0 with MSL DeltaDTM v1.1.

## Authoritative sources

### Native DeltaDTM v1.1

- Pronk, *DeltaDTM v1.1: A global coastal digital terrain model (Version 4)*
- 4TU.ResearchData
- DOI: `10.4121/21997565.v4`

The native DeltaDTM family is geoid-referenced to EGM2008.

### MSL-referenced DeltaDTM v1.1

- Seeger & Minderhoud
- *DeltaDTM v1.1 referenced to mean sea level (MDT HYBRID-CNES-CLS2022)*
- WUR DOI: `10.17887/wur01-gx7s1b`

Access guide:

- Zenodo record: `10.5281/zenodo.18468579`
- includes `DeltaDTM_v1.1.csv`
- includes `DeltaDTM_v1.1_guide.pdf`

---

# What this project does NOT need

Because we use the authors' ready-to-use MSL-referenced DeltaDTM, we do **not** separately reconstruct
the conversion from:

- EGM2008;
- GOCO06s;
- raw MDT;
- coastline extrapolation.

That would unnecessarily reproduce another study's datum-conversion workflow and introduce additional
failure modes.

The ready-made MSL product already embodies that processing.

---

# Strict chronology after this notebook

```text
01B  acquire native + MSL DeltaDTM v1.1 pair
 ↓
02   ingest / mosaic the GBM terrain pair
 ↓
03   paired grid + vertical-reference QA
 ↓
04   compute ΔZ = Z_N - Z_M
 ↓
05   primary T = 1 m VRFSZ classification
 ↓
18A3 terrain/statistical readiness
 ↓
18B→21 combined final statistical analysis
```

Do not proceed to 18B until this chain passes.

# 1 — User controls

Usually only `GEE_PROJECT` may need editing. The terrain download itself does not use Earth Engine.

Large archive download is intentionally protected. The notebook prefers continent/tile-specific files.

In [ ]:
from pathlib import Path
import io
import json
import math
import os
import re
import shutil
import sys
import time
import zipfile
from urllib.parse import urlparse, unquote

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
from shapely.geometry import box
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Project controls
# ------------------------------------------------------------
AOI_RELATIVE = "data/raw/boundaries/gbm_delta.shp"

# Existing project folders expected by Phase A.
NATIVE_DIR_REL = "data/raw/dem/native_tiles"
MSL_DIR_REL = "data/raw/dem/msl_tiles"
SOURCE_META_REL = "data/raw/dem/source_metadata"

# 4TU native source
FOURTU_SEARCH_URL = "https://data.4tu.nl/v2/articles/search"
FOURTU_BASE = "https://data.4tu.nl"
NATIVE_DOI_STEM = "10.4121/21997565"
PREFERRED_NATIVE_VERSION = "v4"

# WUR / Zenodo MSL access guide
ZENODO_RECORD_ID = "18468579"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"
MSL_DOI = "10.17887/wur01-gx7s1b"

# Safety settings
HTTP_TIMEOUT = 90
DOWNLOAD_CHUNK_MB = 4
RETRIES = 4

# Do not silently download one enormous global archive.
ALLOW_FULL_NATIVE_ARCHIVE = False

print("Configuration loaded.")

# 2 — Resolve project root and create only the missing data folders

In [ ]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

AOI_PATH = PROJECT_ROOT / AOI_RELATIVE
NATIVE_DIR = PROJECT_ROOT / NATIVE_DIR_REL
MSL_DIR = PROJECT_ROOT / MSL_DIR_REL
SOURCE_META_DIR = PROJECT_ROOT / SOURCE_META_REL

for d in [NATIVE_DIR, MSL_DIR, SOURCE_META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if not AOI_PATH.exists():
    raise FileNotFoundError(
        f"GBM AOI not found: {AOI_PATH}\n"
        "The boundary must be restored before downloading terrain tiles."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("AOI:", AOI_PATH)
print("Native target:", NATIVE_DIR)
print("MSL target:", MSL_DIR)

# 3 — Core project data audit

This cell distinguishes:

- already complete;
- present but optional;
- genuinely missing.

Sentinel-2 is not a required input to the terrain/RR analysis.

In [ ]:
REC_DIR = PROJECT_ROOT / "data/processed/recurrence"
EVENT_ROOT = PROJECT_ROOT / "data/processed/flood_events"
JRC = PROJECT_ROOT / "data/interim/water/jrc_permanent_water.tif"
S2_OPTIONAL = PROJECT_ROOT / "data/raw/sentinel2_optional"

required_existing = {
    "GBM boundary": AOI_PATH,
    "JRC permanent-water mask": JRC,
    "Flood recurrence": REC_DIR / "flood_recurrence.tif",
    "Flood count": REC_DIR / "flood_count.tif",
    "Observation count": REC_DIR / "observation_count.tif",
    "Flood stack": REC_DIR / "flood_stack.tif",
    "Valid stack": REC_DIR / "valid_stack.tif",
    "Event index": REC_DIR / "event_index.csv",
    "Frozen SAR configuration": PROJECT_ROOT / "config/frozen_sar_workflow.json",
}

rows = []

for name, path in required_existing.items():
    rows.append({
        "asset": name,
        "required_for_core": True,
        "exists": path.exists(),
        "path": str(path),
    })

for event_id in ["EVENT001", "EVENT002", "EVENT003", "EVENT004"]:
    folder = EVENT_ROOT / event_id
    rows.append({
        "asset": f"{event_id} final flood/valid/metadata",
        "required_for_core": True,
        "exists": (
            (folder / "flood_mask.tif").exists()
            and (folder / "valid_mask.tif").exists()
            and (folder / "event_flood_metadata.json").exists()
        ),
        "path": str(folder),
    })

rows.extend([
    {
        "asset": "Sentinel-2 optional QA directory",
        "required_for_core": False,
        "exists": S2_OPTIONAL.exists(),
        "path": str(S2_OPTIONAL),
    },
    {
        "asset": "Native DeltaDTM v1.1 raw tiles",
        "required_for_core": True,
        "exists": len(list(NATIVE_DIR.glob("*.tif"))) > 0,
        "path": str(NATIVE_DIR),
    },
    {
        "asset": "MSL-referenced DeltaDTM v1.1 raw tiles",
        "required_for_core": True,
        "exists": len(list(MSL_DIR.glob("*.tif"))) > 0,
        "path": str(MSL_DIR),
    },
])

audit_df = pd.DataFrame(rows)
display(audit_df)

print("\nMissing mandatory assets:")
display(audit_df[
    audit_df["required_for_core"]
    & (~audit_df["exists"])
][["asset", "path"]])

# 4 — GBM AOI feasibility and expected 1° tile neighborhood

The source repositories use tiled global products.

We derive a conservative set of 1° tile tokens intersecting or immediately surrounding the frozen GBM
boundary. A one-cell safety buffer is included only for source-file matching; Phase 02 will clip the
mosaic to the fixed AOI.

In [ ]:
aoi = gpd.read_file(AOI_PATH)

if aoi.empty or aoi.crs is None:
    raise RuntimeError("GBM boundary is empty or has no CRS.")

aoi4326 = aoi.to_crs(4326)

try:
    geom = aoi4326.geometry.union_all()
except AttributeError:
    geom = aoi4326.geometry.unary_union

if geom.is_empty:
    raise RuntimeError("GBM boundary union is empty.")

minx, miny, maxx, maxy = geom.bounds

print("AOI bounds WGS84:", geom.bounds)


def token(lat, lon):
    ns = "N" if lat >= 0 else "S"
    ew = "E" if lon >= 0 else "W"
    return f"{ns}{abs(int(lat)):02d}{ew}{abs(int(lon)):03d}"


# Conservative token set: cells intersecting AOI plus one degree around bbox.
candidate_tokens = set()

for lat in range(math.floor(miny) - 1, math.ceil(maxy) + 2):
    for lon in range(math.floor(minx) - 1, math.ceil(maxx) + 2):
        cell = box(lon, lat, lon + 1, lat + 1)

        if cell.intersects(geom) or cell.distance(geom) <= 1.5:
            # Southwest-style token
            candidate_tokens.add(token(lat, lon))
            # Northern-edge alternative used by some tiling conventions
            candidate_tokens.add(token(lat + 1, lon))

candidate_tokens = sorted(candidate_tokens)

print("Conservative tile tokens:", len(candidate_tokens))
print(candidate_tokens[:80])

# 5 — Network feasibility helpers

Every external request has:

- explicit status output;
- retry logic;
- timeout;
- streaming progress bar;
- cache/reuse behavior.

No existing valid file is redownloaded.

In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": "VRFSZ-GBM-research/1.0"
})


def request_with_retry(method, url, **kwargs):
    last = None

    for attempt in range(1, RETRIES + 1):
        try:
            response = session.request(
                method,
                url,
                timeout=HTTP_TIMEOUT,
                **kwargs,
            )

            if response.status_code >= 500:
                raise requests.HTTPError(
                    f"Server returned {response.status_code}",
                    response=response,
                )

            response.raise_for_status()
            return response

        except Exception as exc:
            last = exc
            print(
                f"Attempt {attempt}/{RETRIES} failed for {url}: {exc}"
            )

            if attempt < RETRIES:
                time.sleep(min(2 ** attempt, 15))

    raise RuntimeError(f"Request failed after retries: {url}\n{last}")


def download_stream(url, output_path, expected_size=None):
    output_path = Path(output_path)

    if output_path.exists() and output_path.stat().st_size > 0:
        print("CACHE:", output_path)
        return output_path

    response = request_with_retry("GET", url, stream=True)

    total = expected_size

    if total is None:
        try:
            total = int(response.headers.get("Content-Length", "0"))
        except Exception:
            total = 0

    temp = output_path.with_suffix(output_path.suffix + ".part")
    temp.parent.mkdir(parents=True, exist_ok=True)

    chunk = DOWNLOAD_CHUNK_MB * 1024 * 1024

    with open(temp, "wb") as f, tqdm(
        total=total or None,
        unit="B",
        unit_scale=True,
        desc=output_path.name,
    ) as bar:
        for block in response.iter_content(chunk_size=chunk):
            if not block:
                continue
            f.write(block)
            bar.update(len(block))

    temp.replace(output_path)
    return output_path


def filename_from_response(response, url, fallback):
    cd = response.headers.get("Content-Disposition", "")

    match = re.search(r'filename="?([^";]+)"?', cd, flags=re.I)

    if match:
        return Path(unquote(match.group(1))).name

    path_name = Path(unquote(urlparse(url).path)).name

    return path_name or fallback


print("Network helpers ready.")

# 6 — Download the official MSL-access metadata first

This retrieves only the small source guide and CSV from Zenodo.

The CSV is the preferred machine-readable route to the authors' MSL-referenced DeltaDTM v1.1 files.

In [ ]:
zenodo_record = request_with_retry("GET", ZENODO_API).json()

zenodo_meta_path = SOURCE_META_DIR / "zenodo_18468579_record.json"
zenodo_meta_path.write_text(
    json.dumps(zenodo_record, indent=2),
    encoding="utf-8",
)

files = zenodo_record.get("files", [])

display(pd.DataFrame([
    {
        "key": f.get("key"),
        "size_MB": round((f.get("size") or 0) / 1024**2, 3),
        "checksum": f.get("checksum"),
        "download": (
            f.get("links", {}).get("self")
            or f.get("links", {}).get("content")
        ),
    }
    for f in files
]))

wanted = {
    "DeltaDTM_v1.1.csv",
    "DeltaDTM_v1.1_guide.pdf",
}

downloaded_meta = {}

for f in files:
    key = f.get("key")

    if key not in wanted:
        continue

    url = (
        f.get("links", {}).get("self")
        or f.get("links", {}).get("content")
    )

    if not url:
        continue

    out = SOURCE_META_DIR / key

    download_stream(
        url,
        out,
        expected_size=f.get("size"),
    )

    downloaded_meta[key] = out

missing_guides = wanted - set(downloaded_meta)

if missing_guides:
    raise RuntimeError(
        "Could not obtain required DeltaDTM access guide files: "
        + ", ".join(sorted(missing_guides))
    )

print("✅ Official DeltaDTM v1.1 access metadata downloaded.")

# 7 — Native DeltaDTM v1.1 discovery through the official 4TU API

The notebook searches 4TU.ResearchData and prioritizes:

1. DOI family `10.4121/21997565`;
2. Version 4 / v1.1;
3. the DeltaDTM dataset title.

It will **not** silently use an unrelated DEM.

In [ ]:
# ============================================================
# STEP 7/8 REPLACEMENT
# Acquire Native DeltaDTM v1.1 from the GEE mirror
# ============================================================

import json
import math
import gc
from pathlib import Path

import ee
import geemap
import rasterio
from shapely.geometry import box, mapping
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Scientific source identity
# ------------------------------------------------------------

DELTADTM_OFFICIAL_DOI = "10.4121/21997565.v4"

DELTADTM_OFFICIAL_PAGE = (
    "https://data.4tu.nl/datasets/"
    "1da2e70f-6c4d-4b03-86bd-b53e789cc629/4"
)

# GEE is used only as an efficient spatial-access mirror.
DELTADTM_GEE_ASSET = (
    "projects/sat-io/open-datasets/"
    "DELTARES/deltadtm_v1-1"
)

GEE_PROJECT = "ee-tarin1"


print("=" * 80)
print("NATIVE DELTADTM v1.1 ACQUISITION")
print("=" * 80)

print("Official DOI :", DELTADTM_OFFICIAL_DOI)
print("GEE mirror   :", DELTADTM_GEE_ASSET)
print("Output folder:", NATIVE_DIR)


# ------------------------------------------------------------
# 2. Initialize Earth Engine
# ------------------------------------------------------------

try:
    ee.Initialize(project=GEE_PROJECT)

except Exception:

    print(
        "Existing GEE session unavailable. "
        "Starting authentication..."
    )

    ee.Authenticate(
        auth_mode="localhost"
    )

    ee.Initialize(
        project=GEE_PROJECT
    )


# API probe
probe = ee.String(
    "DELTADTM_CHECK"
).getInfo()

if probe != "DELTADTM_CHECK":
    raise RuntimeError(
        "Earth Engine API probe failed."
    )

print("✅ Earth Engine initialized.")


# ------------------------------------------------------------
# 3. Verify the DeltaDTM asset
# ------------------------------------------------------------

try:

    delta = (
        ee.Image(
            DELTADTM_GEE_ASSET
        )
        .select("b1")
        .rename("elevation")
        .toFloat()
    )

    asset_info = delta.getInfo()

    projection_info = (
        delta
        .projection()
        .getInfo()
    )

    nominal_scale = (
        delta
        .projection()
        .nominalScale()
        .getInfo()
    )

except Exception as exc:

    raise RuntimeError(
        "\nThe DeltaDTM GEE mirror could not be opened.\n"
        f"Asset: {DELTADTM_GEE_ASSET}\n"
        f"Error: {exc}"
    )


print("✅ DeltaDTM GEE asset accessible.")
print("Nominal scale:", nominal_scale, "m")
print("Projection:", projection_info)


# ------------------------------------------------------------
# IMPORTANT:
#
# Do NOT apply:
#
#     elevation.updateMask(elevation.neq(30))
#
# The community example does that for visualization because
# DeltaDTM is clipped at 30 m.
#
# For THIS experiment, 30 m pixels are legitimately > 0.5,
# > 1 and > 2 m and therefore belong to Stable-High terrain.
#
# Removing them would bias the Stable-High denominator.
# ------------------------------------------------------------


# ------------------------------------------------------------
# 4. Convert the frozen GBM AOI to EE
#
# `geom` already exists from Step 4 of your notebook.
# ------------------------------------------------------------

if geom is None or geom.is_empty:
    raise RuntimeError(
        "GBM geometry from Step 4 is unavailable."
    )

EE_AOI = ee.Geometry(
    mapping(geom)
)

print("GBM bounds:", geom.bounds)


# ------------------------------------------------------------
# 5. Generate only 1-degree cells intersecting GBM
#
# This avoids downloading the entire 17+ GB Asia archive.
# ------------------------------------------------------------

minx, miny, maxx, maxy = geom.bounds

download_cells = []

for lat in range(
    math.floor(miny),
    math.ceil(maxy)
):

    for lon in range(
        math.floor(minx),
        math.ceil(maxx)
    ):

        tile_geom = box(
            lon,
            lat,
            lon + 1,
            lat + 1
        )

        intersection = (
            tile_geom
            .intersection(geom)
        )

        if intersection.is_empty:
            continue

        download_cells.append(
            {
                "lon": lon,
                "lat": lat,
                "geometry": intersection,
            }
        )


print(
    "GBM-intersecting download cells:",
    len(download_cells)
)


# ------------------------------------------------------------
# 6. DeltaDTM filename helper
#
# DeltaDTM uses the TOP-LEFT tile coordinate.
# ------------------------------------------------------------

def deltadtm_token(lat_bottom, lon_left):

    lat_top = lat_bottom + 1

    ns = (
        "N"
        if lat_top >= 0
        else "S"
    )

    ew = (
        "E"
        if lon_left >= 0
        else "W"
    )

    return (
        f"{ns}{abs(int(lat_top)):02d}"
        f"{ew}{abs(int(lon_left)):03d}"
    )


# ------------------------------------------------------------
# 7. Download one manageable subset at a time
# ------------------------------------------------------------

downloaded_native = []

for item in tqdm(
    download_cells,
    desc="Native DeltaDTM GBM tiles",
    unit="tile",
):

    token_name = deltadtm_token(
        item["lat"],
        item["lon"],
    )

    out = (
        NATIVE_DIR
        / f"DeltaDTM_v1_1_{token_name}.tif"
    )

    # --------------------------------------------------------
    # Reuse verified existing downloads.
    # --------------------------------------------------------

    if out.exists():

        try:

            with rasterio.open(out) as src:

                if (
                    src.width > 0
                    and src.height > 0
                    and src.count == 1
                ):

                    downloaded_native.append(
                        out
                    )

                    print(
                        "CACHE:",
                        out.name
                    )

                    continue

        except Exception:

            print(
                "Existing raster invalid; "
                "redownloading:",
                out.name
            )

            out.unlink(
                missing_ok=True
            )


    ee_region = ee.Geometry(
        mapping(
            item["geometry"]
        )
    )

    image = (
        delta
        .clip(ee_region)
    )

    print(
        "\nDownloading:",
        out.name
    )

    geemap.download_ee_image(
        image=image,
        filename=str(out),
        region=ee_region,
        scale=30,
        crs="EPSG:4326",
        overwrite=True,
        num_threads=4,
    )


    # --------------------------------------------------------
    # Immediate raster integrity check
    # --------------------------------------------------------

    if not out.exists():
        raise RuntimeError(
            f"Download failed: {out}"
        )

    try:

        with rasterio.open(out) as src:

            if (
                src.width <= 0
                or src.height <= 0
                or src.count != 1
            ):
                raise RuntimeError(
                    f"Invalid raster dimensions: {out}"
                )

            print(
                "  size       :",
                src.width,
                "x",
                src.height
            )

            print(
                "  CRS        :",
                src.crs
            )

            print(
                "  resolution :",
                src.res
            )

            print(
                "  bounds     :",
                src.bounds
            )

    except Exception as exc:

        out.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            f"Downloaded raster failed validation: {out}\n"
            f"{exc}"
        )


    downloaded_native.append(
        out
    )


# ------------------------------------------------------------
# 8. Final source audit
# ------------------------------------------------------------

native_tifs = sorted(
    NATIVE_DIR.glob("*.tif")
)

if not native_tifs:

    raise RuntimeError(
        "No native DeltaDTM rasters were produced."
    )


print("\n" + "=" * 80)

print(
    "✅ NATIVE DELTADTM ACQUISITION COMPLETE"
)

print("=" * 80)

print(
    "Raster count:",
    len(native_tifs)
)


for p in native_tifs:

    print(
        " -",
        p.name,
        round(
            p.stat().st_size
            / 1024**2,
            2
        ),
        "MB"
    )


# ------------------------------------------------------------
# 9. Save explicit provenance
# ------------------------------------------------------------

provenance = {

    "dataset":
        "DeltaDTM v1.1",

    "scientific_source":
        "4TU.ResearchData",

    "official_doi":
        DELTADTM_OFFICIAL_DOI,

    "official_landing_page":
        DELTADTM_OFFICIAL_PAGE,

    "acquisition_transport":
        "Google Earth Engine Community Catalog mirror",

    "gee_asset":
        DELTADTM_GEE_ASSET,

    "nominal_resolution_m":
        nominal_scale,

    "requested_export_scale_m":
        30,

    "study_area":
        "GBM delta frozen AOI",

    "tile_count":
        len(native_tifs),

    "important_note":
        (
            "Values equal to 30 m were retained. "
            "The source is clipped at 30 m; these pixels remain "
            "valid Stable-High observations for thresholds <=2 m."
        ),

    "files": [
        p.name
        for p in native_tifs
    ],
}


provenance_path = (
    SOURCE_META_DIR
    / "native_deltadtm_v1_1_provenance.json"
)

provenance_path.write_text(
    json.dumps(
        provenance,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


print(
    "\nProvenance:",
    provenance_path
)

gc.collect()

# 9 — Parse the official DeltaDTM v1.1 MSL access CSV

The guide CSV is used instead of inventing a conversion.

This cell searches each CSV row for:

- a DeltaDTM tile token overlapping the GBM neighborhood;
- an official download URL.

It prints the matched records **before downloading**.

In [ ]:
guide_csv = downloaded_meta["DeltaDTM_v1.1.csv"]

msl_guide_df = pd.read_csv(
    guide_csv,
    dtype=str,
)

print("Guide columns:")
print(list(msl_guide_df.columns))

display(msl_guide_df.head(10))


def row_blob(row):
    return " ".join(
        "" if pd.isna(v) else str(v)
        for v in row.values
    )


blobs = msl_guide_df.apply(
    row_blob,
    axis=1,
)

token_pattern = "|".join(
    re.escape(tok)
    for tok in candidate_tokens
)

matched_mask = blobs.str.contains(
    token_pattern,
    case=False,
    regex=True,
    na=False,
)

msl_matches = msl_guide_df[
    matched_mask
].copy()

print("Rows matching GBM tile neighborhood:", len(msl_matches))

if len(msl_matches):
    display(msl_matches.head(100))
else:
    print(
        "⚠️ No filename token was recognized automatically. "
        "The guide may use a different tile-index scheme."
    )


def extract_http_values(row):
    values = []

    for col, value in row.items():
        if pd.isna(value):
            continue

        text = str(value).strip()

        # Extract one or more URLs embedded in a cell.
        urls = re.findall(
            r'https?://[^\s,;"\']+',
            text,
        )

        for url in urls:
            values.append((col, url))

    return values


download_rows = []

for idx, row in msl_matches.iterrows():
    urls = extract_http_values(row)

    for col, url in urls:
        score = 0
        lc = col.lower()
        lu = url.lower()

        if any(k in lc for k in ["download", "url", "link"]):
            score += 3

        if any(k in lu for k in [".tif", ".tiff", "download"]):
            score += 3

        if any(k in lc for k in ["mdt", "msl", "sea"]):
            score += 4

        download_rows.append({
            "row_index": idx,
            "column": col,
            "url": url,
            "score": score,
        })


download_link_df = pd.DataFrame(download_rows)

if not download_link_df.empty:
    download_link_df = (
        download_link_df
        .sort_values(["row_index", "score"], ascending=[True, False])
        .drop_duplicates(["row_index"], keep="first")
    )

display(download_link_df if not download_link_df.empty else pd.DataFrame())

print(
    "Candidate official MSL download links:",
    len(download_link_df),
)

# 10 — Download MSL-referenced DeltaDTM v1.1 GBM tiles

Only URLs surfaced by the authors' official guide CSV are used.

If the CSV structure does not expose per-tile links automatically, this cell stops with a clear manual
action instead of manufacturing an MSL correction.

In [ ]:
msl_downloaded = []

if download_link_df.empty:
    raise RuntimeError(
        "\nAUTOMATIC MSL TILE DOWNLOAD COULD NOT BE RESOLVED FROM THE OFFICIAL GUIDE CSV.\n\n"
        f"Open: {downloaded_meta['DeltaDTM_v1.1_guide.pdf']}\n"
        f"Use the guide's DeltaDTM v1.1 MSL download route and place the matching "
        f"GBM tiles in:\n{MSL_DIR}\n\n"
        "Do not generate your own local MDT/geoid correction as a shortcut."
    )


for _, rec in tqdm(
    download_link_df.iterrows(),
    total=len(download_link_df),
    desc="MSL source files",
    unit="file",
):
    url = rec["url"]

    # First request only to resolve filename / headers.
    response = request_with_retry(
        "GET",
        url,
        stream=True,
    )

    name = filename_from_response(
        response,
        url,
        fallback=f"DeltaDTM_MSL_{int(rec['row_index']):05d}.tif",
    )

    # Close probe response; stream download is retried cleanly.
    response.close()

    if not name.lower().endswith((".tif", ".tiff", ".zip")):
        # Keep provenance but do not write arbitrary HTML as a raster.
        print(
            "SKIP non-raster/non-archive link:",
            url,
            "resolved as",
            name,
        )
        continue

    out = MSL_DIR / name

    download_stream(
        url,
        out,
    )

    if out.suffix.lower() in [".tif", ".tiff"]:
        try:
            with rasterio.open(out) as src:
                _ = src.width, src.height, src.crs
            msl_downloaded.append(out)
        except Exception:
            out.unlink(missing_ok=True)
            print("Rejected non-readable raster:", out)

    elif out.suffix.lower() == ".zip":
        # Extract matching source tiles and delete archive.
        with zipfile.ZipFile(out, "r") as zf:
            members = [
                info for info in zf.infolist()
                if info.filename.lower().endswith((".tif", ".tiff"))
                and any(
                    tok in Path(info.filename).name.upper()
                    for tok in candidate_tokens
                )
            ]

            for info in members:
                target = MSL_DIR / Path(info.filename).name

                if not target.exists():
                    with zf.open(info) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst)

                msl_downloaded.append(target)

        out.unlink(missing_ok=True)


msl_tifs = sorted(MSL_DIR.glob("*.tif"))

print("MSL GeoTIFFs now present:", len(msl_tifs))

if not msl_tifs:
    raise RuntimeError(
        "No readable MSL-referenced DeltaDTM GeoTIFF was downloaded. "
        "Follow the official guide manually; do not synthesize the correction."
    )

for p in msl_tifs[:50]:
    print(" -", p.name)

# 11 — Source-level terrain-pair sanity gate

This does **not** mosaic or resample.

It confirms:

- both source folders contain readable GeoTIFFs;
- horizontal CRS families are plausible;
- source tiles spatially overlap the GBM AOI;
- the two datasets are the required DeltaDTM v1.1 pair.

Exact pixel correspondence is tested later after the official Phase-02 mosaic procedure.

In [ ]:
native_tifs = sorted(NATIVE_DIR.glob("*.tif"))
msl_tifs = sorted(MSL_DIR.glob("*.tif"))

if not native_tifs:
    raise RuntimeError(
        "Native DeltaDTM v1.1 is still missing."
    )

if not msl_tifs:
    raise RuntimeError(
        "MSL-referenced DeltaDTM v1.1 is still missing."
    )


def inspect_rasters(paths, label):
    rows = []

    for p in tqdm(paths, desc=f"Inspect {label}", unit="tile"):
        try:
            with rasterio.open(p) as src:
                rows.append({
                    "file": p.name,
                    "width": src.width,
                    "height": src.height,
                    "crs": str(src.crs),
                    "res_x": src.res[0],
                    "res_y": src.res[1],
                    "bounds": tuple(src.bounds),
                    "readable": True,
                })
        except Exception as exc:
            rows.append({
                "file": p.name,
                "readable": False,
                "error": str(exc),
            })

    return pd.DataFrame(rows)


native_info = inspect_rasters(native_tifs, "native")
msl_info = inspect_rasters(msl_tifs, "MSL")

display(native_info)
display(msl_info)

if not native_info["readable"].all():
    raise RuntimeError("At least one native DeltaDTM file is unreadable.")

if not msl_info["readable"].all():
    raise RuntimeError("At least one MSL DeltaDTM file is unreadable.")

print("✅ Both required terrain source families are present and readable.")
print(
    "Next scientific step is Phase 02 mosaicking—not the final statistical notebook."
)

# 12 — Updated project readiness table

A green terrain row means the missing data problem is solved.

It still does **not** mean you should jump to 18B. You must rerun the terrain Phase-A processing first.

In [ ]:
status = [
    {
        "component": "Frozen GBM boundary",
        "status": "READY" if AOI_PATH.exists() else "MISSING",
        "next_action": "none" if AOI_PATH.exists() else "restore boundary",
    },
    {
        "component": "Native DeltaDTM v1.1 EGM2008",
        "status": "READY" if len(native_tifs) else "MISSING",
        "next_action": "Phase 02" if len(native_tifs) else "complete official download",
    },
    {
        "component": "DeltaDTM v1.1 MSL / MDT HYBRID-CNES-CLS2022",
        "status": "READY" if len(msl_tifs) else "MISSING",
        "next_action": "Phase 02" if len(msl_tifs) else "complete official download",
    },
    {
        "component": "JRC permanent-water mask",
        "status": "READY" if JRC.exists() else "MISSING",
        "next_action": "none" if JRC.exists() else "export JRC GSW v1.4",
    },
    {
        "component": "Four Sentinel-1 flood events",
        "status": "READY" if all(
            (
                (EVENT_ROOT / e / "flood_mask.tif").exists()
                and (EVENT_ROOT / e / "valid_mask.tif").exists()
            )
            for e in ["EVENT001", "EVENT002", "EVENT003", "EVENT004"]
        ) else "INCOMPLETE",
        "next_action": "do not retune",
    },
    {
        "component": "Multi-event recurrence products",
        "status": "READY" if (REC_DIR / "flood_recurrence.tif").exists() else "MISSING",
        "next_action": "preserve",
    },
    {
        "component": "Sentinel-2",
        "status": "OPTIONAL / PRESENT" if S2_OPTIONAL.exists() else "OPTIONAL / ABSENT",
        "next_action": "use only for independent flood QA if useful",
    },
]

status_df = pd.DataFrame(status)
display(status_df)

out_csv = SOURCE_META_DIR / "core_data_status_after_01B.csv"
status_df.to_csv(out_csv, index=False)

print("Saved:", out_csv)

# 13 — What to run next

If both DeltaDTM rows are READY, stop downloading data.

Run:

```text
02_Ingest_and_Mosaic_Core_Data.ipynb
        ↓
03_Validate_Paired_DEM_Grid.ipynb
        ↓
04_Compute_Vertical_Shift.ipynb
        ↓
05_Classify_Primary_VRFSZ_T1m.ipynb
        ↓
18A3_Terrain_Recovery_and_Statistical_Readiness.ipynb
```

Only after 18A3 ends with:

```text
✅ TECHNICAL READINESS: PASS
```

should you run the final statistical workflow.

---

# Final data scope

Do **not** download extra datasets merely because they are geospatially interesting.

The core paper does not require:

- GRACE;
- GPM / CHIRPS;
- ERA5;
- soil grids;
- HAND;
- hydrodynamic-model forcing;
- WorldPop;
- roads / settlements;
- subsidence;
- SWOT;
- future sea-level scenarios.

Those belong to different research questions.

# STAGE 2 — Exact-grid terrain preparation and VRFSZ readiness

**Source provenance:** `02_to_18A3_Combined_Terrain_VRFSZ_Readiness_v2_ExactGrid.ipynb`.

The paired terrain members remain on the common source grid; no convenience warp is used to force equivalence.

# VRFSZ–GBM — Combined Terrain Phase A + Statistical Readiness

This single notebook replaces the separate sequence:

```text
Source-pair sanity gate
        ↓
02 Ingest + Mosaic Core Terrain
        ↓
03 Validate Paired DEM Grid
        ↓
04 Compute Vertical Shift ΔZ
        ↓
05 Classify Primary VRFSZ T = 1 m
        ↓
18A3 Statistical Readiness
```

It stops immediately at any scientifically important failure.

---

# Core research logic preserved

The experiment compares two vertical representations of the **same underlying DeltaDTM terrain**:

\[
Z_N = \text{native DeltaDTM v1.1 / EGM2008 representation}
\]

\[
Z_M = \text{DeltaDTM v1.1 referenced to MSL/MDT}
\]

and defines:

\[
\Delta Z = Z_N - Z_M.
\]

At the primary threshold \(T=1\) m:

\[
StableLow:\quad Z_N\le T,\;Z_M\le T
\]

\[
VRFSZ:\quad Z_N>T,\;Z_M\le T
\]

\[
StableHigh:\quad Z_N>T,\;Z_M>T.
\]

A fourth logical state is retained explicitly:

\[
Reverse:\quad Z_N\le T,\;Z_M>T.
\]

It is a diagnostic class and is never silently forced into one of the three planned classes.

---

# Critical repair built into this notebook — v2 exact-grid method

Your current `01B` run downloaded:

- MSL source tiles on the original one-arcsecond grid;
- native DeltaDTM from the GEE mirror using `scale=30`.

Those native exports are **not on the same source pixel lattice** as the MSL files.

This notebook does **not** resample those already-resampled native exports back to one arc-second.

Instead, where necessary, it re-exports the authoritative DeltaDTM v1.1 GEE asset directly onto
the exact transform, width, height and bounds of each MSL source tile that intersects the frozen GBM AOI.

Therefore the corrected source pair is pixel-corresponding **before** mosaicking.

---

# What this notebook will NOT do

It will not:

- derive a new local geoid correction;
- replace DeltaDTM with SRTM, Copernicus DEM or NASADEM;
- modify Sentinel-1 flood masks;
- use VRFSZ to tune Sentinel-1;
- calculate relative risk yet;
- run the 0.5/2.0 m final sensitivity analysis yet.

Those inferential steps begin only after the final readiness gate passes.---

# v2 correction after the observed grid-mismatch failure

The earlier version attempted:

```python
geemap.download_ee_image(
    ...,
    crs=...,
    crs_transform=...,
    region=...
)
```

The run proved that this helper still returned a file whose transform/shape did not exactly
match the 3600 × 3600 MSL tile.

That failure does **not** prove that the source datasets are on different lattices.

The executed run showed:

- MSL pixel size = exactly `1/3600°`;
- GEE DeltaDTM source pixel size = exactly `1/3600°`;
- the GEE source translation and MSL translations differ by integer pixel counts.

Therefore the **source grids are mathematically compatible**.

The problem is the export/cropping layer.

## v2 scientifically conservative solution

For every MSL tile:

1. use the MSL tile itself as the authoritative target grid;
2. divide its 3600 × 3600 grid into smaller exact pixel windows;
3. request each native DeltaDTM window directly with Earth Engine
   `Image.getDownloadURL`;
4. specify exactly:
   - CRS,
   - affine `crs_transform`,
   - `dimensions`;
5. do **not** specify `scale`;
6. do **not** use a geographic crop region for the pixel-grid definition;
7. validate every returned chunk against its requested transform and dimensions;
8. write the chunks directly into a blank 3600 × 3600 GeoTIFF that inherits the
   exact MSL tile transform.

No local interpolation occurs.

Because the target lattice is already identical to the DeltaDTM source lattice, this is a
pixel-grid extraction, not a DEM-resampling experiment.

Google Earth Engine documentation explicitly states that when matching an existing grid,
`crs` and `crsTransform` should be used rather than `scale`; `scale` alone can shift pixels.
The direct-download API also documents a 32 MB request limit, so this notebook uses aligned
sub-tiles rather than requesting one complete float32 3600 × 3600 tile at once.

# 1 — User controls

Normally only `GEE_PROJECT` needs to match your Earth Engine project.

The defaults favor reproducibility over convenience.

In [ ]:
# ============================================================
# USER CONTROLS
# ============================================================

GEE_PROJECT = "ee-tarin1"

AOI_RELATIVE = "data/raw/boundaries/gbm_delta.shp"

NATIVE_RAW_REL = "data/raw/dem/native_tiles"
MSL_RAW_REL = "data/raw/dem/msl_tiles"
SOURCE_META_REL = "data/raw/dem/source_metadata"

TERRAIN_OUT_REL = "data/processed/terrain"
TERRAIN_REPORT_REL = "reports/terrain_phase"

NATIVE_GEE_ASSET = (
    "projects/sat-io/open-datasets/"
    "DELTARES/deltadtm_v1-1"
)

PRIMARY_THRESHOLD_M = 1.0
READINESS_THRESHOLDS_M = [0.5, 1.0, 2.0]

FLOAT_NODATA = -9999.0
CLASS_NODATA = 0

# Source-grid repair:
AUTO_REPAIR_NATIVE_GRID = True

# After a fully successful run, delete the superseded scale=30
# native exports from data/raw/dem/native_tiles.
DELETE_SUPERSEDED_NATIVE_AFTER_PASS = True

# Keep the exact source-grid native tiles for reproducibility.
KEEP_EXACT_NATIVE_TILES = True

# Do not delete extra MSL tiles by default.
DELETE_NON_AOI_MSL_TILES = False

# Readiness diagnostics only.
WARN_COMMON_VALID_FRACTION = 0.95
MIN_BOUNDS_COVERAGE = 0.99
MIN_EVENTS = 4

# Sampling for ΔZ descriptive percentiles.
MAX_DZ_SAMPLE_VALUES = 1_000_000
RANDOM_SEED = 20260830

print("Controls loaded.")

# ------------------------------------------------------------
# Exact-grid direct Earth Engine download controls (v2)
# ------------------------------------------------------------

# 1800 × 1800 float32 ≈ 13 MB raw, safely below EE's 32 MB
# getDownloadURL request-size ceiling.
EE_EXACT_CHUNK_PIXELS = 1800

EE_DIRECT_MAX_RETRIES = 5
EE_DIRECT_TIMEOUT_SECONDS = 180

# Direct-download chunks are validated before they are written.
STRICT_EXACT_CHUNK_VALIDATION = True


# 2 — Imports and environment feasibility

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import importlib.util
import inspect
import json
import math
import re
import shutil
import sys
import io
import time
import zipfile
import requests

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import windows
from rasterio.features import geometry_mask
from rasterio.transform import Affine
from rasterio.warp import transform_bounds
from shapely.geometry import mapping, box
from tqdm.auto import tqdm

try:
    from pyproj import CRS, Geod
except Exception as exc:
    raise RuntimeError(
        "pyproj is required for CRS and geodesic area checks."
    ) from exc

required_packages = [
    "numpy",
    "pandas",
    "geopandas",
    "rasterio",
    "pyproj",
    "matplotlib",
    "ee",
    "geemap",
    "requests",
]

pkg_rows = []

for pkg in required_packages:
    pkg_rows.append({
        "package": pkg,
        "available": importlib.util.find_spec(pkg) is not None,
    })

pkg_df = pd.DataFrame(pkg_rows)
display(pkg_df)

missing = pkg_df.loc[~pkg_df["available"], "package"].tolist()

if missing:
    raise RuntimeError(
        "Missing Python packages: " + ", ".join(missing)
    )

import ee
import geemap
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("Environment feasibility PASS.")

# 3 — Resolve project paths

In [ ]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

AOI_PATH = PROJECT_ROOT / AOI_RELATIVE
NATIVE_RAW_DIR = PROJECT_ROOT / NATIVE_RAW_REL
MSL_RAW_DIR = PROJECT_ROOT / MSL_RAW_REL
SOURCE_META_DIR = PROJECT_ROOT / SOURCE_META_REL

TERRAIN_OUT = PROJECT_ROOT / TERRAIN_OUT_REL
TERRAIN_REPORT = PROJECT_ROOT / TERRAIN_REPORT_REL

EXACT_NATIVE_DIR = NATIVE_RAW_DIR / "source_grid_exact"

NATIVE_MOSAIC = TERRAIN_OUT / "DeltaDTM_v1_1_native_GBM.tif"
MSL_MOSAIC = TERRAIN_OUT / "DeltaDTM_v1_1_MSL_GBM.tif"
COMMON_VALID = TERRAIN_OUT / "terrain_common_valid_mask.tif"
DELTA_Z = TERRAIN_OUT / "delta_z_native_minus_msl_GBM.tif"
CLASS_T1 = TERRAIN_OUT / "vrfsz_classes_T1m.tif"

REC_DIR = PROJECT_ROOT / "data/processed/recurrence"
FLOOD_STACK = REC_DIR / "flood_stack.tif"
VALID_STACK = REC_DIR / "valid_stack.tif"
FLOOD_COUNT = REC_DIR / "flood_count.tif"
OBS_COUNT = REC_DIR / "observation_count.tif"
RECURRENCE = REC_DIR / "flood_recurrence.tif"
EVENT_INDEX = REC_DIR / "event_index.csv"

EVENT_ROOT = PROJECT_ROOT / "data/processed/flood_events"
FROZEN_SAR_CONFIG = PROJECT_ROOT / "config/frozen_sar_workflow.json"

for d in [
    NATIVE_RAW_DIR,
    MSL_RAW_DIR,
    SOURCE_META_DIR,
    EXACT_NATIVE_DIR,
    TERRAIN_OUT,
    TERRAIN_REPORT,
]:
    d.mkdir(parents=True, exist_ok=True)

if not AOI_PATH.exists():
    raise FileNotFoundError(AOI_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Terrain outputs:", TERRAIN_OUT)

# 4 — Load and freeze the GBM AOI

The same boundary used in the Sentinel branch is retained.

In [ ]:
aoi = gpd.read_file(AOI_PATH)

if aoi.empty:
    raise RuntimeError("GBM boundary is empty.")

if aoi.crs is None:
    raise RuntimeError("GBM boundary has no CRS.")

if not aoi.geometry.is_valid.all():
    print("Repairing invalid AOI geometry with buffer(0).")
    aoi["geometry"] = aoi.geometry.buffer(0)

aoi4326 = aoi.to_crs(4326)

try:
    AOI_GEOM = aoi4326.geometry.union_all()
except AttributeError:
    AOI_GEOM = aoi4326.geometry.unary_union

if AOI_GEOM.is_empty:
    raise RuntimeError("GBM union geometry is empty.")

print("GBM AOI bounds:", AOI_GEOM.bounds)
print("AOI CRS:", aoi.crs)

# 5 — Preserve the completed independent Sentinel-1 branch

This notebook does not reprocess Sentinel-1.

It only confirms the observation branch still exists before the terrain branch is allowed to proceed.

In [ ]:
observation_assets = {
    "flood_stack": FLOOD_STACK,
    "valid_stack": VALID_STACK,
    "flood_count": FLOOD_COUNT,
    "observation_count": OBS_COUNT,
    "recurrence": RECURRENCE,
    "event_index": EVENT_INDEX,
    "frozen_sar_config": FROZEN_SAR_CONFIG,
}

obs_rows = []

for name, path in observation_assets.items():
    obs_rows.append({
        "asset": name,
        "exists": path.exists(),
        "path": str(path),
    })

obs_df = pd.DataFrame(obs_rows)
display(obs_df)

if not obs_df["exists"].all():
    raise RuntimeError(
        "The previously completed Sentinel-1 recurrence branch is incomplete."
    )

event_index = pd.read_csv(EVENT_INDEX)

if "event_id" not in event_index.columns:
    raise RuntimeError("event_index.csv has no event_id field.")

EVENT_IDS = event_index["event_id"].astype(str).tolist()

if len(EVENT_IDS) < MIN_EVENTS:
    raise RuntimeError(
        f"Only {len(EVENT_IDS)} events found; expected at least {MIN_EVENTS}."
    )

print("Events retained:", EVENT_IDS)
print("✅ Independent flood-observation branch preserved.")

# 6 — SOURCE-PAIR SANITY GATE — inventory the raw terrain data

The project requires the corresponding DeltaDTM pair.

The MSL files are used as the **source-grid template** because the downloaded files are original
one-degree, one-arcsecond tiles.

Current native GEE exports are accepted only if they already match this exact pixel lattice.
Otherwise they are superseded by direct source-grid exports from the same DeltaDTM GEE asset.

In [ ]:
native_existing = sorted(
    p for p in NATIVE_RAW_DIR.glob("*.tif")
    if p.is_file()
)

msl_all = sorted(
    p for p in MSL_RAW_DIR.glob("*.tif")
    if p.is_file()
)

print("Existing native TIFFs:", len(native_existing))
print("MSL TIFFs:", len(msl_all))

if not msl_all:
    raise RuntimeError(
        "No MSL-referenced DeltaDTM source tiles found. "
        "Run/finish 01B first."
    )

if not native_existing and not AUTO_REPAIR_NATIVE_GRID:
    raise RuntimeError(
        "No native DeltaDTM source exists and automatic source-grid export is disabled."
    )


def raster_record(path):
    with rasterio.open(path) as src:
        return {
            "file": path.name,
            "path": str(path),
            "width": src.width,
            "height": src.height,
            "crs": str(src.crs),
            "res_x": float(src.res[0]),
            "res_y": float(abs(src.res[1])),
            "transform": tuple(src.transform),
            "bounds": tuple(src.bounds),
            "nodata": src.nodata,
        }


native_inventory = pd.DataFrame(
    [raster_record(p) for p in native_existing]
) if native_existing else pd.DataFrame()

msl_inventory = pd.DataFrame(
    [raster_record(p) for p in msl_all]
)

print("\nNative source inventory:")
display(native_inventory)

print("\nMSL source inventory:")
display(msl_inventory)

## 6.1 Select only MSL source tiles that truly intersect the frozen AOI

The 01B acquisition intentionally downloaded a conservative neighborhood.  
Extra MSL files are ignored here rather than contaminating the mosaic.

In [ ]:
def raster_bounds_wgs84(path):
    with rasterio.open(path) as src:
        if src.crs is None:
            raise RuntimeError(f"No CRS: {path}")

        if CRS.from_user_input(src.crs).to_epsg() == 4326:
            return src.bounds

        return transform_bounds(
            src.crs,
            "EPSG:4326",
            *src.bounds,
            densify_pts=21,
        )


selected_msl = []

for p in msl_all:
    b = raster_bounds_wgs84(p)
    tile_box = box(*b)

    if tile_box.intersects(AOI_GEOM):
        selected_msl.append(p)

print("MSL tiles intersecting AOI:", len(selected_msl))

if not selected_msl:
    raise RuntimeError(
        "None of the MSL DeltaDTM tiles intersects the GBM AOI."
    )

for p in selected_msl:
    print(" -", p.name)

## 6.2 Validate the MSL source lattice

All selected MSL source tiles must have the same horizontal CRS, resolution and common grid alignment.

In [ ]:
def horizontal_crs(crs_obj):
    crs = CRS.from_user_input(crs_obj)

    if crs.is_compound:
        for sub in crs.sub_crs_list:
            if sub.is_projected or sub.is_geographic:
                return sub

    if crs.is_projected or crs.is_geographic:
        return crs

    return crs.geodetic_crs


def crs_equivalent(a, b):
    return horizontal_crs(a).equals(horizontal_crs(b))


with rasterio.open(selected_msl[0]) as ref:
    MSL_REF_CRS = ref.crs
    MSL_REF_TRANSFORM = ref.transform
    MSL_XRES = float(ref.transform.a)
    MSL_YRES = float(abs(ref.transform.e))

msl_grid_rows = []

for p in selected_msl:
    with rasterio.open(p) as src:
        crs_ok = crs_equivalent(src.crs, MSL_REF_CRS)

        res_ok = np.allclose(
            [src.res[0], abs(src.res[1])],
            [MSL_XRES, MSL_YRES],
            rtol=0,
            atol=1e-12,
        )

        # Check that the tile origin differs from the reference by integer pixels.
        dx = (src.transform.c - MSL_REF_TRANSFORM.c) / MSL_XRES
        dy = (MSL_REF_TRANSFORM.f - src.transform.f) / MSL_YRES

        aligned = (
            abs(dx - round(dx)) < 1e-7
            and abs(dy - round(dy)) < 1e-7
        )

        msl_grid_rows.append({
            "file": p.name,
            "crs_ok": crs_ok,
            "resolution_ok": res_ok,
            "grid_aligned": aligned,
            "width": src.width,
            "height": src.height,
            "res_x": src.res[0],
            "res_y": abs(src.res[1]),
        })

msl_grid_df = pd.DataFrame(msl_grid_rows)
display(msl_grid_df)

if not msl_grid_df[
    ["crs_ok", "resolution_ok", "grid_aligned"]
].all().all():
    raise RuntimeError(
        "Selected MSL source tiles do not share one consistent source grid."
    )

print(
    "MSL source resolution:",
    MSL_XRES,
    "×",
    MSL_YRES,
    "degrees",
)

print("✅ MSL source lattice PASS.")

## 6.3 Diagnose the existing native exports

This explicitly detects the `scale=30` acquisition artifact found in the current 01B run.

A native file is **not** accepted merely because it is approximately 30 m.
It must lie on the exact same one-arcsecond source lattice as the MSL product.

In [ ]:
def source_grid_aligned_to_msl(path):
    with rasterio.open(path) as src:
        if not crs_equivalent(src.crs, MSL_REF_CRS):
            return False

        if not np.allclose(
            [src.res[0], abs(src.res[1])],
            [MSL_XRES, MSL_YRES],
            rtol=0,
            atol=1e-12,
        ):
            return False

        dx = (src.transform.c - MSL_REF_TRANSFORM.c) / MSL_XRES
        dy = (MSL_REF_TRANSFORM.f - src.transform.f) / MSL_YRES

        return (
            abs(dx - round(dx)) < 1e-7
            and abs(dy - round(dy)) < 1e-7
        )


native_grid_rows = []

for p in native_existing:
    with rasterio.open(p) as src:
        native_grid_rows.append({
            "file": p.name,
            "grid_compatible_with_msl": source_grid_aligned_to_msl(p),
            "res_x": src.res[0],
            "res_y": abs(src.res[1]),
            "width": src.width,
            "height": src.height,
        })

native_grid_df = pd.DataFrame(native_grid_rows)

if not native_grid_df.empty:
    display(native_grid_df)

CURRENT_NATIVE_GRID_OK = (
    not native_grid_df.empty
    and native_grid_df["grid_compatible_with_msl"].all()
)

print("Existing native source grid acceptable:", CURRENT_NATIVE_GRID_OK)

if not CURRENT_NATIVE_GRID_OK:
    print(
        "\n⚠️ Existing native exports are not the paired source lattice. "
        "They will NOT be resampled back to 1 arc-second."
    )
    print(
        "The notebook will re-export DeltaDTM from the authoritative GEE mirror "
        "directly onto each selected MSL source tile grid."
    )

# 7 — Correct native source-grid acquisition — direct EE exact-grid method

This replaces the failed `geemap.download_ee_image()` approach.

## Why this is scientifically valid

The previous run established that:

- selected MSL tiles form one exact 1-arcsecond lattice;
- the GEE DeltaDTM source projection also has exactly 1-arcsecond pixels;
- their origins differ by integer numbers of source pixels.

So the source datasets already share one pixel lattice.

The correct fix is therefore **not** bilinear/cubic interpolation.

Instead, the notebook requests the native DeltaDTM values directly on the exact MSL affine grid.

### Earth Engine rule used here

Google Earth Engine documents that:

- `scale` alone can create a shifted pixel grid;
- `crs` + `crsTransform` provide full control over pixel alignment;
- `dimensions` defines the requested number of pixels;
- when `crs` + `crs_transform` are supplied to the direct-download API, a geographic
  `region` is not needed to define the output grid.

The direct API has a 32 MB limit, so each 3600 × 3600 tile is downloaded as four
1800 × 1800 exact-grid chunks and stitched without resampling.

Reference:
https://developers.google.com/earth-engine/apidocs/ee-image-getdownloadurl

Exact-grid export guidance:
https://developers.google.com/earth-engine/guides/exporting_images

In [ ]:
def token_from_name(name):
    match = re.search(
        r"([NS]\d{2}[EW]\d{3})",
        name.upper(),
    )
    return match.group(1) if match else None


def affine_to_ee(transform):
    """
    Rasterio Affine -> Earth Engine row-major 3x2 transform.

    [xScale, xShearing, xTranslation,
     yShearing, yScale, yTranslation]
    """
    return [
        float(transform.a),
        float(transform.b),
        float(transform.c),
        float(transform.d),
        float(transform.e),
        float(transform.f),
    ]


def transforms_close(a, b, atol=1e-12):
    return np.allclose(
        np.asarray(tuple(a), dtype=float),
        np.asarray(tuple(b), dtype=float),
        rtol=0,
        atol=atol,
    )


def exact_chunk_transform(base_transform, col_off, row_off):
    """
    Affine transform of a pixel-aligned subwindow.
    No interpolation is involved.
    """
    return base_transform * Affine.translation(
        int(col_off),
        int(row_off),
    )


def ee_direct_download_zip(image, params):
    """
    Direct Earth Engine Image.getDownloadURL request with retries.

    The API returns a ZIP containing the GeoTIFF because we use
    ZIPPED_GEO_TIFF. The whole request remains below the 32 MB
    direct-download ceiling by using small pixel windows.
    """
    last = None

    for attempt in range(1, EE_DIRECT_MAX_RETRIES + 1):
        try:
            url = image.getDownloadURL(params)

            response = requests.get(
                url,
                timeout=EE_DIRECT_TIMEOUT_SECONDS,
            )

            response.raise_for_status()

            payload = response.content

            if not payload:
                raise RuntimeError(
                    "Earth Engine returned an empty response."
                )

            return payload

        except Exception as exc:
            last = exc

            print(
                f"    direct EE attempt "
                f"{attempt}/{EE_DIRECT_MAX_RETRIES} failed: {exc}"
            )

            if attempt < EE_DIRECT_MAX_RETRIES:
                time.sleep(
                    min(2 ** attempt, 20)
                )

    raise RuntimeError(
        "Direct Earth Engine chunk download failed after retries.\n"
        f"Last error: {last}"
    )


def tif_bytes_from_zip(zip_bytes):
    """
    Extract the single GeoTIFF from EE's zipped download response.
    """
    with zipfile.ZipFile(
        io.BytesIO(zip_bytes),
        "r",
    ) as zf:

        tif_names = [
            name
            for name in zf.namelist()
            if name.lower().endswith(
                (".tif", ".tiff")
            )
        ]

        if len(tif_names) != 1:
            raise RuntimeError(
                "Expected exactly one GeoTIFF in the Earth Engine "
                f"download ZIP; found {len(tif_names)}."
            )

        return zf.read(
            tif_names[0]
        )


def download_exact_ee_chunk(
    image,
    crs_string,
    chunk_transform,
    chunk_width,
    chunk_height,
    label,
):
    """
    Request one exact pixel-grid chunk.

    IMPORTANT:
    - no `scale`
    - no geographic `region`
    - grid is fully defined by CRS + affine transform + dimensions
    """

    params = {
        "name": label,
        "bands": ["elevation"],
        "crs": crs_string,
        "crs_transform": affine_to_ee(
            chunk_transform
        ),
        "dimensions": [
            int(chunk_width),
            int(chunk_height),
        ],
        "format": "ZIPPED_GEO_TIFF",
        "filePerBand": False,
    }

    zip_payload = ee_direct_download_zip(
        image,
        params,
    )

    tif_payload = tif_bytes_from_zip(
        zip_payload
    )

    with rasterio.MemoryFile(
        tif_payload
    ) as mem:

        with mem.open() as src:

            returned = {
                "width": src.width,
                "height": src.height,
                "crs": src.crs,
                "transform": src.transform,
                "count": src.count,
            }

            expected_shape_ok = (
                src.width == int(chunk_width)
                and src.height == int(chunk_height)
            )

            expected_crs_ok = crs_equivalent(
                src.crs,
                crs_string,
            )

            expected_transform_ok = (
                transforms_close(
                    src.transform,
                    chunk_transform,
                    atol=1e-12,
                )
            )

            if STRICT_EXACT_CHUNK_VALIDATION:
                if not expected_shape_ok:
                    raise RuntimeError(
                        f"{label}: EE returned "
                        f"{src.width}×{src.height}; expected "
                        f"{chunk_width}×{chunk_height}."
                    )

                if not expected_crs_ok:
                    raise RuntimeError(
                        f"{label}: returned CRS does not match "
                        "the requested target CRS."
                    )

                if not expected_transform_ok:
                    raise RuntimeError(
                        f"{label}: returned affine transform does not "
                        "match the requested exact pixel grid.\n"
                        f"Expected: {tuple(chunk_transform)}\n"
                        f"Returned: {tuple(src.transform)}"
                    )

                if src.count != 1:
                    raise RuntimeError(
                        f"{label}: expected 1 band; returned {src.count}."
                    )

            arr = src.read(
                1
            ).astype(
                np.float32
            )

    return arr, returned


def create_exact_native_tile_from_msl(
    delta_native,
    msl_path,
    output_path,
):
    """
    Build one native DeltaDTM tile on the exact grid of one MSL tile.

    The full output raster is created locally from the MSL metadata.
    Earth Engine supplies pixel values in aligned chunks.

    There is no local raster resampling.
    """

    with rasterio.open(
        msl_path
    ) as msrc:

        desired_crs = msrc.crs
        desired_transform = msrc.transform
        desired_width = msrc.width
        desired_height = msrc.height

        profile = msrc.profile.copy()

    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=FLOAT_NODATA,
        compress="deflate",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        BIGTIFF="IF_SAFER",
    )

    if output_path.exists():
        output_path.unlink()

    crs_string = desired_crs.to_string()

    n_cols = math.ceil(
        desired_width
        / EE_EXACT_CHUNK_PIXELS
    )

    n_rows = math.ceil(
        desired_height
        / EE_EXACT_CHUNK_PIXELS
    )

    total_chunks = (
        n_cols
        * n_rows
    )

    token = token_from_name(
        msl_path.name
    ) or output_path.stem

    with rasterio.open(
        output_path,
        "w",
        **profile,
    ) as dst:

        chunk_bar = tqdm(
            total=total_chunks,
            desc=f"{token} exact EE chunks",
            unit="chunk",
            leave=False,
        )

        try:
            for row_off in range(
                0,
                desired_height,
                EE_EXACT_CHUNK_PIXELS,
            ):

                chunk_h = min(
                    EE_EXACT_CHUNK_PIXELS,
                    desired_height - row_off,
                )

                for col_off in range(
                    0,
                    desired_width,
                    EE_EXACT_CHUNK_PIXELS,
                ):

                    chunk_w = min(
                        EE_EXACT_CHUNK_PIXELS,
                        desired_width - col_off,
                    )

                    chunk_transform = (
                        exact_chunk_transform(
                            desired_transform,
                            col_off,
                            row_off,
                        )
                    )

                    label = (
                        f"{token}_"
                        f"r{row_off:04d}_"
                        f"c{col_off:04d}"
                    )

                    arr, returned = (
                        download_exact_ee_chunk(
                            image=delta_native,
                            crs_string=crs_string,
                            chunk_transform=chunk_transform,
                            chunk_width=chunk_w,
                            chunk_height=chunk_h,
                            label=label,
                        )
                    )

                    if arr.shape != (
                        chunk_h,
                        chunk_w,
                    ):
                        raise RuntimeError(
                            f"{label}: array shape {arr.shape} "
                            f"does not match requested "
                            f"{(chunk_h, chunk_w)}."
                        )

                    # Standardize non-finite pixels only.
                    invalid = ~np.isfinite(
                        arr
                    )

                    arr[
                        invalid
                    ] = FLOAT_NODATA

                    dst.write(
                        arr,
                        1,
                        window=windows.Window(
                            col_off=int(col_off),
                            row_off=int(row_off),
                            width=int(chunk_w),
                            height=int(chunk_h),
                        ),
                    )

                    chunk_bar.update(1)

        finally:
            chunk_bar.close()

    # --------------------------------------------------------
    # Full-tile validation against the MSL template
    # --------------------------------------------------------
    with (
        rasterio.open(
            output_path
        ) as nsrc,
        rasterio.open(
            msl_path
        ) as msrc,
    ):

        exact = (
            crs_equivalent(
                nsrc.crs,
                msrc.crs,
            )
            and transforms_close(
                nsrc.transform,
                msrc.transform,
                atol=1e-12,
            )
            and nsrc.width == msrc.width
            and nsrc.height == msrc.height
        )

        if not exact:
            raise RuntimeError(
                "\nExact-grid native tile failed final validation.\n"
                f"Native: {output_path.name}\n"
                f"MSL   : {msl_path.name}\n"
                f"Native grid: "
                f"{nsrc.width}×{nsrc.height}, "
                f"{tuple(nsrc.transform)}\n"
                f"MSL grid   : "
                f"{msrc.width}×{msrc.height}, "
                f"{tuple(msrc.transform)}"
            )

    return output_path


# ============================================================
# Initialize Earth Engine and verify source-lattice identity
# ============================================================

if AUTO_REPAIR_NATIVE_GRID or not CURRENT_NATIVE_GRID_OK:

    try:
        ee.Initialize(
            project=GEE_PROJECT
        )

    except Exception:
        print(
            "Starting Earth Engine authentication..."
        )

        ee.Authenticate(
            auth_mode="localhost"
        )

        ee.Initialize(
            project=GEE_PROJECT
        )

    if (
        ee.String(
            "VRFSZ_TERRAIN_V2"
        ).getInfo()
        != "VRFSZ_TERRAIN_V2"
    ):
        raise RuntimeError(
            "Earth Engine API probe failed."
        )

    delta_native = (
        ee.Image(
            NATIVE_GEE_ASSET
        )
        .select("b1")
        .rename("elevation")
        .toFloat()
    )

    gee_projection = (
        delta_native
        .projection()
        .getInfo()
    )

    gee_transform = (
        gee_projection
        .get("transform")
    )

    gee_crs = (
        gee_projection
        .get("crs")
    )

    if (
        not gee_transform
        or len(gee_transform) != 6
    ):
        raise RuntimeError(
            "The GEE DeltaDTM asset did not expose "
            "a usable source transform."
        )

    print(
        "GEE source CRS:",
        gee_crs,
    )

    print(
        "GEE source transform:",
        gee_transform,
    )

    gee_aff = Affine(
        *gee_transform
    )

    # --------------------------------------------------------
    # Scientific source-lattice test
    # --------------------------------------------------------
    source_resolution_same = np.allclose(
        [
            gee_aff.a,
            abs(gee_aff.e),
        ],
        [
            MSL_XRES,
            MSL_YRES,
        ],
        rtol=0,
        atol=1e-12,
    )

    dx = (
        MSL_REF_TRANSFORM.c
        - gee_aff.c
    ) / MSL_XRES

    dy = (
        gee_aff.f
        - MSL_REF_TRANSFORM.f
    ) / MSL_YRES

    source_origin_same_lattice = (
        abs(
            dx - round(dx)
        ) < 1e-7
        and abs(
            dy - round(dy)
        ) < 1e-7
    )

    source_lattice_df = pd.DataFrame([
        {
            "GEE_source_xres": gee_aff.a,
            "MSL_xres": MSL_XRES,
            "GEE_source_yres": abs(gee_aff.e),
            "MSL_yres": MSL_YRES,
            "resolution_identical": source_resolution_same,
            "x_origin_offset_pixels": dx,
            "y_origin_offset_pixels": dy,
            "integer_pixel_origin_offset": source_origin_same_lattice,
        }
    ])

    display(
        source_lattice_df
    )

    if not (
        source_resolution_same
        and source_origin_same_lattice
    ):
        raise RuntimeError(
            "\nThe native GEE source and MSL product are "
            "genuinely on different source lattices.\n"
            "Do not use the direct no-resampling repair."
        )

    print(
        "✅ Scientific source-lattice test PASS."
    )

    # Use a standardized NoData value in the direct request.
    export_image = (
        delta_native
        .unmask(
            FLOAT_NODATA
        )
    )

    exact_native_paths = []

    for msl_path in tqdm(
        selected_msl,
        desc="Native exact-grid tiles",
        unit="tile",
    ):

        token = token_from_name(
            msl_path.name
        )

        if token is None:
            raise RuntimeError(
                f"Could not recover tile token from "
                f"{msl_path.name}"
            )

        out = (
            EXACT_NATIVE_DIR
            / f"DeltaDTM_v1_1_"
              f"{token}_SOURCEGRID.tif"
        )

        # ----------------------------------------------------
        # Reuse only a previously verified exact file.
        # ----------------------------------------------------
        if out.exists():
            try:
                with (
                    rasterio.open(
                        out
                    ) as nsrc,
                    rasterio.open(
                        msl_path
                    ) as msrc,
                ):
                    exact = (
                        crs_equivalent(
                            nsrc.crs,
                            msrc.crs,
                        )
                        and transforms_close(
                            nsrc.transform,
                            msrc.transform,
                            atol=1e-12,
                        )
                        and nsrc.width == msrc.width
                        and nsrc.height == msrc.height
                    )

                if exact:
                    exact_native_paths.append(
                        out
                    )
                    continue

            except Exception:
                pass

            out.unlink(
                missing_ok=True
            )

        create_exact_native_tile_from_msl(
            delta_native=export_image,
            msl_path=msl_path,
            output_path=out,
        )

        exact_native_paths.append(
            out
        )

else:
    exact_native_paths = native_existing


print(
    "Exact-grid native source files:",
    len(exact_native_paths),
)

if len(exact_native_paths) != len(selected_msl):
    raise RuntimeError(
        "Exact-grid native count does not equal selected MSL tile count."
    )

print(
    "✅ Native source-grid repair/acquisition complete."
)

# 8 — Final source-pair sanity gate

Every selected MSL tile must now have exactly one native counterpart with:

- equivalent horizontal CRS;
- identical affine transform;
- identical width;
- identical height.

At this point an exact match is expected because the native tiles were assembled directly from
Earth Engine chunks requested on the MSL affine grid.

No interpolation-based repair is permitted after this gate.

In [ ]:
def exact_grid_signature(path):
    with rasterio.open(path) as src:
        return (
            str(horizontal_crs(src.crs)),
            tuple(src.transform),
            src.width,
            src.height,
        )


# Build native lookup by exact grid signature.
native_by_grid = defaultdict(list)

for p in exact_native_paths:
    native_by_grid[exact_grid_signature(p)].append(p)


pair_rows = []
SOURCE_PAIRS = []

for msl_path in selected_msl:
    sig_m = exact_grid_signature(msl_path)
    candidates = native_by_grid.get(sig_m, [])

    if len(candidates) != 1:
        raise RuntimeError(
            f"{msl_path.name} has {len(candidates)} exact native counterparts; expected 1."
        )

    native_path = candidates[0]

    SOURCE_PAIRS.append(
        (native_path, msl_path)
    )

    pair_rows.append({
        "native_file": native_path.name,
        "msl_file": msl_path.name,
        "exact_same_grid": True,
        "token_native": token_from_name(native_path.name),
        "token_msl": token_from_name(msl_path.name),
    })

source_pair_df = pd.DataFrame(pair_rows)
display(source_pair_df)

source_pair_csv = TERRAIN_REPORT / "source_pair_gate.csv"
source_pair_df.to_csv(source_pair_csv, index=False)

print("Pairs:", len(SOURCE_PAIRS))
print("Saved:", source_pair_csv)
print("✅ SOURCE-PAIR SANITY GATE PASS.")

# 9 — 02: Ingest and mosaic the paired GBM terrain

The output grid is created from the MSL/native one-arcsecond lattice and snapped to the GBM bounding box.

No continuous elevation resampling is performed.

Only exact aligned source pixels are copied into the common grid.
Pixels outside the frozen GBM polygon are set to NoData in both outputs.

In [ ]:
def snapped_grid_for_bounds(bounds, reference_transform):
    minx, miny, maxx, maxy = bounds

    xres = float(reference_transform.a)
    yres = float(abs(reference_transform.e))

    x0 = float(reference_transform.c)
    y0 = float(reference_transform.f)

    col_min = math.floor((minx - x0) / xres)
    col_max = math.ceil((maxx - x0) / xres)

    row_min = math.floor((y0 - maxy) / yres)
    row_max = math.ceil((y0 - miny) / yres)

    left = x0 + col_min * xres
    top = y0 - row_min * yres

    width = int(col_max - col_min)
    height = int(row_max - row_min)

    transform = Affine(
        xres, 0.0, left,
        0.0, -yres, top,
    )

    return transform, width, height


TARGET_TRANSFORM, TARGET_WIDTH, TARGET_HEIGHT = snapped_grid_for_bounds(
    AOI_GEOM.bounds,
    MSL_REF_TRANSFORM,
)

TARGET_CRS = MSL_REF_CRS

TARGET_BOUNDS = rasterio.transform.array_bounds(
    TARGET_HEIGHT,
    TARGET_WIDTH,
    TARGET_TRANSFORM,
)

print("Target CRS:", TARGET_CRS)
print("Target size:", TARGET_WIDTH, "×", TARGET_HEIGHT)
print("Target resolution:", TARGET_TRANSFORM.a, abs(TARGET_TRANSFORM.e))
print("Target bounds:", TARGET_BOUNDS)

In [ ]:
def create_blank_float_raster(path):
    profile = {
        "driver": "GTiff",
        "height": TARGET_HEIGHT,
        "width": TARGET_WIDTH,
        "count": 1,
        "dtype": "float32",
        "crs": TARGET_CRS,
        "transform": TARGET_TRANSFORM,
        "nodata": FLOAT_NODATA,
        "compress": "deflate",
        "tiled": True,
        "blockxsize": 512,
        "blockysize": 512,
        "BIGTIFF": "IF_SAFER",
    }

    with rasterio.open(path, "w", **profile) as dst:
        for _, window in tqdm(
            list(dst.block_windows(1)),
            desc=f"Initialize {Path(path).name}",
            unit="block",
            leave=False,
        ):
            shape = (
                int(window.height),
                int(window.width),
            )

            dst.write(
                np.full(
                    shape,
                    FLOAT_NODATA,
                    dtype=np.float32,
                ),
                1,
                window=window,
            )


def standardize_source_array(src, window):
    arr = src.read(
        1,
        window=window,
        masked=True,
    )

    data = np.asarray(
        arr.filled(FLOAT_NODATA),
        dtype=np.float32,
    )

    invalid = ~np.isfinite(data)

    if src.nodata is not None and np.isfinite(src.nodata):
        invalid |= np.isclose(
            data,
            float(src.nodata),
            rtol=0,
            atol=1e-6,
        )

    invalid |= np.isclose(
        data,
        FLOAT_NODATA,
        rtol=0,
        atol=1e-6,
    )

    data[invalid] = FLOAT_NODATA
    return data


def copy_aligned_source(src_path, dst):
    with rasterio.open(src_path) as src:
        if not crs_equivalent(src.crs, dst.crs):
            raise RuntimeError(
                f"Horizontal CRS mismatch: {src_path}"
            )

        if not np.allclose(
            [src.res[0], abs(src.res[1])],
            [dst.res[0], abs(dst.res[1])],
            rtol=0,
            atol=1e-12,
        ):
            raise RuntimeError(
                f"Resolution mismatch: {src_path}"
            )

        # Intersection between source and target.
        left = max(src.bounds.left, dst.bounds.left)
        right = min(src.bounds.right, dst.bounds.right)
        bottom = max(src.bounds.bottom, dst.bounds.bottom)
        top = min(src.bounds.top, dst.bounds.top)

        if left >= right or bottom >= top:
            return

        src_win = windows.from_bounds(
            left,
            bottom,
            right,
            top,
            transform=src.transform,
        ).round_offsets().round_lengths()

        dst_win = windows.from_bounds(
            left,
            bottom,
            right,
            top,
            transform=dst.transform,
        ).round_offsets().round_lengths()

        src_arr = standardize_source_array(
            src,
            src_win,
        )

        if src_arr.shape != (
            int(dst_win.height),
            int(dst_win.width),
        ):
            raise RuntimeError(
                f"Aligned copy shape mismatch for {src_path.name}: "
                f"{src_arr.shape} vs "
                f"{(int(dst_win.height), int(dst_win.width))}"
            )

        dst.write(
            src_arr,
            1,
            window=dst_win,
        )


def apply_aoi_mask(path):
    with rasterio.open(path, "r+") as dst:
        for _, window in tqdm(
            list(dst.block_windows(1)),
            desc=f"AOI mask {Path(path).name}",
            unit="block",
            leave=False,
        ):
            arr = dst.read(
                1,
                window=window,
            )

            transform = windows.transform(
                window,
                dst.transform,
            )

            inside = geometry_mask(
                [mapping(AOI_GEOM)],
                out_shape=arr.shape,
                transform=transform,
                invert=True,
                all_touched=False,
            )

            arr[~inside] = FLOAT_NODATA

            dst.write(
                arr,
                1,
                window=window,
            )


def build_mosaic(source_paths, output_path, label):
    if output_path.exists():
        output_path.unlink()

    create_blank_float_raster(
        output_path
    )

    with rasterio.open(
        output_path,
        "r+",
    ) as dst:
        for p in tqdm(
            source_paths,
            desc=label,
            unit="tile",
        ):
            copy_aligned_source(
                p,
                dst,
            )

    apply_aoi_mask(
        output_path
    )

    return output_path


native_sources = [
    pair[0]
    for pair in SOURCE_PAIRS
]

msl_sources = [
    pair[1]
    for pair in SOURCE_PAIRS
]

build_mosaic(
    native_sources,
    NATIVE_MOSAIC,
    "Mosaic native",
)

build_mosaic(
    msl_sources,
    MSL_MOSAIC,
    "Mosaic MSL",
)

print("Native mosaic:", NATIVE_MOSAIC)
print("MSL mosaic   :", MSL_MOSAIC)

# 10 — 03: Validate paired DEM grid and common footprint

Hard requirements:

- equivalent horizontal CRS;
- identical transform;
- identical width/height;
- identical pixel resolution and origin.

The full vertical CRS metadata are not required to be identical because the experiment intentionally
changes the vertical reference.

In [ ]:
with (
    rasterio.open(NATIVE_MOSAIC) as nsrc,
    rasterio.open(MSL_MOSAIC) as msrc,
):
    grid_qc = {
        "horizontal_crs_equivalent": crs_equivalent(
            nsrc.crs,
            msrc.crs,
        ),
        "transform_identical": nsrc.transform.almost_equals(
            msrc.transform,
            precision=12,
        ),
        "width_identical": nsrc.width == msrc.width,
        "height_identical": nsrc.height == msrc.height,
        "resolution_identical": np.allclose(
            [nsrc.res[0], abs(nsrc.res[1])],
            [msrc.res[0], abs(msrc.res[1])],
            rtol=0,
            atol=1e-12,
        ),
        "native_width": nsrc.width,
        "native_height": nsrc.height,
        "msl_width": msrc.width,
        "msl_height": msrc.height,
        "native_crs": str(nsrc.crs),
        "msl_crs": str(msrc.crs),
        "pixel_size_x": nsrc.res[0],
        "pixel_size_y": abs(nsrc.res[1]),
    }

grid_qc_df = pd.DataFrame([grid_qc])
display(grid_qc_df)

hard_grid_cols = [
    "horizontal_crs_equivalent",
    "transform_identical",
    "width_identical",
    "height_identical",
    "resolution_identical",
]

if not grid_qc_df[hard_grid_cols].all().all():
    raise RuntimeError(
        "PAIRED DEM GRID GATE FAILED. "
        "Do not calculate ΔZ from misaligned terrain."
    )

grid_qc_df.to_csv(
    TERRAIN_REPORT / "paired_grid_qc.csv",
    index=False,
)

print("✅ PAIRED DEM GRID GATE PASS.")

In [ ]:
common_valid_pixels = 0
native_valid_pixels = 0
msl_valid_pixels = 0
aoi_pixels = 0

with (
    rasterio.open(NATIVE_MOSAIC) as nsrc,
    rasterio.open(MSL_MOSAIC) as msrc,
):
    profile = nsrc.profile.copy()

    valid_profile = profile.copy()
    valid_profile.update(
        dtype="uint8",
        nodata=0,
        compress="deflate",
    )

    with rasterio.open(
        COMMON_VALID,
        "w",
        **valid_profile,
    ) as vdst:

        for _, window in tqdm(
            list(nsrc.block_windows(1)),
            desc="Common valid footprint",
            unit="block",
        ):
            n = nsrc.read(
                1,
                window=window,
            )

            m = msrc.read(
                1,
                window=window,
            )

            native_valid = (
                np.isfinite(n)
                & (~np.isclose(n, FLOAT_NODATA))
            )

            msl_valid = (
                np.isfinite(m)
                & (~np.isclose(m, FLOAT_NODATA))
            )

            common = native_valid & msl_valid

            # The mosaic was already masked to AOI; a pixel is in-AOI if
            # at least one surface is not the standardized outside-AOI nodata.
            inside = native_valid | msl_valid

            native_valid_pixels += int(native_valid.sum())
            msl_valid_pixels += int(msl_valid.sum())
            common_valid_pixels += int(common.sum())
            aoi_pixels += int(inside.sum())

            vdst.write(
                common.astype("uint8"),
                1,
                window=window,
            )

common_fraction = (
    common_valid_pixels
    / max(aoi_pixels, 1)
)

footprint_df = pd.DataFrame([{
    "native_valid_pixels": native_valid_pixels,
    "msl_valid_pixels": msl_valid_pixels,
    "common_valid_pixels": common_valid_pixels,
    "union_valid_pixels": aoi_pixels,
    "common_fraction_of_union": common_fraction,
}])

display(footprint_df)

footprint_df.to_csv(
    TERRAIN_REPORT / "paired_valid_footprint.csv",
    index=False,
)

if common_valid_pixels == 0:
    raise RuntimeError(
        "Native and MSL terrain have zero common valid pixels."
    )

if common_fraction < WARN_COMMON_VALID_FRACTION:
    print(
        "⚠️ Common valid coverage is below the warning threshold:",
        f"{100*common_fraction:.2f}%",
    )

print("✅ Common valid terrain footprint created.")

# 11 — 04: Compute vertical-reference displacement

For every common valid terrain pixel:

\[
\Delta Z = Z_N - Z_M.
\]

The output is calculated window-by-window to avoid loading the complete GBM terrain into RAM.

Full count/mean/standard deviation/min/max are accumulated over all valid pixels.  
Percentiles are estimated from a deterministic spatial sample and explicitly labelled as sampled
percentiles.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

sample_chunks = []

count = 0
sum_x = 0.0
sum_x2 = 0.0
min_x = np.inf
max_x = -np.inf

with (
    rasterio.open(NATIVE_MOSAIC) as nsrc,
    rasterio.open(MSL_MOSAIC) as msrc,
):
    profile = nsrc.profile.copy()
    profile.update(
        dtype="float32",
        nodata=FLOAT_NODATA,
        compress="deflate",
    )

    windows_list = [
        w
        for _, w
        in nsrc.block_windows(1)
    ]

    # Approximate per-window sampling cap so the final sample stays compact.
    per_window_cap = max(
        100,
        int(
            math.ceil(
                MAX_DZ_SAMPLE_VALUES
                / max(len(windows_list), 1)
            )
        ),
    )

    with rasterio.open(
        DELTA_Z,
        "w",
        **profile,
    ) as ddst:

        for window in tqdm(
            windows_list,
            desc="Compute ΔZ",
            unit="block",
        ):
            n = nsrc.read(
                1,
                window=window,
            ).astype(np.float64)

            m = msrc.read(
                1,
                window=window,
            ).astype(np.float64)

            valid = (
                np.isfinite(n)
                & np.isfinite(m)
                & (~np.isclose(n, FLOAT_NODATA))
                & (~np.isclose(m, FLOAT_NODATA))
            )

            out = np.full(
                n.shape,
                FLOAT_NODATA,
                dtype=np.float32,
            )

            if valid.any():
                dz = n[valid] - m[valid]

                out[valid] = dz.astype(
                    np.float32
                )

                count += int(dz.size)
                sum_x += float(dz.sum())
                sum_x2 += float(np.square(dz).sum())
                min_x = min(min_x, float(dz.min()))
                max_x = max(max_x, float(dz.max()))

                if dz.size > per_window_cap:
                    idx = rng.choice(
                        dz.size,
                        size=per_window_cap,
                        replace=False,
                    )
                    sample_chunks.append(
                        dz[idx].astype(np.float32)
                    )
                else:
                    sample_chunks.append(
                        dz.astype(np.float32)
                    )

            ddst.write(
                out,
                1,
                window=window,
            )

if count == 0:
    raise RuntimeError("ΔZ has zero valid pixels.")

mean_x = sum_x / count
variance = max(
    (sum_x2 / count) - mean_x**2,
    0.0,
)
std_x = math.sqrt(variance)

dz_sample = np.concatenate(
    sample_chunks
)

if dz_sample.size > MAX_DZ_SAMPLE_VALUES:
    idx = rng.choice(
        dz_sample.size,
        size=MAX_DZ_SAMPLE_VALUES,
        replace=False,
    )
    dz_sample = dz_sample[idx]

delta_summary = pd.DataFrame([{
    "valid_pixels": count,
    "mean_m": mean_x,
    "std_m": std_x,
    "min_m": min_x,
    "max_m": max_x,
    "sample_n_for_percentiles": int(dz_sample.size),
    "sample_p01_m": float(np.percentile(dz_sample, 1)),
    "sample_p05_m": float(np.percentile(dz_sample, 5)),
    "sample_p25_m": float(np.percentile(dz_sample, 25)),
    "sample_median_m": float(np.percentile(dz_sample, 50)),
    "sample_p75_m": float(np.percentile(dz_sample, 75)),
    "sample_p95_m": float(np.percentile(dz_sample, 95)),
    "sample_p99_m": float(np.percentile(dz_sample, 99)),
}])

display(delta_summary)

delta_summary.to_csv(
    TERRAIN_REPORT / "delta_z_summary.csv",
    index=False,
)

print("ΔZ raster:", DELTA_Z)
print("✅ ΔZ computation complete.")

## 11.1 ΔZ diagnostic visualization

This is QA only. It does not change any threshold or result.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(
    dz_sample,
    bins=100,
)
ax.set_xlabel("ΔZ = native − MSL (m)")
ax.set_ylabel("Sampled pixel count")
ax.set_title("Sampled vertical-reference displacement distribution")
plt.tight_layout()
plt.show()

# 12 — 05: Primary VRFSZ classification at \(T=1.0\) m

Class codes:

| Code | Class | Logical rule |
|---:|---|---|
| 0 | NoData | no common paired terrain |
| 1 | StableLow | \(Z_N\le1,\ Z_M\le1\) |
| 2 | VRFSZ | \(Z_N>1,\ Z_M\le1\) |
| 3 | StableHigh | \(Z_N>1,\ Z_M>1\) |
| 4 | Reverse | \(Z_N\le1,\ Z_M>1\) |

The fourth state is preserved as a diagnostic instead of being hidden.

In [ ]:
CLASS_NAMES = {
    1: "StableLow",
    2: "VRFSZ",
    3: "StableHigh",
    4: "Reverse",
}

geod = Geod(ellps="WGS84")

class_counts = defaultdict(int)
class_area_m2 = defaultdict(float)

row_area_cache = {}


def pixel_area_for_row(global_row):
    if global_row in row_area_cache:
        return row_area_cache[global_row]

    top = (
        TARGET_TRANSFORM.f
        + global_row * TARGET_TRANSFORM.e
    )
    bottom = top + TARGET_TRANSFORM.e

    left = TARGET_TRANSFORM.c
    right = left + TARGET_TRANSFORM.a

    area, _ = geod.polygon_area_perimeter(
        [left, right, right, left],
        [top, top, bottom, bottom],
    )

    area = abs(area)
    row_area_cache[global_row] = area
    return area


with (
    rasterio.open(NATIVE_MOSAIC) as nsrc,
    rasterio.open(MSL_MOSAIC) as msrc,
):
    profile = nsrc.profile.copy()
    profile.update(
        dtype="uint8",
        nodata=CLASS_NODATA,
        compress="deflate",
    )

    with rasterio.open(
        CLASS_T1,
        "w",
        **profile,
    ) as cdst:

        for _, window in tqdm(
            list(nsrc.block_windows(1)),
            desc="Classify T=1 m",
            unit="block",
        ):
            n = nsrc.read(
                1,
                window=window,
            )

            m = msrc.read(
                1,
                window=window,
            )

            valid = (
                np.isfinite(n)
                & np.isfinite(m)
                & (~np.isclose(n, FLOAT_NODATA))
                & (~np.isclose(m, FLOAT_NODATA))
            )

            cls = np.zeros(
                n.shape,
                dtype=np.uint8,
            )

            stable_low = (
                valid
                & (n <= PRIMARY_THRESHOLD_M)
                & (m <= PRIMARY_THRESHOLD_M)
            )

            vrfsz = (
                valid
                & (n > PRIMARY_THRESHOLD_M)
                & (m <= PRIMARY_THRESHOLD_M)
            )

            stable_high = (
                valid
                & (n > PRIMARY_THRESHOLD_M)
                & (m > PRIMARY_THRESHOLD_M)
            )

            reverse = (
                valid
                & (n <= PRIMARY_THRESHOLD_M)
                & (m > PRIMARY_THRESHOLD_M)
            )

            cls[stable_low] = 1
            cls[vrfsz] = 2
            cls[stable_high] = 3
            cls[reverse] = 4

            cdst.write(
                cls,
                1,
                window=window,
            )

            # Pixel counts.
            for code_value in CLASS_NAMES:
                class_counts[code_value] += int(
                    np.count_nonzero(
                        cls == code_value
                    )
                )

            # Geodesic area by raster row.
            row0 = int(window.row_off)

            for local_row in range(
                int(window.height)
            ):
                global_row = row0 + local_row
                px_area = pixel_area_for_row(
                    global_row
                )

                row_values = cls[
                    local_row,
                    :
                ]

                for code_value in CLASS_NAMES:
                    npx = int(
                        np.count_nonzero(
                            row_values
                            == code_value
                        )
                    )

                    if npx:
                        class_area_m2[
                            code_value
                        ] += (
                            npx
                            * px_area
                        )


valid_class_total = sum(
    class_counts.values()
)

if valid_class_total != common_valid_pixels:
    raise RuntimeError(
        "Primary class partition does not equal the common valid terrain count."
    )

class_rows = []

for code_value, name in CLASS_NAMES.items():
    class_rows.append({
        "code": code_value,
        "class": name,
        "pixels": class_counts[code_value],
        "percent_of_common_valid": (
            100.0
            * class_counts[code_value]
            / max(valid_class_total, 1)
        ),
        "area_km2_geodesic": (
            class_area_m2[code_value]
            / 1_000_000.0
        ),
    })

class_stats_t1 = pd.DataFrame(
    class_rows
)

display(class_stats_t1)

class_stats_t1.to_csv(
    TERRAIN_REPORT / "vrfsz_T1_class_stats.csv",
    index=False,
)

legend = {
    "0": "NoData / no paired terrain",
    "1": "StableLow",
    "2": "VRFSZ",
    "3": "StableHigh",
    "4": "Reverse diagnostic",
}

(TERRAIN_REPORT / "vrfsz_class_legend.json").write_text(
    json.dumps(
        legend,
        indent=2,
    ),
    encoding="utf-8",
)

if class_counts[2] == 0:
    raise RuntimeError(
        "Primary threshold produced ZERO VRFSZ pixels."
    )

if class_counts[3] == 0:
    raise RuntimeError(
        "Primary threshold produced ZERO StableHigh pixels."
    )

print("Primary class raster:", CLASS_T1)
print("✅ T=1 m classification complete.")

## 12.1 Lightweight class preview

The display is diagnostic only; the GeoTIFF is the analysis product.

In [ ]:
with rasterio.open(CLASS_T1) as src:
    factor = max(
        src.width / 1200,
        src.height / 1200,
        1,
    )

    out_w = max(
        1,
        int(src.width / factor),
    )

    out_h = max(
        1,
        int(src.height / factor),
    )

    preview = src.read(
        1,
        out_shape=(out_h, out_w),
        resampling=rasterio.enums.Resampling.nearest,
    )

fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(
    np.ma.masked_where(
        preview == 0,
        preview,
    ),
    interpolation="nearest",
)
ax.set_title("Primary terrain classification — T = 1 m")
ax.set_axis_off()
plt.tight_layout()
plt.show()

# 13 — 18A3: Terrain threshold viability at 0.5, 1.0 and 2.0 m

This does **not** create the final sensitivity rasters yet.

It simply verifies that the later statistical analysis is mathematically possible at each
pre-registered threshold and reports the diagnostic Reverse state.

In [ ]:
threshold_counts = {
    T: defaultdict(int)
    for T in READINESS_THRESHOLDS_M
}

with (
    rasterio.open(NATIVE_MOSAIC) as nsrc,
    rasterio.open(MSL_MOSAIC) as msrc,
):
    for _, window in tqdm(
        list(nsrc.block_windows(1)),
        desc="Threshold viability",
        unit="block",
    ):
        n = nsrc.read(
            1,
            window=window,
        )

        m = msrc.read(
            1,
            window=window,
        )

        valid = (
            np.isfinite(n)
            & np.isfinite(m)
            & (~np.isclose(n, FLOAT_NODATA))
            & (~np.isclose(m, FLOAT_NODATA))
        )

        for T in READINESS_THRESHOLDS_M:
            threshold_counts[T]["valid"] += int(
                valid.sum()
            )

            threshold_counts[T]["stable_low"] += int(
                np.count_nonzero(
                    valid
                    & (n <= T)
                    & (m <= T)
                )
            )

            threshold_counts[T]["vrfsz"] += int(
                np.count_nonzero(
                    valid
                    & (n > T)
                    & (m <= T)
                )
            )

            threshold_counts[T]["stable_high"] += int(
                np.count_nonzero(
                    valid
                    & (n > T)
                    & (m > T)
                )
            )

            threshold_counts[T]["reverse"] += int(
                np.count_nonzero(
                    valid
                    & (n <= T)
                    & (m > T)
                )
            )


viability_rows = []

for T in READINESS_THRESHOLDS_M:
    c = threshold_counts[T]
    denom = max(c["valid"], 1)

    viability_rows.append({
        "threshold_m": T,
        "valid_pixels": c["valid"],
        "stable_low_pixels": c["stable_low"],
        "stable_low_pct": 100*c["stable_low"]/denom,
        "vrfsz_pixels": c["vrfsz"],
        "vrfsz_pct": 100*c["vrfsz"]/denom,
        "stable_high_pixels": c["stable_high"],
        "stable_high_pct": 100*c["stable_high"]/denom,
        "reverse_pixels": c["reverse"],
        "reverse_pct": 100*c["reverse"]/denom,
    })

viability_df = pd.DataFrame(
    viability_rows
)

display(viability_df)

viability_df.to_csv(
    TERRAIN_REPORT / "threshold_viability_0p5_1p0_2p0m.csv",
    index=False,
)

for row in viability_rows:
    if row["vrfsz_pixels"] == 0:
        raise RuntimeError(
            f"T={row['threshold_m']} m has zero VRFSZ pixels."
        )

    if row["stable_high_pixels"] == 0:
        raise RuntimeError(
            f"T={row['threshold_m']} m has zero StableHigh pixels."
        )

    if row["reverse_pixels"] > 0:
        print(
            f"⚠️ T={row['threshold_m']} m Reverse pixels: "
            f"{row['reverse_pixels']:,} "
            f"({row['reverse_pct']:.6f}%)."
        )

print("✅ Threshold viability PASS.")

# 14 — 18A3: Re-audit the finished Sentinel-1 recurrence branch

This is a lightweight integrity check before the terrain and flood branches are joined in the next notebook.

In [ ]:
def grid_signature(path):
    with rasterio.open(path) as src:
        return {
            "crs": str(src.crs),
            "transform": tuple(src.transform),
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "nodata": src.nodata,
            "bounds": tuple(src.bounds),
        }


obs_grid_paths = [
    FLOOD_STACK,
    VALID_STACK,
    FLOOD_COUNT,
    OBS_COUNT,
    RECURRENCE,
]

obs_grid_df = pd.DataFrame([
    {
        "file": p.name,
        **grid_signature(p),
    }
    for p in obs_grid_paths
])

display(obs_grid_df)

with (
    rasterio.open(FLOOD_STACK) as fsrc,
    rasterio.open(VALID_STACK) as vsrc,
):
    if (
        fsrc.crs != vsrc.crs
        or fsrc.transform != vsrc.transform
        or fsrc.width != vsrc.width
        or fsrc.height != vsrc.height
        or fsrc.count != vsrc.count
    ):
        raise RuntimeError(
            "Flood/valid stacks no longer share one grid."
        )

    if fsrc.count != len(EVENT_IDS):
        raise RuntimeError(
            "Flood/valid stack band count does not match event_index.csv."
        )

print("✅ Sentinel-1 recurrence grid integrity PASS.")

In [ ]:
frozen_sar = json.loads(
    FROZEN_SAR_CONFIG.read_text(
        encoding="utf-8"
    )
)

frozen_sha = frozen_sar.get(
    "config_sha256"
)

event_meta_rows = []

for event_id in EVENT_IDS:
    meta_path = (
        EVENT_ROOT
        / event_id
        / "event_flood_metadata.json"
    )

    if not meta_path.exists():
        raise RuntimeError(
            f"Missing event metadata: {event_id}"
        )

    meta = json.loads(
        meta_path.read_text(
            encoding="utf-8"
        )
    )

    event_meta_rows.append({
        "event_id": event_id,
        "frozen_sha_match": (
            meta.get("frozen_config_sha256")
            == frozen_sha
        ),
        "otsu_threshold_db": meta.get(
            "event_otsu_threshold_db"
        ),
        "flood_fraction": meta.get(
            "flood_fraction"
        ),
        "documented_reference": meta.get(
            "documented_event_reference"
        ),
    })

event_meta_df = pd.DataFrame(
    event_meta_rows
)

display(event_meta_df)

if not event_meta_df[
    "frozen_sha_match"
].all():
    raise RuntimeError(
        "At least one event does not match the frozen SAR workflow."
    )

if event_meta_df[
    "documented_reference"
].isna().any():
    print(
        "⚠️ Documentary-reference metadata is incomplete for:",
        event_meta_df.loc[
            event_meta_df[
                "documented_reference"
            ].isna(),
            "event_id",
        ].tolist(),
    )

print("✅ Frozen Sentinel-1 method consistency PASS.")

# 15 — 18A3: Terrain / recurrence spatial compatibility

The two branches are allowed to have different resolutions and CRSs.

The later statistical notebook will transfer **categorical terrain membership** to the
Sentinel-1 recurrence grid using nearest-neighbour only.

Here we check only that their geographic coverage is compatible.

In [ ]:
with (
    rasterio.open(NATIVE_MOSAIC) as dsrc,
    rasterio.open(RECURRENCE) as rsrc,
):
    terrain_in_rec_crs = transform_bounds(
        dsrc.crs,
        rsrc.crs,
        *dsrc.bounds,
        densify_pts=21,
    )

    rb = rsrc.bounds

tx0, ty0, tx1, ty1 = terrain_in_rec_crs
rx0, ry0, rx1, ry1 = rb

ix0 = max(tx0, rx0)
iy0 = max(ty0, ry0)
ix1 = min(tx1, rx1)
iy1 = min(ty1, ry1)

intersection = (
    max(0.0, ix1 - ix0)
    * max(0.0, iy1 - iy0)
)

rec_area_box = (
    max(0.0, rx1 - rx0)
    * max(0.0, ry1 - ry0)
)

bounds_coverage = (
    intersection
    / max(rec_area_box, 1e-12)
)

print(
    "Terrain bounds coverage of recurrence extent:",
    f"{100*bounds_coverage:.4f}%",
)

if bounds_coverage < MIN_BOUNDS_COVERAGE:
    raise RuntimeError(
        "Terrain extent does not sufficiently cover the recurrence extent."
    )

print("✅ Terrain/recurrence bounds compatibility PASS.")

# 16 — Phase-A provenance and integrity record

This is not the final paper freeze; it records exactly what entered the future statistics.

In [ ]:
def sha256_file(path, chunk_size=8*1024*1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


phase_a_files = [
    NATIVE_MOSAIC,
    MSL_MOSAIC,
    COMMON_VALID,
    DELTA_Z,
    CLASS_T1,
]

phase_a_hash_rows = []

for p in tqdm(
    phase_a_files,
    desc="Hash Phase-A outputs",
    unit="file",
):
    phase_a_hash_rows.append({
        "file": p.name,
        "path": str(p),
        "size_MB": round(
            p.stat().st_size / 1024**2,
            2,
        ),
        "sha256": sha256_file(p),
    })

phase_a_hash_df = pd.DataFrame(
    phase_a_hash_rows
)

display(phase_a_hash_df)

phase_a_hash_df.to_csv(
    TERRAIN_REPORT / "phaseA_output_hashes.csv",
    index=False,
)

# Record exact-grid extraction method.
exact_grid_method = {
    "method": (
        "Earth Engine Image.getDownloadURL exact-grid chunk extraction"
    ),
    "native_asset": NATIVE_GEE_ASSET,
    "target_grid_source": (
        "DeltaDTM v1.1 MSL source tiles"
    ),
    "crs_transform_used": True,
    "dimensions_used": True,
    "scale_parameter_used": False,
    "region_parameter_used_for_grid": False,
    "chunk_pixels": EE_EXACT_CHUNK_PIXELS,
    "local_continuous_resampling": False,
    "reason": (
        "Avoid helper-induced crop/grid mismatch while preserving "
        "the common 1-arcsecond DeltaDTM source lattice."
    ),
    "earth_engine_docs": [
        "https://developers.google.com/earth-engine/apidocs/ee-image-getdownloadurl",
        "https://developers.google.com/earth-engine/guides/exporting_images",
    ],
}

(
    TERRAIN_REPORT
    / "native_exact_grid_extraction_method.json"
).write_text(
    json.dumps(
        exact_grid_method,
        indent=2,
    ),
    encoding="utf-8",
)


# 17 — FINAL TECHNICAL READINESS GATE

A PASS means the project is ready to start:

```text
18B Class-wise Recurrence
        ↓
19 Relative Risk + Spatial Block Bootstrap
        ↓
20 Threshold Sensitivity 0.5 / 1.0 / 2.0 m
        ↓
21 Final Results Freeze
```

A PASS does **not** mean the hypothesis is supported.  
It means only that the data architecture is scientifically ready to test it.

In [ ]:
final_checks = {
    "source_pair_exact_grid": True,
    "paired_mosaic_grid_pass": bool(
        grid_qc_df[
            hard_grid_cols
        ].all().all()
    ),
    "common_valid_pixels_positive": (
        common_valid_pixels > 0
    ),
    "delta_z_created": DELTA_Z.exists(),
    "primary_T1_class_created": CLASS_T1.exists(),
    "primary_vrfsz_nonzero": (
        class_counts[2] > 0
    ),
    "primary_stable_high_nonzero": (
        class_counts[3] > 0
    ),
    "all_sensitivity_thresholds_viable": all(
        row["vrfsz_pixels"] > 0
        and row["stable_high_pixels"] > 0
        for row in viability_rows
    ),
    "four_or_more_events": (
        len(EVENT_IDS) >= MIN_EVENTS
    ),
    "frozen_sar_consistent": bool(
        event_meta_df[
            "frozen_sha_match"
        ].all()
    ),
    "terrain_recurrence_bounds_compatible": (
        bounds_coverage
        >= MIN_BOUNDS_COVERAGE
    ),
}

final_check_df = pd.DataFrame([
    {
        "check": key,
        "pass": value,
    }
    for key, value
    in final_checks.items()
])

display(final_check_df)

TECHNICAL_READY = all(
    final_checks.values()
)

readiness = {
    "technical_ready_for_18B_to_21": TECHNICAL_READY,
    "project_root": str(PROJECT_ROOT),
    "native_mosaic": str(NATIVE_MOSAIC),
    "msl_mosaic": str(MSL_MOSAIC),
    "delta_z": str(DELTA_Z),
    "primary_class_T1": str(CLASS_T1),
    "primary_threshold_m": PRIMARY_THRESHOLD_M,
    "readiness_thresholds_m": READINESS_THRESHOLDS_M,
    "common_valid_fraction_of_union": common_fraction,
    "terrain_recurrence_bounds_coverage": bounds_coverage,
    "events": EVENT_IDS,
    "frozen_sar_sha": frozen_sha,
    "reverse_pixels_T1": int(
        class_counts[4]
    ),
    "event_metadata_missing_documented_reference": event_meta_df.loc[
        event_meta_df[
            "documented_reference"
        ].isna(),
        "event_id",
    ].tolist(),
    "checks": final_checks,
}

readiness_path = (
    TERRAIN_REPORT
    / "phaseA_to_18A3_readiness.json"
)

readiness_path.write_text(
    json.dumps(
        readiness,
        indent=2,
    ),
    encoding="utf-8",
)

if not TECHNICAL_READY:
    raise RuntimeError(
        "FINAL TECHNICAL READINESS GATE FAILED."
    )

print("\n" + "=" * 84)
print("✅ TECHNICAL READINESS: PASS")
print("=" * 84)
print(
    "The terrain branch and independent flood branch are ready "
    "to be joined in the final statistical workflow."
)
print("\nNext notebook:")
print(
    "18B→21 combined class recurrence / RR / bootstrap / "
    "sensitivity / final-freeze notebook."
)
print("\nReadiness record:", readiness_path)

# 18 — Safe cleanup of superseded acquisition artifacts

Cleanup happens only after the final readiness gate has passed.

The exact source-grid native files are preserved by default.  
Only the old `scale=30` native GEE exports are eligible for automatic deletion.

In [ ]:
if TECHNICAL_READY and DELETE_SUPERSEDED_NATIVE_AFTER_PASS:
    deleted = []
    kept = []

    exact_set = {
        p.resolve()
        for p in exact_native_paths
    }

    for p in native_existing:
        if p.resolve() in exact_set:
            kept.append(p)
            continue

        # The old direct files are scientifically superseded if their
        # grids are not compatible with the paired MSL source lattice.
        if not source_grid_aligned_to_msl(p):
            size = p.stat().st_size
            p.unlink()
            deleted.append(
                (p.name, size)
            )
        else:
            kept.append(p)

    print("Superseded native files deleted:", len(deleted))
    print(
        "Space recovered:",
        round(
            sum(size for _, size in deleted)
            / 1024**2,
            2,
        ),
        "MB",
    )

else:
    print("No superseded native cleanup performed.")


if TECHNICAL_READY and DELETE_NON_AOI_MSL_TILES:
    selected_set = {
        p.resolve()
        for p in selected_msl
    }

    removed = []

    for p in msl_all:
        if p.resolve() not in selected_set:
            size = p.stat().st_size
            p.unlink()
            removed.append(
                (p.name, size)
            )

    print("Non-AOI MSL files deleted:", len(removed))
    print(
        "MSL space recovered:",
        round(
            sum(size for _, size in removed)
            / 1024**2,
            2,
        ),
        "MB",
    )


usage = shutil.disk_usage(
    PROJECT_ROOT
)

print(
    "Free disk after Phase A:",
    round(
        usage.free / 1024**3,
        2,
    ),
    "GB",
)

# End of combined Phase-A notebook

## Permanent outputs

```text
data/processed/terrain/
├── DeltaDTM_v1_1_native_GBM.tif
├── DeltaDTM_v1_1_MSL_GBM.tif
├── terrain_common_valid_mask.tif
├── delta_z_native_minus_msl_GBM.tif
└── vrfsz_classes_T1m.tif
```

Reports:

```text
reports/terrain_phase/
├── source_pair_gate.csv
├── paired_grid_qc.csv
├── paired_valid_footprint.csv
├── delta_z_summary.csv
├── vrfsz_T1_class_stats.csv
├── vrfsz_class_legend.json
├── threshold_viability_0p5_1p0_2p0m.csv
├── phaseA_output_hashes.csv
└── phaseA_to_18A3_readiness.json
```

If the final cell reports:

```text
✅ TECHNICAL READINESS: PASS
```

do not perform more terrain tuning.

Proceed to the final statistical workflow.

Additional v2 provenance:

```text
reports/terrain_phase/
└── native_exact_grid_extraction_method.json
```

The source-grid repair uses **zero local interpolation**.


# STAGE 3 — Full-coverage Sentinel-1 / EO acquisition

**Source provenance:** `C1 VRFSZ_GBM_FullCoverage_Acquisition_v3.ipynb`.

Flood-event selection is independent of the terrain result.

# VRFSZ–GBM Full-Coverage Earth Observation Acquisition — v3

This notebook replaces the earlier **single-Sentinel-1-track** acquisition logic.

Visual QC of the previous run showed that the selected relative orbit covered only the western
part of the GBM Delta even though the exported GeoTIFF had the full AOI bounding-box dimensions.
This version therefore makes **actual valid AOI coverage a mandatory scientific success criterion**.

## Corrected Sentinel-1 workflow

1. Search all Sentinel-1 IW VV/VH acquisitions in the event window.
2. Group by date + orbit direction + relative orbit.
3. Calculate the actual valid GBM coverage of every candidate track/date.
4. Test combinations of complementary relative-orbit tracks.
5. Select the **minimum number of tracks** reaching the target AOI coverage.
6. Allow complementary tracks on nearby dates when no same-date coverage exists.
7. For each selected event track, select a pre-event acquisition from the **same pass and same relative orbit**.
8. Evaluate common pre ∩ event coverage for candidate pairs.
9. Build pair-aware images containing pre and event bands together.
10. Merge those pair-images into a single GBM-wide product.
11. Export provenance showing which relative orbit/date supplies each pixel.
12. Reject the product if final common coverage is below the target.

Sentinel-2 remains optional QA. JRC Global Surface Water remains the permanent-water exclusion
dataset.

## Step 0A — Optional package installation

In [ ]:
# Run only if needed, then restart the kernel.
# %pip install -U earthengine-api geemap geedim geopandas rasterio shapely \
#     pyproj pandas numpy requests tqdm matplotlib affine

## Step 0B — FEASIBILITY: Python environment

In [ ]:
import importlib.util
import pandas as pd
from tqdm.auto import tqdm

REQUIRED_MODULES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "requests": "requests",
    "geopandas": "geopandas",
    "rasterio": "rasterio",
    "shapely": "shapely",
    "pyproj": "pyproj",
    "tqdm": "tqdm",
    "earthengine-api": "ee",
    "geemap": "geemap",
    "geedim": "geedim",
    "matplotlib": "matplotlib",
    "affine": "affine",
}

rows = []
for package, module in tqdm(REQUIRED_MODULES.items(), desc="Checking packages", unit="pkg"):
    rows.append({
        "package": package,
        "import_name": module,
        "available": importlib.util.find_spec(module) is not None,
    })

env_check = pd.DataFrame(rows)
display(env_check)

missing = env_check.loc[~env_check["available"], "package"].tolist()
if missing:
    raise RuntimeError("Missing packages: " + ", ".join(missing))

print("✅ Environment feasibility PASS")

# Step 1 — EDIT THIS CELL: Flood-event dates

The pre-event window is generated automatically unless you override both pre-event dates.

In [ ]:
EVENT_ID = "EVENT001"

EVENT_START = "2026-07-10"
EVENT_END   = "2026-07-30"

PRE_EVENT_LOOKBACK_DAYS = 45
PRE_EVENT_GAP_DAYS = 5

PRE_EVENT_START = None
PRE_EVENT_END = None

# None = midpoint of EVENT_START / EVENT_END
EVENT_REFERENCE_DATE = None

print("✅ Event configuration:", EVENT_ID, EVENT_START, "→", EVENT_END)

# Step 2 — EDIT THIS CELL: AOI, GEE project and scientific controls

In [ ]:
import os

AOI_PATH = "data/raw/boundaries/gbm_delta.shp"
AOI_BBOX = None

GEE_PROJECT = "ee-tarin1"
GEE_AUTH_MODE = "localhost"
FORCE_GEE_REAUTH = False

DOWNLOAD_SENTINEL1 = True
DOWNLOAD_SENTINEL2 = True
DOWNLOAD_JRC_WATER = True

# ---------------- Sentinel-1 full-coverage rules ----------------
S1_SCALE_M = 10
S1_COVERAGE_CHECK_SCALE_M = 1000
S1_TARGET_COVERAGE_PERCENT = 98.0
S1_MAX_TRACKS = 3

MAX_EVENT_TRACK_SPREAD_DAYS = 6
MAX_EVENT_DATES_PER_ORBIT = 2

MAX_PRE_EVENT_GAP_DAYS = 45
MAX_PRE_CANDIDATES_PER_TRACK = 4

ALLOW_MIXED_ORBIT_DIRECTION = False
S1_ORBIT_DIRECTION = None  # None / "ASCENDING" / "DESCENDING"
MAX_S1_METADATA_IMAGES = 200

# ---------------- Sentinel-2 QA ----------------
S2_SCALE_M = 10
S2_MAX_SCENE_CLOUD_PERCENT = 90
S2_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]

# ---------------- JRC ----------------
JRC_SCALE_M = 30
JRC_PERMANENT_MONTHS = 12

OVERWRITE = True

print("✅ Controls loaded")
print("Sentinel-1 target coverage:", S1_TARGET_COVERAGE_PERCENT, "%")

## Step 3A — Imports, paths, logging and visible heartbeat helpers

In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib
import json
import logging
import math
import threading
import time
import warnings

import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.merge import merge as rio_merge
from rasterio.mask import mask as rio_mask
from shapely.geometry import box, mapping
from shapely.ops import unary_union
from affine import Affine
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("default")

def heartbeat_call(label, func, *args, interval=1.0, **kwargs):
    """Show elapsed activity while a remote/blocking call is running."""
    stop_event = threading.Event()
    bar = tqdm(total=None, desc=label, unit="s", leave=True)

    def pulse():
        while not stop_event.wait(interval):
            bar.update(1)
            bar.set_postfix_str("working")

    thread = threading.Thread(target=pulse, daemon=True)
    thread.start()
    started = time.time()
    try:
        result = func(*args, **kwargs)
        bar.set_postfix_str(f"done in {time.time() - started:.1f}s")
        return result
    finally:
        stop_event.set()
        thread.join(timeout=2)
        bar.close()

def feasibility_box(name, checks):
    print("\n" + "=" * 82)
    print("FEASIBILITY —", name)
    print("=" * 82)
    passed = True
    for label, ok, details in checks:
        print(("✅" if ok else "❌"), f"{label}: {details}")
        passed &= bool(ok)
    print("-" * 82)
    print("RESULT:", "PASS ✅" if passed else "FAIL ❌")
    print("=" * 82)
    return passed

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

def project_path(path_like):
    p = Path(path_like)
    return p if p.is_absolute() else (PROJECT_ROOT / p).resolve()

S1_OUT = PROJECT_ROOT / "data/interim/sentinel1" / EVENT_ID
S2_OUT = PROJECT_ROOT / "data/raw/sentinel2_optional" / EVENT_ID
JRC_OUT = PROJECT_ROOT / "data/interim/water"
META_OUT = PROJECT_ROOT / "data/interim/acquisition_metadata" / EVENT_ID
FIG_OUT = META_OUT / "figures"

for folder in [S1_OUT, S2_OUT, JRC_OUT, META_OUT, FIG_OUT]:
    folder.mkdir(parents=True, exist_ok=True)

LOG_FILE = META_OUT / "download.log"
logger = logging.getLogger("vrfsz_full_coverage")
logger.setLevel(logging.INFO)
logger.handlers.clear()
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(fmt)
logger.addHandler(stream_handler)

file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
file_handler.setFormatter(fmt)
logger.addHandler(file_handler)

print("Project root:", PROJECT_ROOT)

## Step 3B — FEASIBILITY: Date windows and GBM AOI

In [ ]:
event_start = pd.Timestamp(EVENT_START)
event_end = pd.Timestamp(EVENT_END)

if event_end < event_start:
    raise ValueError("EVENT_END must be on or after EVENT_START.")

if PRE_EVENT_START is None and PRE_EVENT_END is None:
    pre_end = event_start - pd.Timedelta(days=PRE_EVENT_GAP_DAYS)
    pre_start = pre_end - pd.Timedelta(days=PRE_EVENT_LOOKBACK_DAYS)
elif PRE_EVENT_START is not None and PRE_EVENT_END is not None:
    pre_start = pd.Timestamp(PRE_EVENT_START)
    pre_end = pd.Timestamp(PRE_EVENT_END)
else:
    raise ValueError("Set both PRE_EVENT_START/PRE_EVENT_END or leave both None.")

event_end_exclusive = event_end + pd.Timedelta(days=1)
pre_end_exclusive = pre_end + pd.Timedelta(days=1)

EVENT_REFERENCE = (
    event_start + (event_end - event_start) / 2
    if EVENT_REFERENCE_DATE is None
    else pd.Timestamp(EVENT_REFERENCE_DATE)
)

if AOI_BBOX is not None:
    aoi_gdf = gpd.GeoDataFrame(
        {"name": ["AOI"]},
        geometry=[box(*map(float, AOI_BBOX))],
        crs=4326,
    )
    aoi_source = "AOI_BBOX"
    aoi_path_ok = True
else:
    aoi_path = project_path(AOI_PATH)
    aoi_path_ok = aoi_path.exists()
    if not aoi_path_ok:
        raise FileNotFoundError(aoi_path)
    aoi_gdf = gpd.read_file(aoi_path)
    if aoi_gdf.crs is None:
        raise ValueError("AOI has no CRS.")
    aoi_gdf = aoi_gdf.to_crs(4326)
    aoi_source = str(aoi_path)

aoi_geom = unary_union(aoi_gdf.geometry)
aoi_valid = (not aoi_geom.is_empty) and aoi_geom.is_valid
aoi_geojson = mapping(aoi_geom)
aoi_bounds = tuple(map(float, aoi_geom.bounds))

centroid = aoi_geom.centroid
utm_zone = int((centroid.x + 180) // 6) + 1
LOCAL_UTM_EPSG = (32600 if centroid.y >= 0 else 32700) + utm_zone
aoi_utm = gpd.GeoSeries([aoi_geom], crs=4326).to_crs(LOCAL_UTM_EPSG).iloc[0]

LOCAL_SETUP_OK = feasibility_box(
    "Dates + AOI",
    [
        ("Event window", True, f"{event_start.date()} → {event_end.date()}"),
        ("Pre-event search", pre_end < event_start, f"{pre_start.date()} → {pre_end.date()}"),
        ("Event reference", True, str(EVENT_REFERENCE.date())),
        ("AOI", aoi_path_ok and aoi_valid, aoi_source),
        ("AOI bounds", aoi_valid, str(aoi_bounds)),
        ("Working CRS", True, f"EPSG:{LOCAL_UTM_EPSG}"),
    ],
)

if not LOCAL_SETUP_OK:
    raise RuntimeError("Local setup feasibility failed.")

# Step 4 — Google Earth Engine authentication and API test

In [ ]:
import ee
import geemap

print("=" * 72)
print("GOOGLE EARTH ENGINE SETUP")
print("=" * 72)
print("Project:", GEE_PROJECT)

try:
    if FORCE_GEE_REAUTH:
        raise RuntimeError("Forced reauthentication requested.")
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Existing credentials initialized Earth Engine.")
except Exception as first_error:
    print("⚠️ Existing initialization failed:", first_error)
    heartbeat_call(
        "Waiting for Google authentication",
        lambda: ee.Authenticate(auth_mode=GEE_AUTH_MODE),
    )
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Authentication and initialization completed.")

probe = heartbeat_call(
    "GEE API probe",
    lambda: ee.String("VRFSZ full-coverage connection OK").getInfo(),
)

GEE_AVAILABLE = probe == "VRFSZ full-coverage connection OK"

if not GEE_AVAILABLE:
    raise RuntimeError("Earth Engine API probe failed.")

print("✅ GEE API AVAILABLE")

# Step 5 — Sentinel-1 collection, grouping and coverage functions

A **track group** is one acquisition date + orbit direction + relative orbit. Multiple Sentinel-1
GRD slices belonging to the same track/date are mosaicked before coverage is evaluated.

In [ ]:
def ee_aoi():
    return ee.Geometry(aoi_geojson)

def gee_s1_collection(start, end):
    col = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(ee_aoi())
        .filterDate(
            f"{pd.Timestamp(start):%Y-%m-%d}",
            f"{pd.Timestamp(end):%Y-%m-%d}",
        )
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    )

    if S1_ORBIT_DIRECTION:
        col = col.filter(ee.Filter.eq("orbitProperties_pass", S1_ORBIT_DIRECTION))

    return col

def gee_s1_metadata(start, end, label):
    info = heartbeat_call(
        f"GEE S1 metadata — {label}",
        lambda: (
            gee_s1_collection(start, end)
            .limit(MAX_S1_METADATA_IMAGES)
            .getInfo()
        ),
    )

    rows = []
    for feature in tqdm(info.get("features", []), desc=f"Parsing {label}", unit="scene"):
        p = feature.get("properties", {})
        ms = p.get("system:time_start")
        dt = pd.to_datetime(ms, unit="ms", utc=True) if ms is not None else pd.NaT

        rows.append({
            "id": feature.get("id") or p.get("system:index"),
            "datetime": dt,
            "date": dt.date() if pd.notna(dt) else None,
            "orbit_pass": p.get("orbitProperties_pass"),
            "relative_orbit": p.get("relativeOrbitNumber_start"),
            "platform": p.get("platform_number"),
        })

    return pd.DataFrame(rows)

def group_s1(df):
    if df.empty:
        return df.copy()

    grouped = (
        df.groupby(
            ["date", "orbit_pass", "relative_orbit"],
            dropna=False,
        )
        .agg(
            scene_count=("id", "count"),
            first_datetime=("datetime", "min"),
            last_datetime=("datetime", "max"),
        )
        .reset_index()
    )

    grouped["date"] = pd.to_datetime(grouped["date"])
    return grouped

def build_s1_group_image(group):
    """Mosaic all slices for one date/pass/relative-orbit group."""
    d = pd.Timestamp(group["date"])

    col = (
        gee_s1_collection(d, d + pd.Timedelta(days=1))
        .filter(ee.Filter.eq("orbitProperties_pass", str(group["orbit_pass"])))
        .filter(
            ee.Filter.eq(
                "relativeOrbitNumber_start",
                int(group["relative_orbit"]),
            )
        )
    )

    return (
        col
        .select(["VV", "VH"])
        .mosaic()
        .clip(ee_aoi())
        .toFloat()
    )

def coverage_percent_from_mask(mask_image, band_name="valid"):
    value = heartbeat_call(
        f"Coverage check — {band_name}",
        lambda: mask_image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=ee_aoi(),
            scale=S1_COVERAGE_CHECK_SCALE_M,
            bestEffort=True,
            maxPixels=1e8,
            tileScale=4,
        ).get(band_name).getInfo(),
    )
    return 0.0 if value is None else float(value) * 100.0

def s1_coverage_percent(image, band="VV"):
    valid = image.select(band).mask().gt(0).unmask(0).rename("valid")
    return coverage_percent_from_mask(valid, "valid")

# Step 6 — FEASIBILITY: Discover pre-event and event Sentinel-1 candidates

In [ ]:
event_s1_df = gee_s1_metadata(event_start, event_end_exclusive, "event window")
pre_s1_df = gee_s1_metadata(pre_start, pre_end_exclusive, "pre-event window")

event_groups = group_s1(event_s1_df)
pre_groups = group_s1(pre_s1_df)

display(event_groups)
display(pre_groups)

event_orbits = (
    sorted(event_groups["relative_orbit"].dropna().astype(int).unique().tolist())
    if not event_groups.empty else []
)

S1_SEARCH_OK = feasibility_box(
    "Sentinel-1 candidate discovery",
    [
        ("Event groups", not event_groups.empty, f"{len(event_groups)} group(s)"),
        ("Pre-event groups", not pre_groups.empty, f"{len(pre_groups)} group(s)"),
        ("Distinct event relative orbits", len(event_orbits) >= 2, str(event_orbits)),
    ],
)

if not S1_SEARCH_OK:
    raise RuntimeError(
        "Sentinel-1 candidate discovery failed. "
        "For full GBM coverage, at least two event relative orbits are normally expected."
    )

# Step 7 — Evaluate actual event-track coverage

Every date/pass/relative-orbit group is tested over the GBM AOI. The earlier workflow skipped this
step, which is why a 52%-coverage swath was incorrectly accepted.

In [ ]:
event_rows = []

for _, row in tqdm(
    event_groups.iterrows(),
    total=len(event_groups),
    desc="Evaluating event track coverage",
    unit="track-date",
):
    group = row.to_dict()
    image = build_s1_group_image(group)
    coverage = s1_coverage_percent(image, "VV")
    offset = abs((pd.Timestamp(group["date"]) - EVENT_REFERENCE).days)

    event_rows.append({
        **group,
        "coverage_percent": float(coverage),
        "event_offset_days": int(offset),
    })

event_track_candidates = pd.DataFrame(event_rows)

# Keep only the nearest/best few dates for each pass + relative orbit.
event_track_candidates = (
    event_track_candidates
    .sort_values(
        ["orbit_pass", "relative_orbit", "event_offset_days", "coverage_percent"],
        ascending=[True, True, True, False],
    )
    .groupby(["orbit_pass", "relative_orbit"], as_index=False, group_keys=False)
    .head(MAX_EVENT_DATES_PER_ORBIT)
    .reset_index(drop=True)
)

display(
    event_track_candidates[
        [
            "date",
            "orbit_pass",
            "relative_orbit",
            "scene_count",
            "coverage_percent",
            "event_offset_days",
        ]
    ].sort_values(["orbit_pass", "relative_orbit", "event_offset_days"])
)

if event_track_candidates.empty:
    raise RuntimeError("No event-track candidates survived coverage evaluation.")

# Step 8 — Select the minimum complementary event-track set

The code does **not hardcode two scenes**. It tests 1, 2, then up to `S1_MAX_TRACKS` distinct
relative-orbit tracks and selects the smallest combination reaching the target coverage.

Complementary tracks may be acquired on nearby dates when same-date full coverage is unavailable.

In [ ]:
def union_coverage_percent(groups):
    masks = []

    for group in groups:
        img = build_s1_group_image(group)
        masks.append(
            img.select("VV").mask().gt(0).unmask(0).rename("valid")
        )

    union = ee.ImageCollection.fromImages(masks).max()
    return coverage_percent_from_mask(union, "valid")

def generate_event_solutions(candidate_df):
    solutions = []

    pass_options = (
        [None]
        if ALLOW_MIXED_ORBIT_DIRECTION
        else sorted(candidate_df["orbit_pass"].dropna().unique().tolist())
    )

    for orbit_pass in pass_options:
        subset = (
            candidate_df.copy()
            if orbit_pass is None
            else candidate_df[candidate_df["orbit_pass"] == orbit_pass].copy()
        )

        records = [row.to_dict() for _, row in subset.iterrows()]

        for n_tracks in range(1, min(S1_MAX_TRACKS, len(records)) + 1):
            for combo in combinations(records, n_tracks):

                # Do not select multiple dates from the same relative orbit.
                orbit_keys = [
                    (str(r["orbit_pass"]), int(r["relative_orbit"]))
                    for r in combo
                ]
                if len(set(orbit_keys)) != len(orbit_keys):
                    continue

                dates = [pd.Timestamp(r["date"]) for r in combo]
                date_spread = (max(dates) - min(dates)).days

                if date_spread > MAX_EVENT_TRACK_SPREAD_DAYS:
                    continue

                coverage = union_coverage_percent(combo)

                solutions.append({
                    "tracks": combo,
                    "track_count": int(n_tracks),
                    "coverage_percent": float(coverage),
                    "date_spread_days": int(date_spread),
                    "temporal_cost": int(sum(r["event_offset_days"] for r in combo)),
                    "orbit_pass": orbit_pass if orbit_pass is not None else "MIXED",
                })

    return solutions

event_solutions = generate_event_solutions(event_track_candidates)

if not event_solutions:
    raise RuntimeError("No valid complementary Sentinel-1 track combinations were constructed.")

acceptable = [
    s for s in event_solutions
    if s["coverage_percent"] >= S1_TARGET_COVERAGE_PERCENT
]

if not acceptable:
    best = max(event_solutions, key=lambda s: s["coverage_percent"])
    raise RuntimeError(
        "No event-track combination reaches the required coverage. "
        f"Best={best['coverage_percent']:.2f}% with {best['track_count']} track(s)."
    )

acceptable.sort(
    key=lambda s: (
        s["track_count"],
        -s["coverage_percent"],
        s["temporal_cost"],
        s["date_spread_days"],
    )
)

EVENT_PLAN = acceptable[0]

print("\nSELECTED EVENT-TRACK PLAN")
print("=" * 72)
print("Combined coverage:", f"{EVENT_PLAN['coverage_percent']:.2f}%")
print("Number of tracks:", EVENT_PLAN["track_count"])
print("Orbit direction:", EVENT_PLAN["orbit_pass"])
print("Date spread:", EVENT_PLAN["date_spread_days"], "day(s)")

for idx, track in enumerate(EVENT_PLAN["tracks"], 1):
    print(
        f"Track {idx}: date={pd.Timestamp(track['date']).date()} | "
        f"pass={track['orbit_pass']} | "
        f"relative orbit={int(track['relative_orbit'])} | "
        f"individual coverage={track['coverage_percent']:.2f}%"
    )

# Step 9 — Pair every selected event track with the best pre-event track

For each selected event relative orbit, only pre-event acquisitions from the **same orbit direction
and same relative orbit** are considered.

The nearest few candidates are tested. Selection prefers:

1. highest common pre ∩ event coverage;
2. then the smallest temporal gap.

In [ ]:
def pair_common_coverage_percent(pre_group, event_group):
    pre_img = build_s1_group_image(pre_group)
    event_img = build_s1_group_image(event_group)

    common = (
        pre_img.select("VV").mask().gt(0)
        .And(event_img.select("VV").mask().gt(0))
        .unmask(0)
        .rename("common")
    )

    return coverage_percent_from_mask(common, "common")

def select_pre_for_event_track(event_track):
    event_date = pd.Timestamp(event_track["date"])

    candidates = pre_groups[
        (pre_groups["orbit_pass"] == event_track["orbit_pass"])
        & (pre_groups["relative_orbit"] == event_track["relative_orbit"])
        & (pre_groups["date"] < event_date)
    ].copy()

    if candidates.empty:
        raise RuntimeError(
            f"No pre-event acquisition for pass={event_track['orbit_pass']} "
            f"relative orbit={int(event_track['relative_orbit'])}."
        )

    candidates["pre_event_gap_days"] = (
        event_date - pd.to_datetime(candidates["date"])
    ).dt.days

    candidates = (
        candidates[
            candidates["pre_event_gap_days"] <= MAX_PRE_EVENT_GAP_DAYS
        ]
        .sort_values("pre_event_gap_days")
        .head(MAX_PRE_CANDIDATES_PER_TRACK)
    )

    if candidates.empty:
        raise RuntimeError(
            f"No pre-event acquisition for relative orbit "
            f"{int(event_track['relative_orbit'])} within "
            f"{MAX_PRE_EVENT_GAP_DAYS} days."
        )

    evaluated = []

    for _, pre_row in candidates.iterrows():
        pre_group = pre_row.to_dict()
        common_cov = pair_common_coverage_percent(pre_group, event_track)

        evaluated.append({
            **pre_group,
            "common_coverage_percent": float(common_cov),
            "pre_event_gap_days": int(pre_row["pre_event_gap_days"]),
        })

    evaluated.sort(
        key=lambda r: (
            -r["common_coverage_percent"],
            r["pre_event_gap_days"],
        )
    )

    return evaluated[0], evaluated

paired_tracks = []
pre_evaluation_rows = []

for pair_id, event_track in enumerate(EVENT_PLAN["tracks"], 1):
    selected_pre, evaluated = select_pre_for_event_track(event_track)

    for candidate in evaluated:
        pre_evaluation_rows.append({
            "pair_id": int(pair_id),
            "event_date": str(pd.Timestamp(event_track["date"]).date()),
            "event_relative_orbit": int(event_track["relative_orbit"]),
            **candidate,
        })

    paired_tracks.append({
        "pair_id": int(pair_id),
        "event": event_track,
        "pre": selected_pre,
    })

pre_eval_df = pd.DataFrame(pre_evaluation_rows)
display(pre_eval_df)

print("\nSELECTED PRE/EVENT PAIRS")
print("=" * 72)

for pair in paired_tracks:
    event_track = pair["event"]
    pre_track = pair["pre"]

    print(
        f"Pair {pair['pair_id']} | "
        f"orbit={int(event_track['relative_orbit'])} {event_track['orbit_pass']} | "
        f"pre={pd.Timestamp(pre_track['date']).date()} | "
        f"event={pd.Timestamp(event_track['date']).date()} | "
        f"gap={int(pre_track['pre_event_gap_days'])} d | "
        f"common={pre_track['common_coverage_percent']:.2f}%"
    )

# Step 10 — Build the pair-aware unified Sentinel-1 mosaic

Pre and event bands remain in the **same pair image** before the final mosaic is made. This prevents
an overlap pixel from using pre-event values from one relative orbit and event values from another.

Overlap priority is based on event-date proximity to the event reference date.

In [ ]:
def build_paired_track_image(pair):
    pair_id = int(pair["pair_id"])
    event_track = pair["event"]
    pre_track = pair["pre"]

    pre_img = build_s1_group_image(pre_track).rename(["pre_VV", "pre_VH"])
    event_img = build_s1_group_image(event_track).rename(["event_VV", "event_VH"])

    common_mask = (
        pre_img.select("pre_VV").mask().gt(0)
        .And(event_img.select("event_VV").mask().gt(0))
    )

    event_date = pd.Timestamp(event_track["date"])
    pre_date = pd.Timestamp(pre_track["date"])

    event_offset = abs((event_date - EVENT_REFERENCE).days)
    pre_event_gap = int((event_date - pre_date).days)

    quality = (
        ee.Image.constant(-float(event_offset))
        .rename("quality")
        .toFloat()
    )

    provenance = (
        ee.Image.constant(int(event_track["relative_orbit"]))
        .rename("relative_orbit")
        .toInt16()
        .addBands(
            ee.Image.constant(pair_id)
            .rename("pair_id")
            .toInt16()
        )
        .addBands(
            ee.Image.constant(int(pre_date.strftime("%Y%m%d")))
            .rename("pre_date")
            .toInt32()
        )
        .addBands(
            ee.Image.constant(int(event_date.strftime("%Y%m%d")))
            .rename("event_date")
            .toInt32()
        )
        .addBands(
            ee.Image.constant(int(event_offset))
            .rename("event_offset_days")
            .toInt16()
        )
        .addBands(
            ee.Image.constant(pre_event_gap)
            .rename("pre_event_gap_days")
            .toInt16()
        )
    )

    return (
        pre_img
        .addBands(event_img)
        .addBands(quality)
        .addBands(provenance)
        .updateMask(common_mask)
        .clip(ee_aoi())
    )

pair_images = [
    build_paired_track_image(pair)
    for pair in paired_tracks
]

FULL_S1 = (
    ee.ImageCollection.fromImages(pair_images)
    .qualityMosaic("quality")
    .clip(ee_aoi())
)

print("Unified bands:")
print(FULL_S1.bandNames().getInfo())

# Step 11 — HARD FEASIBILITY: Final common Sentinel-1 coverage

This replaces the old weak test that only checked whether a raster file existed and had dimensions.

In [ ]:
FINAL_COMMON_COVERAGE = s1_coverage_percent(
    FULL_S1,
    band="event_VV",
)

FINAL_S1_OK = feasibility_box(
    "Final pair-aware Sentinel-1 mosaic",
    [
        (
            "Common pre/event coverage",
            FINAL_COMMON_COVERAGE >= S1_TARGET_COVERAGE_PERCENT,
            f"{FINAL_COMMON_COVERAGE:.2f}% "
            f"(target ≥ {S1_TARGET_COVERAGE_PERCENT:.2f}%)",
        ),
        (
            "Paired-track count",
            1 <= len(paired_tracks) <= S1_MAX_TRACKS,
            str(len(paired_tracks)),
        ),
        (
            "Event-track date spread",
            EVENT_PLAN["date_spread_days"] <= MAX_EVENT_TRACK_SPREAD_DAYS,
            f"{EVENT_PLAN['date_spread_days']} day(s)",
        ),
    ],
)

if not FINAL_S1_OK:
    raise RuntimeError(
        "STOP: the corrected Sentinel-1 mosaic does not satisfy the full-coverage requirement."
    )

# Step 12 — Download the corrected unified Sentinel-1 products

SAR output is cast to **float32** before export. This avoids the unnecessary float64 inflation seen
in the earlier run.

In [ ]:
def gee_download(image, out, scale, label):
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)

    if out.exists() and not OVERWRITE:
        print("♻️ Reusing:", out)
        return out

    heartbeat_call(
        label,
        lambda: geemap.download_ee_image(
            image=image,
            filename=str(out),
            region=ee_aoi(),
            scale=scale,
            crs=f"EPSG:{LOCAL_UTM_EPSG}",
            overwrite=True,
            num_threads=4,
        ),
    )

    if not out.exists() or out.stat().st_size == 0:
        raise RuntimeError(f"Download failed: {out}")

    return out

def split_bands(src_path, output_map):
    with rasterio.open(src_path) as src:
        for band_idx, out_path in tqdm(
            output_map.items(),
            desc=f"Splitting {Path(src_path).name}",
            unit="band",
        ):
            out_path = Path(out_path)
            profile = src.profile.copy()
            profile.update(count=1, compress="deflate")

            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(src.read(band_idx), 1)

S1_PRE_TMP = S1_OUT / "_full_gbm_pre_VV_VH.tif"
S1_EVENT_TMP = S1_OUT / "_full_gbm_event_VV_VH.tif"

pre_export = FULL_S1.select(["pre_VV", "pre_VH"]).toFloat()
event_export = FULL_S1.select(["event_VV", "event_VH"]).toFloat()

common_valid_export = (
    FULL_S1.select("event_VV")
    .mask()
    .gt(0)
    .rename("common_valid")
    .uint8()
)

provenance_export = FULL_S1.select([
    "relative_orbit",
    "pair_id",
    "pre_date",
    "event_date",
    "event_offset_days",
    "pre_event_gap_days",
])

with tqdm(
    total=4,
    desc="Downloading corrected Sentinel-1 outputs",
    unit="product",
) as bar:

    gee_download(
        pre_export,
        S1_PRE_TMP,
        S1_SCALE_M,
        "Downloading FULL GBM pre-event VV/VH",
    )
    bar.update(1)

    gee_download(
        event_export,
        S1_EVENT_TMP,
        S1_SCALE_M,
        "Downloading FULL GBM event VV/VH",
    )
    bar.update(1)

    COMMON_VALID_OUT = S1_OUT / "common_valid.tif"
    gee_download(
        common_valid_export,
        COMMON_VALID_OUT,
        S1_SCALE_M,
        "Downloading common-valid mask",
    )
    bar.update(1)

    PROVENANCE_OUT = S1_OUT / "S1_pair_provenance.tif"
    gee_download(
        provenance_export,
        PROVENANCE_OUT,
        S1_SCALE_M,
        "Downloading Sentinel-1 pair provenance",
    )
    bar.update(1)

PRE_VV = S1_OUT / "pre_VV.tif"
PRE_VH = S1_OUT / "pre_VH.tif"
EVENT_VV = S1_OUT / "event_VV.tif"
EVENT_VH = S1_OUT / "event_VH.tif"

split_bands(S1_PRE_TMP, {1: PRE_VV, 2: PRE_VH})
split_bands(S1_EVENT_TMP, {1: EVENT_VV, 2: EVENT_VH})

S1_PRE_TMP.unlink(missing_ok=True)
S1_EVENT_TMP.unlink(missing_ok=True)

print("✅ Corrected Sentinel-1 products written.")

# Step 13 — Sentinel-2 optional QA mosaic

For a large GBM AOI, one Sentinel-2 tile is not enough. This QA product uses all event-window L2A
images up to a permissive scene-cloud limit, applies SCL cloud masking per pixel, then builds a
median mosaic. Sentinel-2 remains QA only.

In [ ]:
def mask_s2(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    return (
        image
        .select(S2_BANDS)
        .multiply(0.0001)
        .updateMask(clear)
        .copyProperties(image, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])
    )

S2_RESULT = None

if DOWNLOAD_SENTINEL2:
    s2_col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_aoi())
        .filterDate(
            f"{event_start:%Y-%m-%d}",
            f"{event_end_exclusive:%Y-%m-%d}",
        )
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                S2_MAX_SCENE_CLOUD_PERCENT,
            )
        )
    )

    s2_count = heartbeat_call(
        "Counting Sentinel-2 QA scenes",
        lambda: int(s2_col.size().getInfo()),
    )

    if s2_count == 0:
        print("⚠️ No Sentinel-2 QA scenes; skipping.")
    else:
        s2_comp = s2_col.map(mask_s2).median().clip(ee_aoi())
        s2_valid = (
            s2_comp.mask().reduce(ee.Reducer.min())
            .rename("valid")
            .uint8()
        )

        S2_OUTFILE = S2_OUT / "S2_event_window_SR_composite.tif"

        gee_download(
            s2_comp.addBands(s2_valid),
            S2_OUTFILE,
            S2_SCALE_M,
            "Downloading Sentinel-2 QA mosaic",
        )

        S2_RESULT = {
            "file": S2_OUTFILE,
            "scene_count": s2_count,
        }

        print("✅ Sentinel-2 QA mosaic written:", S2_OUTFILE)

# Step 14 — JRC Global Surface Water v1.4

GEE is attempted first. If it fails, the notebook uses the official JRC/Google 10°×10° seasonality
tiles and mosaics/clips them locally.

In [ ]:
JRC_BASE = (
    "https://storage.googleapis.com/global-surface-water/"
    "downloads2021/seasonality"
)

def jrc_tile_ids(bounds):
    xmin, ymin, xmax, ymax = bounds
    eps = 1e-9

    x0 = int(math.floor(xmin / 10) * 10)
    x1 = int(math.floor((xmax - eps) / 10) * 10)
    y0 = int(math.floor(ymin / 10) * 10)
    y1 = int(math.floor((ymax - eps) / 10) * 10)

    ids = []
    for x in range(x0, x1 + 1, 10):
        for y_bottom in range(y0, y1 + 1, 10):
            y_top = y_bottom + 10
            ids.append(
                f"{abs(x)}{'E' if x >= 0 else 'W'}_"
                f"{abs(y_top)}{'N' if y_top >= 0 else 'S'}"
            )
    return ids

def jrc_urls(tile_id):
    return [
        f"{JRC_BASE}/seasonality_{tile_id}_v1_4_2021.tif",
        f"{JRC_BASE}/seasonality_{tile_id}v1_4_2021.tif",
        f"{JRC_BASE}/seasonality_{tile_id}_v1_4.tif",
    ]

def stream_download(url, out):
    out = Path(out)

    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))

        with out.open("wb") as f, tqdm(
            total=total if total > 0 else None,
            desc=f"Downloading {out.name}",
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

def acquire_jrc_direct():
    tile_dir = META_OUT / "jrc_direct_tiles"
    tile_dir.mkdir(parents=True, exist_ok=True)

    local_tiles = []

    for tile_id in tqdm(jrc_tile_ids(aoi_bounds), desc="JRC direct tiles", unit="tile"):
        selected = None

        for url in jrc_urls(tile_id):
            out = tile_dir / Path(url).name
            try:
                if not out.exists() or OVERWRITE:
                    stream_download(url, out)

                with rasterio.open(out):
                    pass

                selected = out
                break
            except Exception:
                out.unlink(missing_ok=True)

        if selected is None:
            raise RuntimeError(f"Official JRC tile failed: {tile_id}")

        local_tiles.append(selected)

    sources = [rasterio.open(p) for p in local_tiles]

    try:
        arr, transform = rio_merge(sources)

        profile = sources[0].profile.copy()
        profile.update(
            height=arr.shape[1],
            width=arr.shape[2],
            transform=transform,
            count=1,
            compress="deflate",
        )

        temp = META_OUT / "_jrc_direct_mosaic.tif"

        with rasterio.open(temp, "w", **profile) as dst:
            dst.write(arr[0], 1)

        with rasterio.open(temp) as src:
            geom = gpd.GeoSeries([aoi_geom], crs=4326).to_crs(src.crs).iloc[0]

            clipped, clipped_transform = rio_mask(
                src,
                [mapping(geom)],
                crop=True,
                filled=True,
                nodata=255,
            )

            p = src.profile.copy()
            p.update(
                height=clipped.shape[1],
                width=clipped.shape[2],
                transform=clipped_transform,
                count=1,
                dtype="uint8",
                nodata=255,
                compress="deflate",
            )

            seasonality_out = JRC_OUT / "jrc_gsw_v1_4_seasonality.tif"

            with rasterio.open(seasonality_out, "w", **p) as dst:
                dst.write(clipped[0].astype("uint8"), 1)

            permanent_out = JRC_OUT / "jrc_permanent_water.tif"
            pp = p.copy()
            pp.update(nodata=0)

            with rasterio.open(permanent_out, "w", **pp) as dst:
                dst.write(
                    (clipped[0] == JRC_PERMANENT_MONTHS).astype("uint8"),
                    1,
                )

        temp.unlink(missing_ok=True)

    finally:
        for src in sources:
            src.close()

    return seasonality_out, permanent_out

JRC_RESULT = None

if DOWNLOAD_JRC_WATER:
    season = (
        ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
        .select("seasonality")
        .clip(ee_aoi())
    )

    permanent = (
        season.eq(JRC_PERMANENT_MONTHS)
        .rename("permanent_water")
        .uint8()
    )

    seasonality_out = JRC_OUT / "jrc_gsw_v1_4_seasonality.tif"
    permanent_out = JRC_OUT / "jrc_permanent_water.tif"

    try:
        gee_download(
            season,
            seasonality_out,
            JRC_SCALE_M,
            "Downloading JRC seasonality through GEE",
        )

        gee_download(
            permanent,
            permanent_out,
            JRC_SCALE_M,
            "Downloading JRC permanent water through GEE",
        )

        provider = "GEE"

    except Exception as exc:
        print("⚠️ GEE JRC download failed; trying official direct tiles.")
        print("Reason:", exc)

        seasonality_out, permanent_out = acquire_jrc_direct()
        provider = "JRC_DIRECT"

    JRC_RESULT = {
        "provider": provider,
        "seasonality": seasonality_out,
        "permanent_water": permanent_out,
    }

    print("✅ JRC provider:", provider)

# Step 15 — Local-output QC

All Sentinel-1 products must exist, open successfully, and share an identical grid.

In [ ]:
required_s1_files = {
    "pre_VV": PRE_VV,
    "pre_VH": PRE_VH,
    "event_VV": EVENT_VV,
    "event_VH": EVENT_VH,
    "common_valid": COMMON_VALID_OUT,
    "pair_provenance": PROVENANCE_OUT,
}

qc_rows = []
reference_grid = None

for role, path in tqdm(
    required_s1_files.items(),
    desc="Checking corrected S1 outputs",
    unit="file",
):
    path = Path(path)
    exists = path.exists() and path.stat().st_size > 0
    raster_ok = False
    grid_ok = False
    details = "missing"

    if exists:
        with rasterio.open(path) as src:
            raster_ok = src.width > 0 and src.height > 0 and src.count > 0
            grid = (str(src.crs), src.transform, src.width, src.height)

            if reference_grid is None:
                reference_grid = grid
                grid_ok = True
            else:
                grid_ok = grid == reference_grid

            details = (
                f"{src.width}×{src.height} | {src.count} band(s) | "
                f"{src.dtypes} | {path.stat().st_size / 1024**2:.1f} MB"
            )

    qc_rows.append({
        "role": role,
        "exists": exists,
        "raster_ok": raster_ok,
        "same_grid": grid_ok,
        "details": details,
        "path": str(path),
    })

qc_df = pd.DataFrame(qc_rows)
display(qc_df)

LOCAL_S1_QC_OK = (
    qc_df["exists"].all()
    and qc_df["raster_ok"].all()
    and qc_df["same_grid"].all()
)

if not LOCAL_S1_QC_OK:
    raise RuntimeError("Corrected Sentinel-1 local-output QC failed.")

print("✅ Corrected Sentinel-1 local-output QC PASS")

# Step 16 — Safe visual QC over the GBM shapefile

The preview reader downsamples only the AOI window, so it does not load the full multi-gigabyte SAR
rasters into RAM.

In [ ]:
def read_aoi_preview(
    raster_path,
    aoi_gdf,
    band=1,
    max_size=1600,
    categorical=False,
):
    raster_path = Path(raster_path)

    with rasterio.open(raster_path) as src:
        aoi_raster = aoi_gdf.to_crs(src.crs)
        xmin, ymin, xmax, ymax = aoi_raster.total_bounds

        window = from_bounds(
            xmin,
            ymin,
            xmax,
            ymax,
            transform=src.transform,
        )

        full_window = rasterio.windows.Window(
            0,
            0,
            src.width,
            src.height,
        )

        try:
            window = window.intersection(full_window)
        except Exception:
            raise RuntimeError(f"AOI does not intersect raster: {raster_path}")

        native_width = max(1, int(round(window.width)))
        native_height = max(1, int(round(window.height)))

        scale_factor = max(
            native_width / max_size,
            native_height / max_size,
            1,
        )

        out_width = max(1, int(native_width / scale_factor))
        out_height = max(1, int(native_height / scale_factor))

        method = Resampling.nearest if categorical else Resampling.average

        data = src.read(
            band,
            window=window,
            out_shape=(out_height, out_width),
            resampling=method,
            masked=True,
        )

        native_transform = src.window_transform(window)

        preview_transform = (
            native_transform
            * Affine.scale(
                window.width / out_width,
                window.height / out_height,
            )
        )

        shapes = [
            geom.__geo_interface__
            for geom in aoi_raster.geometry
            if geom is not None and not geom.is_empty
        ]

        inside_aoi = geometry_mask(
            shapes,
            out_shape=(out_height, out_width),
            transform=preview_transform,
            invert=True,
        )

    return data, preview_transform, aoi_raster, inside_aoi

def preview_extent(transform, width, height):
    xmin = transform.c
    ymax = transform.f
    xmax = xmin + transform.a * width
    ymin = ymax + transform.e * height
    return [xmin, xmax, ymin, ymax]

aoi_local = gpd.read_file(project_path(AOI_PATH))

pre_preview, pre_transform, pre_aoi, pre_inside = read_aoi_preview(
    PRE_VV,
    aoi_local,
)

event_preview, event_transform, event_aoi, event_inside = read_aoi_preview(
    EVENT_VV,
    aoi_local,
)

pre_arr = pre_preview.filled(np.nan)
event_arr = event_preview.filled(np.nan)

pre_arr[~np.isfinite(pre_arr)] = np.nan
event_arr[~np.isfinite(event_arr)] = np.nan

extent = preview_extent(
    pre_transform,
    pre_arr.shape[1],
    pre_arr.shape[0],
)

pre_valid = np.isfinite(pre_arr) & pre_inside
event_valid = np.isfinite(event_arr) & event_inside
common_valid_preview = pre_valid & event_valid

aoi_pixels = np.count_nonzero(pre_inside)

preview_coverage = pd.DataFrame({
    "coverage": [
        "Pre-event Sentinel-1",
        "Event Sentinel-1",
        "Common pre ∩ event",
    ],
    "percent_AOI": [
        100 * np.count_nonzero(pre_valid) / aoi_pixels,
        100 * np.count_nonzero(event_valid) / aoi_pixels,
        100 * np.count_nonzero(common_valid_preview) / aoi_pixels,
    ],
})

display(preview_coverage.round(2))

combined = np.concatenate([pre_arr[pre_valid], event_arr[event_valid]])
vmin = np.nanpercentile(combined, 2)
vmax = np.nanpercentile(combined, 98)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(
    pre_arr,
    extent=extent,
    origin="upper",
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
)
pre_aoi.boundary.plot(ax=axes[0], linewidth=0.8)
axes[0].set_title("Corrected S1 VV — Pre-event")

axes[1].imshow(
    event_arr,
    extent=extent,
    origin="upper",
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
)
event_aoi.boundary.plot(ax=axes[1], linewidth=0.8)
axes[1].set_title("Corrected S1 VV — Event")

axes[2].imshow(
    common_valid_preview,
    extent=extent,
    origin="upper",
)
event_aoi.boundary.plot(ax=axes[2], linewidth=0.8)
axes[2].set_title("Common Pre ∩ Event Coverage")

plt.tight_layout()
fig.savefig(FIG_OUT / "S1_full_coverage_visual_QC.png", dpi=200)
plt.show()

# Step 17 — Visualize Sentinel-1 relative-orbit provenance

This should visibly show multiple complementary track zones across the GBM rather than the previous
single western swath.

In [ ]:
prov_preview, prov_transform, prov_aoi, prov_inside = read_aoi_preview(
    PROVENANCE_OUT,
    aoi_local,
    band=1,
    categorical=True,
)

prov_arr = prov_preview.filled(0)

prov_extent = preview_extent(
    prov_transform,
    prov_arr.shape[1],
    prov_arr.shape[0],
)

fig, ax = plt.subplots(figsize=(11, 9))

im = ax.imshow(
    prov_arr,
    extent=prov_extent,
    origin="upper",
    interpolation="nearest",
)

prov_aoi.boundary.plot(ax=ax, linewidth=1.0)

ax.set_title(
    "Sentinel-1 Relative-Orbit Provenance — Unified GBM Mosaic"
)

plt.colorbar(im, ax=ax, label="Relative orbit")
plt.tight_layout()

fig.savefig(
    FIG_OUT / "S1_relative_orbit_provenance.png",
    dpi=200,
)

plt.show()

unique_orbits = np.unique(
    prov_arr[
        (prov_arr > 0)
        & prov_inside
    ]
)

print(
    "Relative orbits present in final mosaic:",
    unique_orbits.tolist(),
)

# Step 18 — Reproducibility metadata and file manifest

In [ ]:
pair_plan_records = []

for pair in paired_tracks:
    event_track = pair["event"]
    pre_track = pair["pre"]

    pair_plan_records.append({
        "pair_id": int(pair["pair_id"]),
        "orbit_pass": str(event_track["orbit_pass"]),
        "relative_orbit": int(event_track["relative_orbit"]),
        "pre_date": str(pd.Timestamp(pre_track["date"]).date()),
        "event_date": str(pd.Timestamp(event_track["date"]).date()),
        "pre_event_gap_days": int(pre_track["pre_event_gap_days"]),
        "pair_common_coverage_percent": float(
            pre_track["common_coverage_percent"]
        ),
        "event_individual_coverage_percent": float(
            event_track["coverage_percent"]
        ),
        "event_offset_days": int(event_track["event_offset_days"]),
    })

pair_plan_df = pd.DataFrame(pair_plan_records)

PAIR_PLAN_CSV = META_OUT / "sentinel1_paired_track_plan.csv"
pair_plan_df.to_csv(PAIR_PLAN_CSV, index=False)

display(pair_plan_df)

all_output_files = {
    "sentinel1_pre_VV": PRE_VV,
    "sentinel1_pre_VH": PRE_VH,
    "sentinel1_event_VV": EVENT_VV,
    "sentinel1_event_VH": EVENT_VH,
    "sentinel1_common_valid": COMMON_VALID_OUT,
    "sentinel1_pair_provenance": PROVENANCE_OUT,
}

if S2_RESULT is not None:
    all_output_files["sentinel2_QA"] = S2_RESULT["file"]

if JRC_RESULT is not None:
    all_output_files["jrc_seasonality"] = JRC_RESULT["seasonality"]
    all_output_files["jrc_permanent_water"] = JRC_RESULT["permanent_water"]

manifest_rows = []

for role, path in tqdm(
    all_output_files.items(),
    desc="Hashing final products",
    unit="file",
):
    path = Path(path)

    with rasterio.open(path) as src:
        file_hash = heartbeat_call(
            f"SHA-256 {path.name}",
            lambda p=path: sha256_file(p),
        )

        manifest_rows.append({
            "role": role,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "size_mb": round(path.stat().st_size / 1024**2, 3),
            "sha256": file_hash,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "dtype": ";".join(src.dtypes),
            "nodata": src.nodata,
        })

manifest_df = pd.DataFrame(manifest_rows)

MANIFEST_CSV = META_OUT / "download_manifest.csv"
MANIFEST_JSON = META_OUT / "download_manifest.json"

manifest_df.to_csv(MANIFEST_CSV, index=False)

metadata = {
    "notebook_version": "VRFSZ_GBM_FullCoverage_Acquisition_v3",
    "event_id": EVENT_ID,
    "event_window": [
        str(event_start.date()),
        str(event_end.date()),
    ],
    "event_reference": str(EVENT_REFERENCE.date()),
    "pre_search_window": [
        str(pre_start.date()),
        str(pre_end.date()),
    ],
    "aoi_bounds_wgs84": list(aoi_bounds),
    "working_crs": f"EPSG:{LOCAL_UTM_EPSG}",
    "s1_target_coverage_percent": S1_TARGET_COVERAGE_PERCENT,
    "s1_final_common_coverage_percent": FINAL_COMMON_COVERAGE,
    "s1_event_plan": {
        "coverage_percent": EVENT_PLAN["coverage_percent"],
        "track_count": EVENT_PLAN["track_count"],
        "orbit_pass": EVENT_PLAN["orbit_pass"],
        "date_spread_days": EVENT_PLAN["date_spread_days"],
    },
    "s1_pairs": pair_plan_records,
    "sentinel2": (
        {
            "scene_count": S2_RESULT["scene_count"],
            "file": str(Path(S2_RESULT["file"]).relative_to(PROJECT_ROOT)),
        }
        if S2_RESULT is not None
        else None
    ),
    "jrc_provider": JRC_RESULT["provider"] if JRC_RESULT else None,
    "files": manifest_rows,
}

MANIFEST_JSON.write_text(
    json.dumps(metadata, indent=2, default=str),
    encoding="utf-8",
)

display(manifest_df)

print("\n✅ FULL-COVERAGE ACQUISITION WORKFLOW COMPLETE")
print("Pair plan:", PAIR_PLAN_CSV)
print("Manifest:", MANIFEST_JSON)
print("Figures:", FIG_OUT)
print("Log:", LOG_FILE)

# Step 19 — Final decision gate before tiled/windowed Rasterio processing

Proceed only if:

- the remote common-coverage gate passed;
- all local Sentinel-1 outputs share the same grid;
- visual QC shows the eastern and western GBM sectors covered;
- the provenance map shows the complementary relative-orbit tracks;
- the paired-track metadata has been saved.

**Execution order:** after a kernel restart, run the notebook from the top through Step 12,
then run the final provenance-repair cell. The repair cell depends on the in-memory objects
created by those acquisition cells and cannot reconstruct them from an isolated execution.

In [ ]:
final_decision = {
    "remote_common_coverage_percent": round(FINAL_COMMON_COVERAGE, 2),
    "coverage_target_percent": S1_TARGET_COVERAGE_PERCENT,
    "paired_track_count": len(paired_tracks),
    "local_grid_qc": bool(LOCAL_S1_QC_OK),
    "ready_for_windowed_processing": bool(
        FINAL_COMMON_COVERAGE >= S1_TARGET_COVERAGE_PERCENT
        and LOCAL_S1_QC_OK
    ),
}

display(pd.DataFrame([final_decision]))

if final_decision["ready_for_windowed_processing"]:
    print("✅ READY for tiled/windowed Rasterio processing.")
else:
    print("❌ NOT READY. Resolve acquisition/QC problems first.")

In [ ]:
# =============================================================================
# SELF-CONTAINED FINAL REPAIR CELL
# SENTINEL-1 CATEGORICAL PAIR / ORBIT PROVENANCE
# =============================================================================
#
# WHY THIS CELL EXISTS
# --------------------
# The earlier S1_pair_provenance.tif was invalid because categorical metadata
# such as relative-orbit IDs were handled like numerical raster values.
#
# Example:
#
#     real selected orbits = {12, 114}
#
# but the exported provenance contained:
#
#     12, 13, 14, 15, ... 113, 114
#
# Those intermediate orbit IDs are impossible and are therefore artifacts.
#
#
# SCIENTIFIC FIX
# --------------
#
# This cell NEVER interpolates:
#
#     - relative orbit
#     - pair ID
#     - acquisition dates
#
# Instead it:
#
# 1. Reads the authoritative saved pair plan:
#
#       sentinel1_paired_track_plan.csv
#
# 2. Reconstructs the exact Sentinel-1 pre/event track for each pair directly
#    from Google Earth Engine.
#
# 3. Creates one BINARY valid-footprint mask for each pair:
#
#       0 = this pair does not supply this pixel
#       1 = this pair supplies valid pre + event observations
#
# 4. Downloads ONLY these small categorical masks.
#
# 5. Aligns the masks locally to the already-downloaded pre_VV.tif grid using:
#
#       NEAREST-NEIGHBOUR ONLY
#
# 6. Creates:
#
#       S1_pair_assignment.tif
#
#       0 = no valid pair
#       1 = Pair 1
#       2 = Pair 2
#       ...
#
# 7. Derives relative orbit FROM pair ID through a lookup table.
#
# 8. Keeps dates in CSV/JSON rather than rasterizing them.
#
# 9. Audits the entire full-resolution output.
#
# 10. Replaces the corrupted old provenance raster only AFTER all QC passes.
#
# 11. Produces visual QC maps at the end.
#
#
# IMPORTANT
# ---------
# This cell repairs PROVENANCE ONLY.
#
# It does not change:
#
#       pre_VV.tif
#       pre_VH.tif
#       event_VV.tif
#       event_VH.tif
#
# Therefore the large Sentinel-1 rasters are NOT redownloaded.
#
# =============================================================================


# =============================================================================
# 0. USER SETTINGS
# =============================================================================

EVENT_ID = "EVENT001"

GEE_PROJECT = "ee-tarin1"

AOI_PATH = (
    "data/raw/boundaries/gbm_delta.shp"
)

S1_SCALE_M = 10

# Re-download pair masks if they already exist.
OVERWRITE_PAIR_MASKS = True

# Size of final visual previews.
MAX_PREVIEW_SIZE = 1600


# =============================================================================
# 1. IMPORTS
# =============================================================================

from pathlib import Path

import hashlib
import json
import shutil
import time

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio

from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

from shapely.geometry import mapping

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

import ee
import geemap


print("=" * 92)

print(
    "SENTINEL-1 CATEGORICAL "
    "PROVENANCE REPAIR"
)

print("=" * 92)


# =============================================================================
# 2. LOCATE PROJECT ROOT
# =============================================================================

cwd = Path.cwd().resolve()


if cwd.name.lower() == "notebooks":

    PROJECT_ROOT = cwd.parent

else:

    PROJECT_ROOT = cwd


print(
    "Project root:"
)

print(
    PROJECT_ROOT
)


# =============================================================================
# 3. DEFINE EXISTING PROJECT FILES
# =============================================================================

S1_DIR = (

    PROJECT_ROOT
    / "data"
    / "interim"
    / "sentinel1"
    / EVENT_ID

)


META_DIR = (

    PROJECT_ROOT
    / "data"
    / "interim"
    / "acquisition_metadata"
    / EVENT_ID

)


AOI_FILE = (

    PROJECT_ROOT
    / AOI_PATH

)


PRE_VV = (

    S1_DIR
    / "pre_VV.tif"

)


COMMON_VALID = (

    S1_DIR
    / "common_valid.tif"

)


PAIR_PLAN_CSV = (

    META_DIR
    / "sentinel1_paired_track_plan.csv"

)


OLD_PROVENANCE = (

    S1_DIR
    / "S1_pair_provenance.tif"

)


# =============================================================================
# 4. NEW CLEAN OUTPUT FILES
# =============================================================================

PAIR_MASK_DIR = (

    META_DIR
    / "pair_source_masks"

)


PAIR_MASK_DIR.mkdir(

    parents=True,

    exist_ok=True,

)


PAIR_ASSIGNMENT_OUT = (

    S1_DIR
    / "S1_pair_assignment.tif"

)


RELATIVE_ORBIT_OUT = (

    S1_DIR
    / "S1_relative_orbit_categorical.tif"

)


PAIR_LOOKUP_CSV = (

    META_DIR
    / "sentinel1_pair_assignment_lookup.csv"

)


PAIR_LOOKUP_JSON = (

    META_DIR
    / "sentinel1_pair_assignment_lookup.json"

)


QC_JSON = (

    META_DIR
    / "categorical_provenance_repair_QC.json"

)


FIG_DIR = (

    META_DIR
    / "figures"

)


FIG_DIR.mkdir(

    parents=True,

    exist_ok=True,

)


QC_FIGURE = (

    FIG_DIR
    / "S1_categorical_provenance_QC.png"

)


# =============================================================================
# 5. BASIC INPUT FEASIBILITY
# =============================================================================

required_files = {

    "AOI shapefile":
        AOI_FILE,

    "Reference Sentinel-1 raster":
        PRE_VV,

    "Common-valid mask":
        COMMON_VALID,

    "Paired-track plan":
        PAIR_PLAN_CSV,

}


missing = []


for name, path in required_files.items():

    exists = path.exists()

    print(
        (
            "✅"
            if exists
            else "❌"
        ),
        name,
        "→",
        path,
    )

    if not exists:

        missing.append(
            f"{name}: {path}"
        )


if missing:

    raise FileNotFoundError(

        "\nMissing required files:\n\n"

        + "\n".join(
            missing
        )

    )


print(
    "\n✅ Input feasibility PASS"
)


# =============================================================================
# 6. READ AUTHORITATIVE PAIR PLAN
# =============================================================================

pair_plan = pd.read_csv(

    PAIR_PLAN_CSV

)


required_columns = {

    "pair_id",

    "orbit_pass",

    "relative_orbit",

    "pre_date",

    "event_date",

}


missing_columns = (

    required_columns

    - set(
        pair_plan.columns
    )

)


if missing_columns:

    raise RuntimeError(

        "Pair-plan CSV is missing columns:\n"

        + ", ".join(
            sorted(
                missing_columns
            )
        )

    )


pair_plan[
    "pair_id"
] = (

    pair_plan[
        "pair_id"
    ]

    .astype(int)

)


pair_plan[
    "relative_orbit"
] = (

    pair_plan[
        "relative_orbit"
    ]

    .astype(int)

)


pair_plan[
    "pre_date"
] = pd.to_datetime(

    pair_plan[
        "pre_date"
    ]

)


pair_plan[
    "event_date"
] = pd.to_datetime(

    pair_plan[
        "event_date"
    ]

)


# =============================================================================
# 7. RECOVER EVENT-OFFSET PRIORITY
# =============================================================================
#
# The original v3 quality mosaic preferred the event acquisition closest to
# EVENT_REFERENCE.
#
# If event_offset_days is already saved, use it directly.
#
# Otherwise reconstruct a deterministic reference from the event dates.
#
# =============================================================================

if "event_offset_days" not in pair_plan.columns:


    reference_date = (

        pair_plan[
            "event_date"
        ].min()

        +

        (
            pair_plan[
                "event_date"
            ].max()

            -

            pair_plan[
                "event_date"
            ].min()

        )
        / 2

    )


    pair_plan[
        "event_offset_days"
    ] = (

        pair_plan[
            "event_date"
        ]

        .sub(
            reference_date
        )

        .abs()

        .dt.days

    )


else:


    pair_plan[
        "event_offset_days"
    ] = (

        pair_plan[
            "event_offset_days"
        ]

        .astype(int)

    )


if "pre_event_gap_days" not in pair_plan.columns:


    pair_plan[
        "pre_event_gap_days"
    ] = (

        pair_plan[
            "event_date"
        ]

        -

        pair_plan[
            "pre_date"
        ]

    ).dt.days


pair_plan = (

    pair_plan

    .sort_values(
        "pair_id"
    )

    .reset_index(
        drop=True
    )

)


print(
    "\nAUTHORITATIVE PAIR PLAN"
)


display(

    pair_plan[
        [
            "pair_id",
            "orbit_pass",
            "relative_orbit",
            "pre_date",
            "event_date",
            "pre_event_gap_days",
            "event_offset_days",
        ]
    ]

)


# =============================================================================
# 8. EXPECTED CATEGORICAL VALUES
# =============================================================================

EXPECTED_PAIR_IDS = set(

    pair_plan[
        "pair_id"
    ]

    .astype(int)

)


EXPECTED_ORBITS = set(

    pair_plan[
        "relative_orbit"
    ]

    .astype(int)

)


PAIR_TO_ORBIT = dict(

    zip(

        pair_plan[
            "pair_id"
        ].astype(int),

        pair_plan[
            "relative_orbit"
        ].astype(int),

    )

)


print(
    "\nExpected pair IDs:",
    sorted(
        EXPECTED_PAIR_IDS
    )
)


print(
    "Expected relative orbits:",
    sorted(
        EXPECTED_ORBITS
    )
)


print(
    "Pair → orbit lookup:"
)


print(
    PAIR_TO_ORBIT
)


# =============================================================================
# 9. LOAD GBM AOI LOCALLY
# =============================================================================
#
# IMPORTANT:
# At this stage we use GeoPandas/Shapely ONLY.
#
# Do NOT call ee.Geometry() before Earth Engine has been initialized.
#
# =============================================================================

print(
    "\nLoading GBM AOI..."
)


aoi = gpd.read_file(
    AOI_FILE
)


if aoi.empty:

    raise RuntimeError(
        "AOI shapefile is empty."
    )


if aoi.crs is None:

    raise RuntimeError(
        "AOI shapefile has no CRS."
    )


print(
    "Original AOI CRS:",
    aoi.crs
)


# Convert AOI to WGS84 because Earth Engine expects geographic coordinates.
aoi_4326 = (
    aoi
    .to_crs(
        "EPSG:4326"
    )
)


# -------------------------------------------------------------------------
# Merge all shapefile features into one study-area geometry.
# -------------------------------------------------------------------------

try:

    # Newer GeoPandas
    aoi_geom = (
        aoi_4326
        .geometry
        .union_all()
    )

except AttributeError:

    # Older GeoPandas
    aoi_geom = (
        aoi_4326
        .geometry
        .unary_union
    )


if aoi_geom is None:

    raise RuntimeError(
        "AOI union returned None."
    )


if aoi_geom.is_empty:

    raise RuntimeError(
        "AOI geometry is empty."
    )


if not aoi_geom.is_valid:

    print(
        "⚠️ AOI geometry is invalid."
    )

    print(
        "Attempting geometry repair..."
    )


    # Shapely-compatible repair.
    try:

        from shapely import make_valid

        aoi_geom = make_valid(
            aoi_geom
        )

    except Exception:

        # buffer(0) fallback for older Shapely.
        aoi_geom = aoi_geom.buffer(
            0
        )


    if aoi_geom.is_empty:

        raise RuntimeError(
            "AOI geometry repair failed."
        )


print(
    "✅ AOI geometry valid"
)


print(
    "AOI WGS84 bounds:"
)


print(
    aoi_geom.bounds
)


# -------------------------------------------------------------------------
# Convert to normal Python GeoJSON.
#
# This does NOT require Earth Engine initialization.
# -------------------------------------------------------------------------

aoi_geojson = mapping(
    aoi_geom
)


print(
    "✅ AOI converted to local GeoJSON"
)



# =============================================================================
# 10. READ FINAL SENTINEL-1 REFERENCE GRID
# =============================================================================

print(
    "\nReading final Sentinel-1 grid..."
)


with rasterio.open(
    PRE_VV
) as ref:


    TARGET_CRS = (
        ref.crs.to_string()
    )


    TARGET_TRANSFORM = (
        ref.transform
    )


    TARGET_WIDTH = (
        ref.width
    )


    TARGET_HEIGHT = (
        ref.height
    )


    TARGET_PROFILE = (
        ref.profile.copy()
    )


    TARGET_RESOLUTION = (
        abs(
            ref.transform.a
        )
    )


print(
    "✅ Final Sentinel-1 grid loaded"
)


print(
    "CRS:",
    TARGET_CRS
)


print(
    "Size:",
    TARGET_WIDTH,
    "×",
    TARGET_HEIGHT
)


print(
    "Resolution:",
    TARGET_RESOLUTION,
    "m"
)



# =============================================================================
# 11. INITIALIZE GOOGLE EARTH ENGINE
# =============================================================================
#
# CRITICAL ORDER:
#
# ee.Initialize()
#       ↓
# ee.Geometry()
#
# NOT the other way around.
#
# =============================================================================

print(
    "\n"
    + "=" * 92
)


print(
    "GOOGLE EARTH ENGINE INITIALIZATION"
)


print(
    "=" * 92
)


print(
    "Cloud project:",
    GEE_PROJECT
)


GEE_INITIALIZED = False


# -----------------------------------------------------------------------------
# First try existing saved credentials.
# -----------------------------------------------------------------------------

try:

    ee.Initialize(
        project=GEE_PROJECT
    )


    GEE_INITIALIZED = True


    print(
        "✅ Existing Earth Engine "
        "credentials initialized."
    )


except Exception as first_error:


    print(
        "⚠️ Existing Earth Engine "
        "initialization failed."
    )


    print(
        "Reason:"
    )


    print(
        first_error
    )


    print(
        "\nStarting browser authentication..."
    )


    # -------------------------------------------------------------------------
    # Local VS Code / Jupyter authentication
    # -------------------------------------------------------------------------

    ee.Authenticate(
        auth_mode="localhost"
    )


    print(
        "✅ Authentication completed."
    )


    print(
        "\nInitializing project..."
    )


    ee.Initialize(
        project=GEE_PROJECT
    )


    GEE_INITIALIZED = True


# -----------------------------------------------------------------------------
# Hard initialization check
# -----------------------------------------------------------------------------

if not GEE_INITIALIZED:

    raise RuntimeError(
        "Google Earth Engine initialization failed."
    )


# -----------------------------------------------------------------------------
# Real server-side API probe
# -----------------------------------------------------------------------------

print(
    "\nTesting Earth Engine server..."
)


probe = (
    ee.String(
        "VRFSZ provenance repair API OK"
    )
    .getInfo()
)


if (
    probe
    != "VRFSZ provenance repair API OK"
):

    raise RuntimeError(
        "Earth Engine API probe failed."
    )


print(
    "✅ Earth Engine API PASS"
)


print(
    "✅ Active project:",
    GEE_PROJECT
)



# =============================================================================
# 12. NOW CREATE THE EARTH ENGINE AOI
# =============================================================================
#
# Earth Engine is initialized at this point, so ee.Geometry() is safe.
#
# =============================================================================

print(
    "\nCreating Earth Engine AOI..."
)


try:

    EE_AOI = ee.Geometry(
        aoi_geojson
    )


except Exception as geometry_error:

    raise RuntimeError(
        "Could not construct Earth Engine AOI "
        "from the GBM shapefile."
    ) from geometry_error


# -----------------------------------------------------------------------------
# Real Earth Engine geometry test
# -----------------------------------------------------------------------------

ee_bounds = (
    EE_AOI
    .bounds()
    .coordinates()
    .getInfo()
)


print(
    "✅ Earth Engine AOI created"
)


print(
    "GEE AOI bounds successfully queried."
)


# Optional lightweight area check.
#
# Use a coarse maxError so EE is not forced to solve the polygon at
# unnecessarily high precision.
aoi_area_km2 = (
    EE_AOI
    .area(
        maxError=100
    )
    .divide(
        1e6
    )
    .getInfo()
)


print(
    f"Approximate AOI area: "
    f"{aoi_area_km2:,.2f} km²"
)


if (
    aoi_area_km2
    <= 0
):

    raise RuntimeError(
        "Earth Engine AOI has zero/negative area."
    )


print(
    "\n✅ AOI + GEE feasibility PASS"
)

# =============================================================================
# 12. FUNCTION — REBUILD ONE EXACT SENTINEL-1 TRACK/DATE
# =============================================================================

def build_exact_s1_track(

    acquisition_date,

    orbit_pass,

    relative_orbit,

):


    acquisition_date = pd.Timestamp(

        acquisition_date

    )


    next_date = (

        acquisition_date

        + pd.Timedelta(
            days=1
        )

    )


    collection = (

        ee.ImageCollection(
            "COPERNICUS/S1_GRD"
        )

        .filterBounds(
            EE_AOI
        )

        .filterDate(

            acquisition_date.strftime(
                "%Y-%m-%d"
            ),

            next_date.strftime(
                "%Y-%m-%d"
            ),

        )

        .filter(

            ee.Filter.eq(
                "instrumentMode",
                "IW",
            )

        )

        .filter(

            ee.Filter.listContains(

                "transmitterReceiverPolarisation",

                "VV",

            )

        )

        .filter(

            ee.Filter.listContains(

                "transmitterReceiverPolarisation",

                "VH",

            )

        )

        .filter(

            ee.Filter.eq(

                "orbitProperties_pass",

                str(
                    orbit_pass
                ),

            )

        )

        .filter(

            ee.Filter.eq(

                "relativeOrbitNumber_start",

                int(
                    relative_orbit
                ),

            )

        )

    )


    count = int(

        collection
        .size()
        .getInfo()

    )


    if count == 0:

        raise RuntimeError(

            "No Sentinel-1 scene found for:\n"

            f"date={acquisition_date.date()}\n"

            f"pass={orbit_pass}\n"

            f"relative orbit={relative_orbit}"

        )


    image = (

        collection

        .select(
            [
                "VV",
                "VH",
            ]
        )

        .mosaic()

        .clip(
            EE_AOI
        )

        .toFloat()

    )


    return (
        image,
        count,
    )


# =============================================================================
# 13. FUNCTION — DOWNLOAD BINARY PAIR MASK
# =============================================================================

def download_binary_mask(

    image,

    output_path,

):


    output_path = Path(

        output_path

    )


    if (
        output_path.exists()
        and not OVERWRITE_PAIR_MASKS
    ):

        print(
            "♻️ Reusing:",
            output_path
        )

        return


    geemap.download_ee_image(

        image=image,

        filename=str(
            output_path
        ),

        region=EE_AOI,

        scale=S1_SCALE_M,

        crs=TARGET_CRS,

        overwrite=True,

        num_threads=4,

    )


    if (

        not output_path.exists()

        or output_path.stat().st_size
        == 0

    ):

        raise RuntimeError(

            "Pair-mask download failed:\n"

            f"{output_path}"

        )


# =============================================================================
# 14. RECONSTRUCT EACH PAIR AND EXPORT ONLY BINARY SOURCE MEMBERSHIP
# =============================================================================

pair_mask_records = []


print(
    "\n"
    + "=" * 92
)


print(
    "RECONSTRUCTING SELECTED PAIRS"
)


print(
    "=" * 92
)


for _, row in tqdm(

    pair_plan.iterrows(),

    total=len(
        pair_plan
    ),

    desc="Reconstructing pairs",

    unit="pair",

):


    pair_id = int(

        row[
            "pair_id"
        ]

    )


    orbit_pass = str(

        row[
            "orbit_pass"
        ]

    )


    relative_orbit = int(

        row[
            "relative_orbit"
        ]

    )


    pre_date = pd.Timestamp(

        row[
            "pre_date"
        ]

    )


    event_date = pd.Timestamp(

        row[
            "event_date"
        ]

    )


    print(
        "\n"
        + "-" * 72
    )


    print(
        f"PAIR {pair_id}"
    )


    print(
        "Pass:",
        orbit_pass
    )


    print(
        "Relative orbit:",
        relative_orbit
    )


    print(
        "Pre:",
        pre_date.date()
    )


    print(
        "Event:",
        event_date.date()
    )


    # -------------------------------------------------------------------------
    # Reconstruct exact pre track
    # -------------------------------------------------------------------------

    pre_img, pre_scene_count = (

        build_exact_s1_track(

            pre_date,

            orbit_pass,

            relative_orbit,

        )

    )


    # -------------------------------------------------------------------------
    # Reconstruct exact event track
    # -------------------------------------------------------------------------

    event_img, event_scene_count = (

        build_exact_s1_track(

            event_date,

            orbit_pass,

            relative_orbit,

        )

    )


    print(
        "Pre GRD slice count:",
        pre_scene_count
    )


    print(
        "Event GRD slice count:",
        event_scene_count
    )


    # -------------------------------------------------------------------------
    # Require validity of BOTH VV and VH in the pre-event track.
    # -------------------------------------------------------------------------

    pre_valid = (

        pre_img

        .mask()

        .reduce(
            ee.Reducer.min()
        )

        .gt(0)

    )


    # -------------------------------------------------------------------------
    # Require validity of BOTH VV and VH in the event track.
    # -------------------------------------------------------------------------

    event_valid = (

        event_img

        .mask()

        .reduce(
            ee.Reducer.min()
        )

        .gt(0)

    )


    # -------------------------------------------------------------------------
    # Binary common-valid pair footprint.
    #
    # 0 = pair unavailable
    # 1 = both pre and event observations valid
    #
    # -------------------------------------------------------------------------

    pair_valid = (

        pre_valid

        .And(
            event_valid
        )

        .rename(
            "pair_valid"
        )

        .unmask(0)

        .uint8()

    )


    pair_mask_path = (

        PAIR_MASK_DIR

        / (
            f"pair_"
            f"{pair_id:02d}"
            f"_valid.tif"
        )

    )


    print(
        "Downloading binary pair mask..."
    )


    download_binary_mask(

        pair_valid,

        pair_mask_path,

    )


    # -------------------------------------------------------------------------
    # STRICT CATEGORICAL AUDIT OF RAW EXPORTED MASK
    # -------------------------------------------------------------------------

    raw_values = set()


    with rasterio.open(

        pair_mask_path

    ) as src:


        for _, window in src.block_windows(
            1
        ):


            arr = src.read(

                1,

                window=window,

            )


            raw_values.update(

                int(v)

                for v
                in np.unique(
                    arr
                )

            )


    print(
        "Exported values:",
        sorted(
            raw_values
        )
    )


    if not raw_values.issubset(

        {
            0,
            1,
        }

    ):


        raise RuntimeError(

            "\nBINARY MASK QC FAILED.\n\n"

            f"Pair {pair_id} contains:\n"

            f"{sorted(raw_values)}\n\n"

            "Expected ONLY 0 and 1.\n"

            "Stopping before any provenance raster is created."

        )


    pair_mask_records.append(

        {

            "pair_id":
                pair_id,

            "mask_path":
                pair_mask_path,

            "orbit_pass":
                orbit_pass,

            "relative_orbit":
                relative_orbit,

            "pre_date":
                pre_date,

            "event_date":
                event_date,

            "pre_event_gap_days":
                int(
                    row[
                        "pre_event_gap_days"
                    ]
                ),

            "event_offset_days":
                int(
                    row[
                        "event_offset_days"
                    ]
                ),

        }

    )


print(
    "\n✅ Every pair mask is binary."
)


# =============================================================================
# 15. CREATE LOCAL CATEGORICAL PAIR ASSIGNMENT
# =============================================================================
#
# IMPORTANT:
#
# The pair masks may come from slightly different source grids.
#
# They are therefore aligned to PRE_VV using:
#
#       Resampling.nearest
#
# This is the correct resampling rule for nominal/categorical data.
#
#
# Overlap priority:
#
#       farther event date written first
#       closer event date written last
#
# Therefore the event observation closest to the reference period controls
# overlapping valid areas, matching the intent of the v3 quality mosaic.
#
# =============================================================================

print(
    "\n"
    + "=" * 92
)


print(
    "BUILDING CATEGORICAL PAIR ASSIGNMENT"
)


print(
    "=" * 92
)


# Worst temporal match first.
# Best temporal match last.
pair_mask_records = sorted(

    pair_mask_records,

    key=lambda x:
        x[
            "event_offset_days"
        ],

    reverse=True,

)


with (
    rasterio.open(
        PRE_VV
    ) as ref,

    rasterio.open(
        COMMON_VALID
    ) as common_src,
):


    assignment_profile = (

        ref.profile.copy()

    )


    assignment_profile.update(

        dtype="uint8",

        count=1,

        nodata=0,

        compress="deflate",

    )


    # -------------------------------------------------------------------------
    # Open binary masks as VRTs on EXACTLY the final S1 grid.
    # -------------------------------------------------------------------------

    aligned_masks = []


    for rec in pair_mask_records:


        source = rasterio.open(

            rec[
                "mask_path"
            ]

        )


        vrt = WarpedVRT(

            source,

            crs=ref.crs,

            transform=ref.transform,

            width=ref.width,

            height=ref.height,

            resampling=(
                Resampling.nearest
            ),

        )


        aligned_masks.append(

            (
                rec,
                source,
                vrt,
            )

        )


    try:


        with rasterio.open(

            PAIR_ASSIGNMENT_OUT,

            "w",

            **assignment_profile,

        ) as pair_dst:


            windows = [

                window

                for _, window

                in ref.block_windows(
                    1
                )

            ]


            for window in tqdm(

                windows,

                desc=(
                    "Writing categorical "
                    "pair assignment"
                ),

                unit="block",

            ):


                assignment = np.zeros(

                    (
                        int(
                            window.height
                        ),

                        int(
                            window.width
                        ),
                    ),

                    dtype="uint8",

                )


                # -------------------------------------------------------------
                # Write every pair footprint.
                #
                # Better temporal match is later in the list and therefore
                # overwrites the overlap.
                # -------------------------------------------------------------

                for (
                    rec,
                    source,
                    vrt,
                ) in aligned_masks:


                    pair_mask = (

                        vrt.read(

                            1,

                            window=window,

                        )

                        > 0

                    )


                    assignment[
                        pair_mask
                    ] = int(

                        rec[
                            "pair_id"
                        ]

                    )


                # -------------------------------------------------------------
                # Restrict provenance to pixels actually considered valid by
                # the final v3 Sentinel-1 product.
                # -------------------------------------------------------------

                final_common = (

                    common_src.read(

                        1,

                        window=window,

                    )

                    > 0

                )


                assignment[
                    ~final_common
                ] = 0


                pair_dst.write(

                    assignment,

                    1,

                    window=window,

                )


    finally:


        for (
            rec,
            source,
            vrt,
        ) in aligned_masks:


            vrt.close()

            source.close()


print(
    "✅ Pair assignment written:"
)


print(
    PAIR_ASSIGNMENT_OUT
)


# =============================================================================
# 16. FULL-RESOLUTION PAIR-ID AUDIT
# =============================================================================

observed_pair_values = set()


pair_pixel_counts = {

    pair_id: 0

    for pair_id
    in EXPECTED_PAIR_IDS

}


total_assigned = 0


with rasterio.open(

    PAIR_ASSIGNMENT_OUT

) as src:


    windows = [

        window

        for _, window

        in src.block_windows(
            1
        )

    ]


    for window in tqdm(

        windows,

        desc="Auditing pair assignment",

        unit="block",

    ):


        arr = src.read(

            1,

            window=window,

        )


        observed_pair_values.update(

            int(v)

            for v
            in np.unique(
                arr
            )

        )


        total_assigned += int(

            np.count_nonzero(
                arr > 0
            )

        )


        for pair_id in EXPECTED_PAIR_IDS:


            pair_pixel_counts[
                pair_id
            ] += int(

                np.count_nonzero(

                    arr
                    == pair_id

                )

            )


EXPECTED_PAIR_VALUES = (

    {0}

    | EXPECTED_PAIR_IDS

)


print(
    "\nExpected pair raster values:",
    sorted(
        EXPECTED_PAIR_VALUES
    )
)


print(
    "Observed pair raster values:",
    sorted(
        observed_pair_values
    )
)


if observed_pair_values != EXPECTED_PAIR_VALUES:


    raise RuntimeError(

        "\nPAIR ASSIGNMENT QC FAILED.\n\n"

        f"Expected:\n"
        f"{sorted(EXPECTED_PAIR_VALUES)}\n\n"

        f"Observed:\n"
        f"{sorted(observed_pair_values)}"

    )


print(
    "✅ Pair-ID categorical QC PASS"
)


# =============================================================================
# 17. DERIVE RELATIVE-ORBIT RASTER FROM PAIR ID
# =============================================================================
#
# THIS IS THE CORE SCIENTIFIC FIX.
#
# We never resample/interpolate the number 12 or 114.
#
# We simply apply:
#
#       pair 1 -> its known orbit
#       pair 2 -> its known orbit
#
# =============================================================================

print(
    "\nDeriving relative-orbit raster "
    "from pair IDs..."
)


with rasterio.open(

    PAIR_ASSIGNMENT_OUT

) as pair_src:


    orbit_profile = (

        pair_src.profile.copy()

    )


    orbit_profile.update(

        dtype="uint16",

        count=1,

        nodata=0,

        compress="deflate",

    )


    with rasterio.open(

        RELATIVE_ORBIT_OUT,

        "w",

        **orbit_profile,

    ) as orbit_dst:


        windows = [

            window

            for _, window

            in pair_src.block_windows(
                1
            )

        ]


        for window in tqdm(

            windows,

            desc=(
                "Mapping pair ID "
                "to orbit"
            ),

            unit="block",

        ):


            pair_arr = pair_src.read(

                1,

                window=window,

            )


            orbit_arr = np.zeros(

                pair_arr.shape,

                dtype="uint16",

            )


            for (
                pair_id,
                orbit_id,
            ) in PAIR_TO_ORBIT.items():


                orbit_arr[

                    pair_arr
                    == int(
                        pair_id
                    )

                ] = int(

                    orbit_id

                )


            orbit_dst.write(

                orbit_arr,

                1,

                window=window,

            )


print(
    "✅ Relative-orbit raster written:"
)


print(
    RELATIVE_ORBIT_OUT
)


# =============================================================================
# 18. FULL-RESOLUTION ORBIT AUDIT
# =============================================================================

observed_orbits = set()


with rasterio.open(

    RELATIVE_ORBIT_OUT

) as src:


    windows = [

        window

        for _, window

        in src.block_windows(
            1
        )

    ]


    for window in tqdm(

        windows,

        desc="Auditing relative orbits",

        unit="block",

    ):


        arr = src.read(

            1,

            window=window,

        )


        observed_orbits.update(

            int(v)

            for v
            in np.unique(
                arr
            )

            if int(v) > 0

        )


print(
    "\nExpected relative orbits:",
    sorted(
        EXPECTED_ORBITS
    )
)


print(
    "Observed relative orbits:",
    sorted(
        observed_orbits
    )
)


if observed_orbits != EXPECTED_ORBITS:


    raise RuntimeError(

        "\nRELATIVE-ORBIT QC FAILED.\n\n"

        f"Expected:\n"
        f"{sorted(EXPECTED_ORBITS)}\n\n"

        f"Observed:\n"
        f"{sorted(observed_orbits)}"

    )


print(
    "✅ Relative-orbit categorical QC PASS"
)


# =============================================================================
# 19. CREATE AUTHORITATIVE LOOKUP TABLE
# =============================================================================

lookup = pair_plan[

    [
        "pair_id",
        "orbit_pass",
        "relative_orbit",
        "pre_date",
        "event_date",
        "pre_event_gap_days",
        "event_offset_days",
    ]

].copy()


lookup[
    "pre_date"
] = (

    lookup[
        "pre_date"
    ]

    .dt.strftime(
        "%Y-%m-%d"
    )

)


lookup[
    "event_date"
] = (

    lookup[
        "event_date"
    ]

    .dt.strftime(
        "%Y-%m-%d"
    )

)


lookup.to_csv(

    PAIR_LOOKUP_CSV,

    index=False,

)


lookup_payload = {

    "description": (
        "Authoritative Sentinel-1 "
        "categorical provenance lookup. "
        "Dates and orbit metadata are "
        "stored here rather than "
        "interpolated as raster values."
    ),

    "event_id":
        EVENT_ID,

    "pair_assignment_raster":
        str(
            PAIR_ASSIGNMENT_OUT
        ),

    "relative_orbit_raster":
        str(
            RELATIVE_ORBIT_OUT
        ),

    "pairs":
        lookup.to_dict(
            orient="records"
        ),

}


PAIR_LOOKUP_JSON.write_text(

    json.dumps(

        lookup_payload,

        indent=2,

    ),

    encoding="utf-8",

)


print(
    "\n✅ Lookup CSV:"
)


print(
    PAIR_LOOKUP_CSV
)


# =============================================================================
# 20. BACK UP OLD CORRUPTED PROVENANCE
# =============================================================================
#
# We only reach this point after BOTH new categorical audits have passed.
#
# =============================================================================

if OLD_PROVENANCE.exists():


    OLD_BACKUP = (

        META_DIR

        / (
            "_INVALID_OLD_"
            "S1_pair_provenance_"
            "interpolated.tif"
        )

    )


    if not OLD_BACKUP.exists():


        shutil.copy2(

            OLD_PROVENANCE,

            OLD_BACKUP,

        )


        print(
            "\n⚠️ Old invalid provenance archived:"
        )


        print(
            OLD_BACKUP
        )


# =============================================================================
# 21. REPLACE OLD WORKING PROVENANCE WITH CLEAN SINGLE-BAND ORBIT MAP
# =============================================================================
#
# This keeps Notebook 07B compatible because it reads band 1 as relative orbit.
#
# The authoritative detailed provenance is now:
#
#       S1_pair_assignment.tif
#             +
#       sentinel1_pair_assignment_lookup.csv
#
# =============================================================================

shutil.copy2(

    RELATIVE_ORBIT_OUT,

    OLD_PROVENANCE,

)


print(
    "\n✅ Working S1_pair_provenance.tif "
    "replaced with clean categorical orbit raster."
)


# =============================================================================
# 22. FINAL COMPATIBILITY AUDIT
# =============================================================================

compat_values = set()


with rasterio.open(

    OLD_PROVENANCE

) as src:


    print(
        "Replacement provenance bands:",
        src.count
    )


    print(
        "Replacement provenance dtype:",
        src.dtypes
    )


    for _, window in src.block_windows(
        1
    ):


        arr = src.read(

            1,

            window=window,

        )


        compat_values.update(

            int(v)

            for v
            in np.unique(
                arr
            )

            if int(v) > 0

        )


print(
    "Replacement provenance orbits:",
    sorted(
        compat_values
    )
)


if compat_values != EXPECTED_ORBITS:


    raise RuntimeError(

        "Backward-compatible provenance "
        "replacement failed QC."

    )


print(
    "✅ Backward-compatible provenance PASS"
)


# =============================================================================
# 23. PAIR DISTRIBUTION TABLE
# =============================================================================

distribution_rows = []


for _, row in lookup.iterrows():


    pair_id = int(

        row[
            "pair_id"
        ]

    )


    pixel_count = int(

        pair_pixel_counts[
            pair_id
        ]

    )


    area_share = (

        100.0
        * pixel_count
        / max(
            total_assigned,
            1,
        )

    )


    distribution_rows.append(

        {

            "pair_id":
                pair_id,

            "relative_orbit":
                int(
                    row[
                        "relative_orbit"
                    ]
                ),

            "pre_date":
                row[
                    "pre_date"
                ],

            "event_date":
                row[
                    "event_date"
                ],

            "pre_event_gap_days":
                int(
                    row[
                        "pre_event_gap_days"
                    ]
                ),

            "pixel_count":
                pixel_count,

            "assigned_area_share_percent":
                area_share,

        }

    )


distribution_df = pd.DataFrame(

    distribution_rows

)


print(
    "\nPAIR ASSIGNMENT DISTRIBUTION"
)


display(

    distribution_df

)


# =============================================================================
# 24. SHA-256 HELPER
# =============================================================================

def sha256_file(

    path,

    chunk_size=1024 * 1024,

):


    digest = hashlib.sha256()


    with Path(path).open(

        "rb"

    ) as file_obj:


        while True:


            chunk = file_obj.read(

                chunk_size

            )


            if not chunk:

                break


            digest.update(

                chunk

            )


    return digest.hexdigest()


# =============================================================================
# 25. WRITE REPAIR QC REPORT
# =============================================================================

qc_report = {

    "event_id":
        EVENT_ID,

    "status":
        "PASS",

    "method": (
        "Binary selected-pair footprints "
        "reconstructed from exact GEE "
        "date/pass/relative-orbit members; "
        "locally aligned to final SAR grid "
        "with nearest-neighbour resampling; "
        "pair ID mapped to relative orbit "
        "through lookup table."
    ),

    "expected_pair_values":
        sorted(
            EXPECTED_PAIR_VALUES
        ),

    "observed_pair_values":
        sorted(
            observed_pair_values
        ),

    "expected_relative_orbits":
        sorted(
            EXPECTED_ORBITS
        ),

    "observed_relative_orbits":
        sorted(
            observed_orbits
        ),

    "pair_assignment":
        str(
            PAIR_ASSIGNMENT_OUT
        ),

    "pair_assignment_sha256":
        sha256_file(
            PAIR_ASSIGNMENT_OUT
        ),

    "relative_orbit_raster":
        str(
            RELATIVE_ORBIT_OUT
        ),

    "relative_orbit_sha256":
        sha256_file(
            RELATIVE_ORBIT_OUT
        ),

    "lookup_csv":
        str(
            PAIR_LOOKUP_CSV
        ),

    "pair_distribution":
        distribution_rows,

}


QC_JSON.write_text(

    json.dumps(

        qc_report,

        indent=2,

    ),

    encoding="utf-8",

)


print(
    "\n✅ Repair QC report:"
)


print(
    QC_JSON
)


# =============================================================================
# 26. SAFE CATEGORICAL PREVIEW FUNCTION
# =============================================================================

def categorical_preview(

    path,

    max_size=1600,

):


    with rasterio.open(

        path

    ) as src:


        scale_factor = max(

            src.width
            / max_size,

            src.height
            / max_size,

            1,

        )


        width = max(

            1,

            int(
                src.width
                / scale_factor
            ),

        )


        height = max(

            1,

            int(
                src.height
                / scale_factor
            ),

        )


        array = src.read(

            1,

            out_shape=(

                height,

                width,

            ),

            resampling=(
                Resampling.nearest
            ),

        )


        bounds = src.bounds


        extent = [

            bounds.left,

            bounds.right,

            bounds.bottom,

            bounds.top,

        ]


        return (

            array,

            extent,

            src.crs,

        )


# =============================================================================
# 27. CREATE VISUAL PREVIEWS
# =============================================================================

pair_preview, pair_extent, pair_crs = (

    categorical_preview(

        PAIR_ASSIGNMENT_OUT,

        MAX_PREVIEW_SIZE,

    )

)


orbit_preview, orbit_extent, orbit_crs = (

    categorical_preview(

        RELATIVE_ORBIT_OUT,

        MAX_PREVIEW_SIZE,

    )

)


# =============================================================================
# 28. AOI BOUNDARY IN FINAL SAR CRS
# =============================================================================

aoi_map = (

    aoi

    .to_crs(
        pair_crs
    )

)


# =============================================================================
# 29. DISCRETE VISUALIZATION
# =============================================================================

fig, axes = plt.subplots(

    1,

    2,

    figsize=(18, 8),

)


# -----------------------------------------------------------------------------
# PAIR ASSIGNMENT MAP
# -----------------------------------------------------------------------------

pair_ids_for_plot = [

    0,

    *sorted(
        EXPECTED_PAIR_IDS
    ),

]


pair_max = max(

    pair_ids_for_plot

)


pair_cmap = plt.get_cmap(

    "tab20",

    pair_max + 1,

)


pair_boundaries = (

    np.arange(

        -0.5,

        pair_max + 1.5,

        1,

    )

)


pair_norm = BoundaryNorm(

    pair_boundaries,

    pair_cmap.N,

)


im1 = axes[0].imshow(

    pair_preview,

    extent=pair_extent,

    origin="upper",

    interpolation="nearest",

    cmap=pair_cmap,

    norm=pair_norm,

)


aoi_map.boundary.plot(

    ax=axes[0],

    linewidth=0.9,

)


axes[0].set_title(

    "Sentinel-1 Pair Assignment\n"
    "Categorical source membership"

)


axes[0].set_xlabel(
    "Easting"
)


axes[0].set_ylabel(
    "Northing"
)


cbar1 = plt.colorbar(

    im1,

    ax=axes[0],

    ticks=pair_ids_for_plot,

    shrink=0.82,

)


cbar1.set_label(

    "Pair ID"

)


# -----------------------------------------------------------------------------
# RELATIVE-ORBIT MAP
# -----------------------------------------------------------------------------

orbit_values_for_plot = [

    0,

    *sorted(
        EXPECTED_ORBITS
    ),

]


# Convert orbit IDs to compact categorical indices for display ONLY.
#
# The actual raster remains the true orbit numbers.
#
orbit_to_display = {

    orbit_id: idx

    for idx, orbit_id

    in enumerate(
        orbit_values_for_plot
    )

}


orbit_display = np.zeros_like(

    orbit_preview,

    dtype=np.uint8,

)


for (
    orbit_id,
    display_id,
) in orbit_to_display.items():


    orbit_display[

        orbit_preview
        == orbit_id

    ] = display_id


orbit_cmap = plt.get_cmap(

    "tab20",

    len(
        orbit_values_for_plot
    ),

)


orbit_boundaries = (

    np.arange(

        -0.5,

        len(
            orbit_values_for_plot
        )
        + 0.5,

        1,

    )

)


orbit_norm = BoundaryNorm(

    orbit_boundaries,

    orbit_cmap.N,

)


im2 = axes[1].imshow(

    orbit_display,

    extent=orbit_extent,

    origin="upper",

    interpolation="nearest",

    cmap=orbit_cmap,

    norm=orbit_norm,

)


aoi_map.boundary.plot(

    ax=axes[1],

    linewidth=0.9,

)


axes[1].set_title(

    "Sentinel-1 Relative-Orbit Provenance\n"
    "Derived from Pair ID — no interpolation"

)


axes[1].set_xlabel(
    "Easting"
)


axes[1].set_ylabel(
    "Northing"
)


cbar2 = plt.colorbar(

    im2,

    ax=axes[1],

    ticks=list(

        range(
            len(
                orbit_values_for_plot
            )
        )

    ),

    shrink=0.82,

)


cbar2.ax.set_yticklabels(

    [
        str(v)

        for v
        in orbit_values_for_plot
    ]

)


cbar2.set_label(

    "Relative orbit"

)


plt.tight_layout()


fig.savefig(

    QC_FIGURE,

    dpi=200,

    bbox_inches="tight",

)


plt.show()


# =============================================================================
# 30. FINAL RESULT
# =============================================================================

print(
    "\n"
    + "=" * 92
)


print(
    "CATEGORICAL PROVENANCE REPAIR COMPLETE"
)


print(
    "=" * 92
)


print(
    "✅ Pair raster values:",
    sorted(
        observed_pair_values
    )
)


print(
    "✅ Relative-orbit values:",
    sorted(
        observed_orbits
    )
)


print(
    "\nGenerated files:"
)


print(
    "1.",
    PAIR_ASSIGNMENT_OUT
)


print(
    "2.",
    RELATIVE_ORBIT_OUT
)


print(
    "3.",
    PAIR_LOOKUP_CSV
)


print(
    "4.",
    PAIR_LOOKUP_JSON
)


print(
    "5.",
    QC_JSON
)


print(
    "6.",
    QC_FIGURE
)


print(
    "\nOld S1_pair_provenance.tif has now "
    "been replaced by a clean single-band "
    "categorical relative-orbit raster."
)


print(
    "\nDates remain in the lookup table; "
    "they are no longer raster-interpolated."
)


print(
    "\nNEXT ACTION:"
)


print(
    "Rerun 07B_Audit_Acquisition_Scientific_Readiness.ipynb."
)


print(
    "The provenance audit should now PASS."
)


print(
    "=" * 92
)

# STAGE 4 — Acquisition scientific-readiness audit

**Source provenance:** `C2 07B_Audit_Acquisition_Scientific_Readiness.ipynb`.

# 07B — Acquisition Scientific Readiness Audit

Run this immediately after the full-coverage acquisition notebook.

It independently checks the **downloaded local files** rather than trusting only the remote GEE
coverage gate. It also checks whether the provenance raster is truly categorical.

### Decisions written by this notebook

- `core_ready_for_pilot`: the SAR data are safe to use in Notebook 08.
- `paper_metadata_ready`: timing/provenance metadata are also publication-ready.

The second can fail while the first still passes. Flood classification does not use provenance as a
predictor, but the provenance must be repaired before final paper delivery.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from tqdm.auto import tqdm

EVENT_ID = "EVENT001"
AOI_PATH = "data/raw/boundaries/gbm_delta.shp"

MIN_EXACT_COMMON_COVERAGE_PERCENT = 98.0
PREFERRED_MAX_PRE_EVENT_GAP_DAYS = 21
PREFERRED_MAX_EVENT_TRACK_SPREAD_DAYS = 2

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

S1_DIR = PROJECT_ROOT / "data/interim/sentinel1" / EVENT_ID
META_DIR = PROJECT_ROOT / "data/interim/acquisition_metadata" / EVENT_ID

PRE_VV = S1_DIR / "pre_VV.tif"
PRE_VH = S1_DIR / "pre_VH.tif"
EVENT_VV = S1_DIR / "event_VV.tif"
EVENT_VH = S1_DIR / "event_VH.tif"
COMMON_VALID = S1_DIR / "common_valid.tif"
PROVENANCE = S1_DIR / "S1_pair_provenance.tif"
PAIR_PLAN = META_DIR / "sentinel1_paired_track_plan.csv"
GATE_OUT = META_DIR / "acquisition_scientific_gate.json"

required = [
    PRE_VV, PRE_VH, EVENT_VV, EVENT_VH, COMMON_VALID,
    PROVENANCE, PAIR_PLAN, PROJECT_ROOT / AOI_PATH,
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required input(s):\n" + "\n".join(missing))

print("✅ All acquisition-audit inputs found.")

## 1. Verify identical Sentinel-1 grids

In [ ]:
raster_paths = {
    "pre_VV": PRE_VV,
    "pre_VH": PRE_VH,
    "event_VV": EVENT_VV,
    "event_VH": EVENT_VH,
    "common_valid": COMMON_VALID,
}

rows = []
reference = None

for role, path in raster_paths.items():
    with rasterio.open(path) as src:
        grid = (str(src.crs), src.transform, src.width, src.height)
        if reference is None:
            reference = grid
        rows.append({
            "role": role,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "same_grid": grid == reference,
        })

grid_df = pd.DataFrame(rows)
display(grid_df)

GRID_PASS = bool(grid_df["same_grid"].all())

if not GRID_PASS:
    raise RuntimeError("Sentinel-1 members do not share an identical grid.")

print("✅ Grid integrity PASS")

## 2. Exact native-resolution valid coverage

The denominator is the actual rasterized GBM polygon, not the rectangular GeoTIFF extent.
The check is blockwise, so the 40k × 40k rasters are never loaded into RAM at once.

In [ ]:
aoi = gpd.read_file(PROJECT_ROOT / AOI_PATH)

with rasterio.open(PRE_VV) as ref:
    aoi_r = aoi.to_crs(ref.crs)
    geoms = [
        geom.__geo_interface__
        for geom in aoi_r.geometry
        if geom is not None and not geom.is_empty
    ]

    aoi_count = 0
    pre_valid_count = 0
    event_valid_count = 0
    common_count = 0

    with (
        rasterio.open(PRE_VH) as pre_vh,
        rasterio.open(EVENT_VV) as event_vv,
        rasterio.open(EVENT_VH) as event_vh,
        rasterio.open(COMMON_VALID) as common_src,
    ):
        windows = [w for _, w in ref.block_windows(1)]

        for window in tqdm(windows, desc="Exact coverage audit", unit="block"):
            inside = geometry_mask(
                geoms,
                out_shape=(int(window.height), int(window.width)),
                transform=ref.window_transform(window),
                invert=True,
            )

            if not inside.any():
                continue

            pvv = ref.read(1, window=window)
            pvh = pre_vh.read(1, window=window)
            evv = event_vv.read(1, window=window)
            evh = event_vh.read(1, window=window)
            cm = common_src.read(1, window=window)

            pre_ok = inside & np.isfinite(pvv) & np.isfinite(pvh)
            event_ok = inside & np.isfinite(evv) & np.isfinite(evh)
            common_ok = pre_ok & event_ok & (cm > 0)

            aoi_count += int(inside.sum())
            pre_valid_count += int(pre_ok.sum())
            event_valid_count += int(event_ok.sum())
            common_count += int(common_ok.sum())

coverage = {
    "pre_percent": 100.0 * pre_valid_count / aoi_count,
    "event_percent": 100.0 * event_valid_count / aoi_count,
    "common_percent": 100.0 * common_count / aoi_count,
}

display(pd.DataFrame([coverage]))

EXACT_COVERAGE_PASS = (
    coverage["common_percent"] >= MIN_EXACT_COMMON_COVERAGE_PERCENT
)

print("EXACT_COVERAGE_PASS =", EXACT_COVERAGE_PASS)

## 3. Paired-track timing audit

In [ ]:
pair_plan = pd.read_csv(PAIR_PLAN)
pair_plan["pre_date"] = pd.to_datetime(pair_plan["pre_date"])
pair_plan["event_date"] = pd.to_datetime(pair_plan["event_date"])

display(pair_plan)

expected_orbits = sorted(
    pair_plan["relative_orbit"].astype(int).unique().tolist()
)

pre_gap_max = int(pair_plan["pre_event_gap_days"].max())
event_spread = int(
    (pair_plan["event_date"].max() - pair_plan["event_date"].min()).days
)

PRE_GAP_WARNING = pre_gap_max > PREFERRED_MAX_PRE_EVENT_GAP_DAYS
EVENT_SPREAD_WARNING = (
    event_spread > PREFERRED_MAX_EVENT_TRACK_SPREAD_DAYS
)

print("Expected relative orbits:", expected_orbits)
print("Maximum pre-event gap:", pre_gap_max, "days")
print("Event-track spread:", event_spread, "days")

if PRE_GAP_WARNING:
    print(
        "⚠️ At least one pre/event baseline is longer than the preferred "
        f"{PREFERRED_MAX_PRE_EVENT_GAP_DAYS} days."
    )

if EVENT_SPREAD_WARNING:
    print(
        "⚠️ Complementary event tracks span multiple days. "
        "Gauge/hydrological evidence should confirm that they represent one flood episode."
    )

## 4. Provenance categorical-value audit

The relative-orbit band should contain **only** the orbit IDs listed in the pair plan.

In [ ]:
observed_orbits = set()

with rasterio.open(PROVENANCE) as prov:
    aoi_r = aoi.to_crs(prov.crs)
    geoms = [
        geom.__geo_interface__
        for geom in aoi_r.geometry
        if geom is not None and not geom.is_empty
    ]

    windows = [w for _, w in prov.block_windows(1)]

    for window in tqdm(windows, desc="Provenance audit", unit="block"):
        inside = geometry_mask(
            geoms,
            out_shape=(int(window.height), int(window.width)),
            transform=prov.window_transform(window),
            invert=True,
        )

        if not inside.any():
            continue

        arr = prov.read(1, window=window)
        valid = inside

        if prov.nodata is not None:
            valid &= arr != prov.nodata

        vals = np.unique(arr[valid])
        observed_orbits.update(int(v) for v in vals if int(v) > 0)

observed_orbits = sorted(observed_orbits)

print("Expected orbit values:", expected_orbits)
print("Observed unique count:", len(observed_orbits))
print(
    "Observed values:",
    observed_orbits[:40],
    "..." if len(observed_orbits) > 40 else "",
)

PROVENANCE_PASS = set(observed_orbits) == set(expected_orbits)

if not PROVENANCE_PASS:
    print(
        "⚠️ Provenance is not categorical as intended. "
        "Core VV/VH processing can still continue if the core gate passes, "
        "but the provenance layer must be repaired before final publication."
    )
else:
    print("✅ Provenance categorical values PASS")

## 5. Write the scientific readiness gate

In [ ]:
CORE_READY_FOR_PILOT = bool(GRID_PASS and EXACT_COVERAGE_PASS)

PAPER_METADATA_READY = bool(
    CORE_READY_FOR_PILOT
    and PROVENANCE_PASS
    and not PRE_GAP_WARNING
    and not EVENT_SPREAD_WARNING
)

report = {
    "event_id": EVENT_ID,
    "grid_pass": GRID_PASS,
    "exact_coverage": coverage,
    "exact_coverage_pass": EXACT_COVERAGE_PASS,
    "expected_relative_orbits": expected_orbits,
    "observed_provenance_orbits": observed_orbits,
    "provenance_pass": PROVENANCE_PASS,
    "max_pre_event_gap_days": pre_gap_max,
    "preferred_max_pre_event_gap_days": PREFERRED_MAX_PRE_EVENT_GAP_DAYS,
    "pre_gap_warning": PRE_GAP_WARNING,
    "event_track_date_spread_days": event_spread,
    "preferred_max_event_track_spread_days": PREFERRED_MAX_EVENT_TRACK_SPREAD_DAYS,
    "event_spread_warning": EVENT_SPREAD_WARNING,
    "core_ready_for_pilot": CORE_READY_FOR_PILOT,
    "paper_metadata_ready": PAPER_METADATA_READY,
}

GATE_OUT.write_text(json.dumps(report, indent=2), encoding="utf-8")

display(pd.DataFrame([report]))

print("Gate:", GATE_OUT)
print("CORE_READY_FOR_PILOT =", CORE_READY_FOR_PILOT)
print("PAPER_METADATA_READY =", PAPER_METADATA_READY)

if not CORE_READY_FOR_PILOT:
    raise RuntimeError("Do not run Notebook 08: core acquisition gate failed.")

# STAGE 5 — Pilot SAR flood classifier

**Source provenance:** `C3 08_Pilot_SAR_Flood_Detection_Windowed.ipynb`.

# 08 — Pilot Sentinel-1 Flood Detection (Windowed / Memory-Safe)

This is the **single pilot event** used to select and then freeze the SAR flood-detection workflow.

## Anti-circularity

This notebook does **not** load:

- native DeltaDTM;
- MSL-aligned DeltaDTM;
- VRFSZ classes.

The flood observation therefore remains independent of the terrain classification being tested.

## Pilot change metric

For each valid land pixel:

\[
\Delta VV = VV_{event} - VV_{pre}
\]

\[
\Delta VH = VH_{event} - VH_{pre}
\]

\[
S = 0.5\,\Delta VV + 0.5\,\Delta VH
\]

Strong negative change is the open-water flood-candidate direction. A deterministic Otsu threshold is
estimated from a spatial sample after JRC permanent-water exclusion.

This is deliberately lean. No DEM conditioning and no neural network are introduced.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

EVENT_ID = "EVENT001"

RANDOM_SEED = 20260828
MAX_SAMPLE_WINDOWS = 500
MAX_SAMPLES_PER_WINDOW = 3000
HISTOGRAM_BINS = 512

VV_WEIGHT = 0.5
VH_WEIGHT = 0.5

MIN_PLAUSIBLE_FLOOD_FRACTION = 0.001
MAX_PLAUSIBLE_FLOOD_FRACTION = 0.60

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

S1_DIR = PROJECT_ROOT / "data/interim/sentinel1" / EVENT_ID
META_DIR = PROJECT_ROOT / "data/interim/acquisition_metadata" / EVENT_ID
OUT_DIR = PROJECT_ROOT / "data/processed/flood_events" / EVENT_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRE_VV = S1_DIR / "pre_VV.tif"
PRE_VH = S1_DIR / "pre_VH.tif"
EVENT_VV = S1_DIR / "event_VV.tif"
EVENT_VH = S1_DIR / "event_VH.tif"
COMMON_VALID = S1_DIR / "common_valid.tif"

JRC_WATER = PROJECT_ROOT / "data/interim/water/jrc_permanent_water.tif"
GATE_JSON = META_DIR / "acquisition_scientific_gate.json"

FLOOD_OUT = OUT_DIR / "flood_mask_raw.tif"
VALID_OUT = OUT_DIR / "valid_land_mask.tif"
DIAG_OUT = OUT_DIR / "pilot_method_diagnostics.json"
FIG_OUT = OUT_DIR / "pilot_preview.png"

required = [
    PRE_VV, PRE_VH, EVENT_VV, EVENT_VH,
    COMMON_VALID, JRC_WATER, GATE_JSON,
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing input(s):\n" + "\n".join(missing))

gate = json.loads(GATE_JSON.read_text(encoding="utf-8"))

if not gate.get("core_ready_for_pilot", False):
    raise RuntimeError(
        "Notebook 07B did not approve the acquisition core for pilot processing."
    )

print("✅ Acquisition core gate passed.")

if not gate.get("paper_metadata_ready", False):
    print(
        "⚠️ Acquisition metadata are not fully paper-ready. "
        "This does not feed the classifier, so pilot processing may continue."
    )

## 1. Verify SAR grids and prepare JRC on-the-fly alignment

In [ ]:
with rasterio.open(PRE_VV) as ref:
    ref_grid = (str(ref.crs), ref.transform, ref.width, ref.height)
    profile = ref.profile.copy()

for path in [PRE_VH, EVENT_VV, EVENT_VH, COMMON_VALID]:
    with rasterio.open(path) as src:
        grid = (str(src.crs), src.transform, src.width, src.height)
        if grid != ref_grid:
            raise RuntimeError(f"Grid mismatch: {path}")

print("✅ Sentinel-1 grids are identical.")

with rasterio.open(PRE_VV) as s1, rasterio.open(JRC_WATER) as jrc:
    print("S1:", s1.crs, s1.res, s1.width, s1.height)
    print("JRC:", jrc.crs, jrc.res, jrc.width, jrc.height)
    print(
        "JRC is read through WarpedVRT using nearest-neighbour resampling. "
        "The source water raster is not modified."
    )

## 2. Deterministic spatial sampling and Otsu threshold

In [ ]:
def otsu_threshold(values, bins=512):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if values.size < 1000:
        raise RuntimeError("Too few valid samples for Otsu thresholding.")

    lo, hi = np.percentile(values, [1, 99])
    values = values[(values >= lo) & (values <= hi)]

    hist, edges = np.histogram(values, bins=bins)
    hist = hist.astype(np.float64)
    centers = (edges[:-1] + edges[1:]) / 2.0

    w1 = np.cumsum(hist)
    w2 = np.cumsum(hist[::-1])[::-1]

    m1 = np.cumsum(hist * centers) / np.maximum(w1, 1)
    m2 = (
        np.cumsum((hist * centers)[::-1])
        / np.maximum(w2[::-1], 1)
    )[::-1]

    between = (
        w1[:-1]
        * w2[1:]
        * (m1[:-1] - m2[1:]) ** 2
    )

    idx = int(np.nanargmax(between))
    return float(centers[idx])

rng = np.random.default_rng(RANDOM_SEED)
sample_scores = []

with (
    rasterio.open(PRE_VV) as pvv_src,
    rasterio.open(PRE_VH) as pvh_src,
    rasterio.open(EVENT_VV) as evv_src,
    rasterio.open(EVENT_VH) as evh_src,
    rasterio.open(COMMON_VALID) as common_src,
    rasterio.open(JRC_WATER) as jrc_src,
):
    windows = [w for _, w in pvv_src.block_windows(1)]

    if len(windows) > MAX_SAMPLE_WINDOWS:
        idx = np.sort(
            rng.choice(
                len(windows),
                size=MAX_SAMPLE_WINDOWS,
                replace=False,
            )
        )
        windows = [windows[i] for i in idx]

    with WarpedVRT(
        jrc_src,
        crs=pvv_src.crs,
        transform=pvv_src.transform,
        width=pvv_src.width,
        height=pvv_src.height,
        resampling=Resampling.nearest,
    ) as jrc_vrt:

        for window in tqdm(windows, desc="Sampling SAR change", unit="block"):
            pvv = pvv_src.read(1, window=window)
            pvh = pvh_src.read(1, window=window)
            evv = evv_src.read(1, window=window)
            evh = evh_src.read(1, window=window)
            common = common_src.read(1, window=window) > 0
            permanent = jrc_vrt.read(1, window=window) > 0

            valid = (
                common
                & np.isfinite(pvv)
                & np.isfinite(pvh)
                & np.isfinite(evv)
                & np.isfinite(evh)
                & (~permanent)
            )

            if not valid.any():
                continue

            dvv = evv[valid] - pvv[valid]
            dvh = evh[valid] - pvh[valid]

            score = VV_WEIGHT * dvv + VH_WEIGHT * dvh

            if score.size > MAX_SAMPLES_PER_WINDOW:
                take = rng.choice(
                    score.size,
                    MAX_SAMPLES_PER_WINDOW,
                    replace=False,
                )
                score = score[take]

            sample_scores.append(score.astype(np.float32))

if not sample_scores:
    raise RuntimeError("No valid land samples were collected.")

sample_scores = np.concatenate(sample_scores)
threshold = otsu_threshold(sample_scores, HISTOGRAM_BINS)

print("Sample count:", f"{sample_scores.size:,}")
print(
    "Percentiles:",
    np.percentile(sample_scores, [1, 5, 25, 50, 75, 95, 99]),
)
print("Otsu threshold:", threshold, "dB")

if threshold >= 0:
    print(
        "⚠️ The Otsu threshold is non-negative. "
        "Inspect the event carefully before freezing this method."
    )

## 3. Threshold diagnostic

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sample_scores, bins=200, density=True)
ax.axvline(
    threshold,
    linestyle="--",
    linewidth=2,
    label=f"Otsu = {threshold:.2f} dB",
)
ax.set_xlabel("Combined change score (event - pre, dB)")
ax.set_ylabel("Density")
ax.set_title(f"{EVENT_ID} — Pilot SAR change threshold")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Full-resolution blockwise classification

The output valid mask is the land-domain observation mask after:

- common pre/event SAR validity;
- finite VV/VH values;
- permanent-water exclusion.

Flood is always a subset of valid land.

In [ ]:
out_profile = profile.copy()
out_profile.update(
    dtype="uint8",
    count=1,
    nodata=0,
    compress="deflate",
)

total_valid_land = 0
total_flood = 0

with (
    rasterio.open(PRE_VV) as pvv_src,
    rasterio.open(PRE_VH) as pvh_src,
    rasterio.open(EVENT_VV) as evv_src,
    rasterio.open(EVENT_VH) as evh_src,
    rasterio.open(COMMON_VALID) as common_src,
    rasterio.open(JRC_WATER) as jrc_src,
    rasterio.open(FLOOD_OUT, "w", **out_profile) as flood_dst,
    rasterio.open(VALID_OUT, "w", **out_profile) as valid_dst,
):
    windows = [w for _, w in pvv_src.block_windows(1)]

    with WarpedVRT(
        jrc_src,
        crs=pvv_src.crs,
        transform=pvv_src.transform,
        width=pvv_src.width,
        height=pvv_src.height,
        resampling=Resampling.nearest,
    ) as jrc_vrt:

        for window in tqdm(
            windows,
            desc="Classifying pilot flood",
            unit="block",
        ):
            pvv = pvv_src.read(1, window=window)
            pvh = pvh_src.read(1, window=window)
            evv = evv_src.read(1, window=window)
            evh = evh_src.read(1, window=window)
            common = common_src.read(1, window=window) > 0
            permanent = jrc_vrt.read(1, window=window) > 0

            valid = (
                common
                & np.isfinite(pvv)
                & np.isfinite(pvh)
                & np.isfinite(evv)
                & np.isfinite(evh)
                & (~permanent)
            )

            score = np.full(
                pvv.shape,
                np.nan,
                dtype=np.float32,
            )

            score[valid] = (
                VV_WEIGHT * (evv[valid] - pvv[valid])
                + VH_WEIGHT * (evh[valid] - pvh[valid])
            )

            flood = valid & (score <= threshold)

            valid_dst.write(
                valid.astype("uint8"),
                1,
                window=window,
            )
            flood_dst.write(
                flood.astype("uint8"),
                1,
                window=window,
            )

            total_valid_land += int(valid.sum())
            total_flood += int(flood.sum())

flood_fraction = total_flood / max(total_valid_land, 1)

print("Valid land pixels:", f"{total_valid_land:,}")
print("Flood pixels:", f"{total_flood:,}")
print("Flood fraction:", f"{100*flood_fraction:.2f}%")

if flood_fraction < MIN_PLAUSIBLE_FLOOD_FRACTION:
    print("⚠️ Extremely small flood fraction; inspect the map.")
if flood_fraction > MAX_PLAUSIBLE_FLOOD_FRACTION:
    print(
        "⚠️ Extremely large flood fraction; seasonal-change contamination is possible."
    )

## 5. Safe pilot preview

In [ ]:
def downsample_one(path, max_size=1400, resampling=Resampling.nearest):
    with rasterio.open(path) as src:
        scale = max(src.width / max_size, src.height / max_size, 1)
        width = max(1, int(src.width / scale))
        height = max(1, int(src.height / scale))
        arr = src.read(
            1,
            out_shape=(height, width),
            resampling=resampling,
        )
        return arr, src.bounds

event_preview, bounds = downsample_one(
    EVENT_VV,
    resampling=Resampling.average,
)

flood_preview, _ = downsample_one(
    FLOOD_OUT,
    resampling=Resampling.nearest,
)

extent = [
    bounds.left,
    bounds.right,
    bounds.bottom,
    bounds.top,
]

vals = event_preview[np.isfinite(event_preview)]
vmin, vmax = np.percentile(vals, [2, 98])

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

axes[0].imshow(
    event_preview,
    extent=extent,
    origin="upper",
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
)
axes[0].set_title("Event VV")

axes[1].imshow(
    flood_preview,
    extent=extent,
    origin="upper",
    interpolation="nearest",
)
axes[1].set_title("Pilot raw flood mask")

plt.tight_layout()
fig.savefig(FIG_OUT, dpi=200)
plt.show()

## 6. Save pilot diagnostics for Notebook 09

In [ ]:
diagnostics = {
    "event_id": EVENT_ID,
    "algorithm": "dual_polarization_db_change_otsu",
    "change_score": (
        "VV_WEIGHT*(event_VV-pre_VV) + "
        "VH_WEIGHT*(event_VH-pre_VH)"
    ),
    "vv_weight": VV_WEIGHT,
    "vh_weight": VH_WEIGHT,
    "threshold_method": "Otsu on deterministic spatial sample",
    "pilot_otsu_threshold_db": float(threshold),
    "random_seed": RANDOM_SEED,
    "max_sample_windows": MAX_SAMPLE_WINDOWS,
    "max_samples_per_window": MAX_SAMPLES_PER_WINDOW,
    "histogram_bins": HISTOGRAM_BINS,
    "permanent_water_rule": (
        "JRC permanent water excluded before thresholding and classification"
    ),
    "spatial_cleaning": "none in pilot raw mask",
    "valid_land_pixels": int(total_valid_land),
    "flood_pixels": int(total_flood),
    "flood_fraction": float(flood_fraction),
    "uses_dem_or_vrfsz": False,
    "flood_mask": str(FLOOD_OUT.relative_to(PROJECT_ROOT)),
    "valid_mask": str(VALID_OUT.relative_to(PROJECT_ROOT)),
}

DIAG_OUT.write_text(
    json.dumps(diagnostics, indent=2),
    encoding="utf-8",
)

display(pd.DataFrame([diagnostics]))

print("Diagnostics:", DIAG_OUT)
print("Next: inspect the result, then run Notebook 09.")

# STAGE 6 — Freeze SAR classifier

**Source provenance:** `C4 09_Freeze_SAR_Workflow.ipynb`.

The classifier is frozen before applying it across the retained events.

# 09 — Freeze the SAR Flood-Detection Workflow

This is the project's deliberate **manual pilot gate**.

Run it only after inspecting Notebook 08 output against independent context/optical evidence.

Once accepted, the algorithm is frozen before remaining events are processed. The final workflow uses
the same automatic Otsu procedure for every event; it does not hand-tune each flood mask.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

EVENT_ID = "EVENT001"

# ============================================================
# MANUAL GATE
# Change to True only after reviewing Notebook 08.
# ============================================================
PILOT_ACCEPTED = True

PILOT_REVIEW_NOTE = (
    "Pilot accepted after visual/independent QA; "
    "no DEM or VRFSZ information was used to tune flood pixels."
)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

PILOT_JSON = (
    PROJECT_ROOT
    / "data/processed/flood_events"
    / EVENT_ID
    / "pilot_method_diagnostics.json"
)

FROZEN_CONFIG = (
    PROJECT_ROOT
    / "config/frozen_sar_workflow.json"
)

FROZEN_CONFIG.parent.mkdir(parents=True, exist_ok=True)

if not PILOT_JSON.exists():
    raise FileNotFoundError(PILOT_JSON)

pilot = json.loads(
    PILOT_JSON.read_text(encoding="utf-8")
)

display(pd.DataFrame([pilot]))

if not PILOT_ACCEPTED:
    raise RuntimeError(
        "Pilot is not yet accepted. "
        "Review Notebook 08 outputs, then set PILOT_ACCEPTED=True."
    )

In [ ]:
frozen = {
    "schema_version": "1.0",
    "pilot_event_id": EVENT_ID,
    "pilot_review_note": PILOT_REVIEW_NOTE,
    "algorithm": pilot["algorithm"],
    "vv_weight": pilot["vv_weight"],
    "vh_weight": pilot["vh_weight"],
    "threshold_strategy": "event_specific_otsu_same_algorithm",
    "random_seed": pilot["random_seed"],
    "max_sample_windows": pilot["max_sample_windows"],
    "max_samples_per_window": pilot["max_samples_per_window"],
    "histogram_bins": pilot["histogram_bins"],
    "permanent_water_rule": pilot["permanent_water_rule"],
    "spatial_cleaning": pilot["spatial_cleaning"],
    "uses_dem_or_vrfsz": False,
    "pilot_threshold_db_for_traceability": pilot["pilot_otsu_threshold_db"],
}

canonical = json.dumps(
    frozen,
    sort_keys=True,
    separators=(",", ":"),
)

frozen["config_sha256"] = hashlib.sha256(
    canonical.encode("utf-8")
).hexdigest()

FROZEN_CONFIG.write_text(
    json.dumps(frozen, indent=2),
    encoding="utf-8",
)

display(pd.DataFrame([frozen]))

print("✅ SAR workflow FROZEN")
print("Config:", FROZEN_CONFIG)
print("SHA-256:", frozen["config_sha256"])

# STAGE 7 — Apply the frozen classifier to the four retained events

**Source provenance:** `C5 10_Map_All_Flood_Events_Windowed.ipynb`.

# 10 — Map All Flood Events with the Frozen SAR Workflow

This notebook applies the Notebook 09 configuration to every acquired event under:

`data/interim/sentinel1/EVENT*/`

There is **no event-specific manual threshold tuning**.

Each event gets:

- `flood_mask.tif`
- `valid_mask.tif`
- `event_flood_metadata.json`

The event-specific Otsu number may differ, but the algorithm, sampling strategy, weights,
permanent-water rule, and absence of DEM/VRFSZ predictors remain frozen.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

FROZEN_CONFIG = PROJECT_ROOT / "config/frozen_sar_workflow.json"
JRC_WATER = PROJECT_ROOT / "data/interim/water/jrc_permanent_water.tif"
S1_ROOT = PROJECT_ROOT / "data/interim/sentinel1"
OUT_ROOT = PROJECT_ROOT / "data/processed/flood_events"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

OVERWRITE = True

if not FROZEN_CONFIG.exists():
    raise FileNotFoundError(
        "Run Notebook 09 first: " + str(FROZEN_CONFIG)
    )

if not JRC_WATER.exists():
    raise FileNotFoundError(JRC_WATER)

frozen = json.loads(
    FROZEN_CONFIG.read_text(encoding="utf-8")
)

print("Frozen algorithm:", frozen["algorithm"])
print("Frozen config SHA:", frozen["config_sha256"])

event_dirs = sorted(
    p
    for p in S1_ROOT.iterdir()
    if p.is_dir()
    and p.name.upper().startswith("EVENT")
)

if not event_dirs:
    raise RuntimeError("No EVENT* Sentinel-1 folders found.")

print("Events discovered:", [p.name for p in event_dirs])

## Shared frozen processing functions

In [ ]:
def otsu_threshold(values, bins):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if values.size < 1000:
        raise RuntimeError("Too few samples for Otsu.")

    lo, hi = np.percentile(values, [1, 99])
    values = values[(values >= lo) & (values <= hi)]

    hist, edges = np.histogram(values, bins=bins)
    hist = hist.astype(np.float64)
    centers = (edges[:-1] + edges[1:]) / 2.0

    w1 = np.cumsum(hist)
    w2 = np.cumsum(hist[::-1])[::-1]

    m1 = np.cumsum(hist * centers) / np.maximum(w1, 1)
    m2 = (
        np.cumsum((hist * centers)[::-1])
        / np.maximum(w2[::-1], 1)
    )[::-1]

    between = (
        w1[:-1]
        * w2[1:]
        * (m1[:-1] - m2[1:]) ** 2
    )

    return float(
        centers[int(np.nanargmax(between))]
    )


def validate_event_files(event_dir):
    files = {
        "pre_VV": event_dir / "pre_VV.tif",
        "pre_VH": event_dir / "pre_VH.tif",
        "event_VV": event_dir / "event_VV.tif",
        "event_VH": event_dir / "event_VH.tif",
        "common_valid": event_dir / "common_valid.tif",
    }

    missing = [
        str(p)
        for p in files.values()
        if not p.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Missing event files:\n"
            + "\n".join(missing)
        )

    reference = None

    for path in files.values():
        with rasterio.open(path) as src:
            grid = (
                str(src.crs),
                src.transform,
                src.width,
                src.height,
            )

            if reference is None:
                reference = grid
            elif grid != reference:
                raise RuntimeError(
                    f"Grid mismatch in {event_dir.name}: "
                    f"{path.name}"
                )

    return files


def sample_event_score(files, config):
    rng = np.random.default_rng(
        int(config["random_seed"])
    )

    samples = []

    with (
        rasterio.open(files["pre_VV"]) as pvv,
        rasterio.open(files["pre_VH"]) as pvh,
        rasterio.open(files["event_VV"]) as evv,
        rasterio.open(files["event_VH"]) as evh,
        rasterio.open(files["common_valid"]) as common_src,
        rasterio.open(JRC_WATER) as jrc,
    ):
        windows = [
            w
            for _, w in pvv.block_windows(1)
        ]

        max_windows = int(
            config["max_sample_windows"]
        )

        if len(windows) > max_windows:
            idx = np.sort(
                rng.choice(
                    len(windows),
                    max_windows,
                    replace=False,
                )
            )
            windows = [
                windows[i]
                for i in idx
            ]

        with WarpedVRT(
            jrc,
            crs=pvv.crs,
            transform=pvv.transform,
            width=pvv.width,
            height=pvv.height,
            resampling=Resampling.nearest,
        ) as jrc_vrt:

            for window in tqdm(
                windows,
                desc="Sampling event",
                unit="block",
                leave=False,
            ):
                a = pvv.read(1, window=window)
                b = pvh.read(1, window=window)
                c = evv.read(1, window=window)
                d = evh.read(1, window=window)

                common = (
                    common_src.read(
                        1,
                        window=window,
                    )
                    > 0
                )

                permanent = (
                    jrc_vrt.read(
                        1,
                        window=window,
                    )
                    > 0
                )

                valid = (
                    common
                    & np.isfinite(a)
                    & np.isfinite(b)
                    & np.isfinite(c)
                    & np.isfinite(d)
                    & (~permanent)
                )

                if not valid.any():
                    continue

                score = (
                    float(config["vv_weight"])
                    * (c[valid] - a[valid])
                    + float(config["vh_weight"])
                    * (d[valid] - b[valid])
                )

                cap = int(
                    config["max_samples_per_window"]
                )

                if score.size > cap:
                    take = rng.choice(
                        score.size,
                        cap,
                        replace=False,
                    )
                    score = score[take]

                samples.append(
                    score.astype(np.float32)
                )

    if not samples:
        raise RuntimeError(
            "No valid samples for this event."
        )

    return np.concatenate(samples)


def classify_event(event_id, files, config):
    out_dir = OUT_ROOT / event_id
    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    flood_out = out_dir / "flood_mask.tif"
    valid_out = out_dir / "valid_mask.tif"
    meta_out = out_dir / "event_flood_metadata.json"

    if (
        flood_out.exists()
        and valid_out.exists()
        and meta_out.exists()
        and not OVERWRITE
    ):
        return json.loads(
            meta_out.read_text(
                encoding="utf-8"
            )
        )

    scores = sample_event_score(
        files,
        config,
    )

    threshold = otsu_threshold(
        scores,
        int(config["histogram_bins"]),
    )

    with rasterio.open(
        files["pre_VV"]
    ) as ref:
        profile = ref.profile.copy()

    profile.update(
        dtype="uint8",
        count=1,
        nodata=0,
        compress="deflate",
    )

    total_valid = 0
    total_flood = 0

    with (
        rasterio.open(files["pre_VV"]) as pvv,
        rasterio.open(files["pre_VH"]) as pvh,
        rasterio.open(files["event_VV"]) as evv,
        rasterio.open(files["event_VH"]) as evh,
        rasterio.open(files["common_valid"]) as common_src,
        rasterio.open(JRC_WATER) as jrc,
        rasterio.open(
            flood_out,
            "w",
            **profile,
        ) as flood_dst,
        rasterio.open(
            valid_out,
            "w",
            **profile,
        ) as valid_dst,
    ):
        windows = [
            w
            for _, w in pvv.block_windows(1)
        ]

        with WarpedVRT(
            jrc,
            crs=pvv.crs,
            transform=pvv.transform,
            width=pvv.width,
            height=pvv.height,
            resampling=Resampling.nearest,
        ) as jrc_vrt:

            for window in tqdm(
                windows,
                desc=f"{event_id} classify",
                unit="block",
            ):
                a = pvv.read(1, window=window)
                b = pvh.read(1, window=window)
                c = evv.read(1, window=window)
                d = evh.read(1, window=window)

                common = (
                    common_src.read(
                        1,
                        window=window,
                    )
                    > 0
                )

                permanent = (
                    jrc_vrt.read(
                        1,
                        window=window,
                    )
                    > 0
                )

                valid = (
                    common
                    & np.isfinite(a)
                    & np.isfinite(b)
                    & np.isfinite(c)
                    & np.isfinite(d)
                    & (~permanent)
                )

                score = np.full(
                    a.shape,
                    np.nan,
                    dtype=np.float32,
                )

                score[valid] = (
                    float(config["vv_weight"])
                    * (c[valid] - a[valid])
                    + float(config["vh_weight"])
                    * (d[valid] - b[valid])
                )

                flood = (
                    valid
                    & (score <= threshold)
                )

                valid_dst.write(
                    valid.astype("uint8"),
                    1,
                    window=window,
                )

                flood_dst.write(
                    flood.astype("uint8"),
                    1,
                    window=window,
                )

                total_valid += int(
                    valid.sum()
                )

                total_flood += int(
                    flood.sum()
                )

    metadata = {
        "event_id": event_id,
        "algorithm": config["algorithm"],
        "threshold_strategy": config["threshold_strategy"],
        "event_otsu_threshold_db": float(threshold),
        "valid_land_pixels": int(total_valid),
        "flood_pixels": int(total_flood),
        "flood_fraction": float(
            total_flood
            / max(total_valid, 1)
        ),
        "frozen_config_sha256": config["config_sha256"],
        "flood_mask": str(
            flood_out.relative_to(
                PROJECT_ROOT
            )
        ),
        "valid_mask": str(
            valid_out.relative_to(
                PROJECT_ROOT
            )
        ),
    }

    meta_out.write_text(
        json.dumps(
            metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    return metadata

## Process every event folder

In [ ]:
results = []

for event_dir in event_dirs:
    event_id = event_dir.name

    print("\n" + "=" * 72)
    print("PROCESSING", event_id)
    print("=" * 72)

    try:
        files = validate_event_files(
            event_dir
        )

        metadata = classify_event(
            event_id,
            files,
            frozen,
        )

        results.append(metadata)

        print(
            "✅",
            event_id,
            "complete",
        )

    except Exception as exc:
        print(
            "❌",
            event_id,
            "failed:",
            exc,
        )

        results.append({
            "event_id": event_id,
            "error": (
                f"{type(exc).__name__}: "
                f"{exc}"
            ),
        })

summary = pd.DataFrame(results)
display(summary)

SUMMARY_CSV = (
    OUT_ROOT
    / "flood_event_processing_summary.csv"
)

summary.to_csv(
    SUMMARY_CSV,
    index=False,
)

print("Summary:", SUMMARY_CSV)

# STAGE 8 — Correct EVENT003 baseline branch

**Source provenance:** `VRFSZ_V3_EVENT003_Master_Notebook.ipynb, sections 0–2 only`.

Preserves the event acquisitions, reselects defensible same-orbit dry baselines, and rebuilds EVENT003 without re-downloading the other events.

# V3 master rebuild — EVENT003 baseline correction → four-event recurrence → Notebook 26 freeze

This is a **single, self-contained execution notebook** assembled from the current v2 project workflow.

## Scientific scope

Only the **EVENT003 pre-event Sentinel-1 baseline selection** is changed. The EVENT003 event acquisitions are preserved from the existing accepted pair plan; EVENT001, EVENT002 and EVENT004 flood products are reused unchanged after hard QC. The frozen SAR classifier is not retuned.

The EVENT003 baseline rule is changed because the FFWC 2020 flood chronology places 18/19 July inside active flood spells. This v3 workflow therefore excludes candidate pre-event acquisitions on or after **27 June 2020** and selects the nearest same-orbit candidate among coverage-equivalent eligible observations.

The master then rebuilds the four-event recurrence and reruns the downstream statistical, robustness, paper-output and validation workflow through Notebook 25.

### About “Notebook 26”

The supplied v2 package contains notebooks through **25**; there is no source Notebook 26. This master therefore adds a new **Section 26 — V3 freeze and reproducibility package**. Section 26 does not add a new scientific analysis; it freezes the v3 outputs, hashes them, compares v2/v3 results where available, and prepares a lightweight archive manifest.

## Non-negotiable gates

- EVENT001/2/4 are audited but not reprocessed.
- EVENT003 event dates/orbits are read from the existing accepted EVENT003 pair plan and are not changed.
- EVENT003 candidate baselines on/after 2020-06-27 are rejected before outcome inspection.
- The same frozen SAR configuration SHA must be used by all four events.
- The previous EVENT003 and downstream v2 outputs are archived before replacement.
- Any failed hard gate stops execution instead of silently falling back.

## 0 — Master controls

In [ ]:
from pathlib import Path
import os, json, shutil, hashlib, platform, sys
from datetime import datetime, timezone
import pandas as pd

# -----------------------------------------------------------------------------
# EDIT THIS ONLY IF AUTO-DETECTION DOES NOT FIND THE PROJECT ROOT.
# Example: r"E:\\VRFSZ project\\vrfsz\\VRFSZ"
# -----------------------------------------------------------------------------
MASTER_PROJECT_ROOT_OVERRIDE = None

# EVENT003 documentary pre-flood cutoff.
# FFWC Annual Flood Report 2020 identifies the first 2020 flood spell as
# beginning 27 June; therefore 27 June and later cannot be treated as a clean
# documentary pre-flood baseline for this v3 selection rule.
EVENT003_LAST_ALLOWED_BASELINE_DATE = "2020-06-26"
EVENT003_MAX_PRE_EVENT_GAP_DAYS = 60
EVENT003_COVERAGE_EQUIVALENCE_TOLERANCE_PP = 0.25

# Preserve the EVENT003 event acquisitions from the existing accepted pair plan.
PRESERVE_EVENT003_EVENT_ACQUISITIONS = True

# Safety: refuse to overwrite an already-v3 EVENT003 unless explicitly enabled.
ALLOW_OVERWRITE_EXISTING_V3_EVENT003 = False

# Downstream execution controls. Defaults follow the user's requested full run.
RUN_STAGE_11 = True
RUN_STAGE_18A3 = True
RUN_STAGE_18B_21 = True
RUN_STAGE_19B = True
RUN_STAGE_22B = True
RUN_STAGE_23 = True
RUN_STAGE_24 = True
RUN_STAGE_24B = True
RUN_STAGE_24C = True
RUN_STAGE_25 = True
RUN_STAGE_26 = True

MASTER_NOTEBOOK_FILENAME = "V3_EVENT003_REBUILD_TO_NOTEBOOK26_MASTER.ipynb"

SOURCE_NOTEBOOK_LINEAGE = [
    "10C_Audit_Event1_2_Process_Event3_4_Fix_All4.ipynb (EVENT003 processing basis; modified baseline rule)",
    "11_to_LAST_Combined_MultiEvent_Recurrence.ipynb",
    "18A3_Terrain_Recovery_and_Statistical_Readiness.ipynb",
    "18B_to_21_Combined_Final_Statistical_Analysis_and_Freeze.ipynb",
    "19B_Spatial_Block_Range_and_Bootstrap_Robustness.ipynb",
    "22B_Final_Paper_Table1_and_Figures_v2_1_FIXED.ipynb (paper output directory patched to v3)",
    "23_Independent_SAR_Validation_ReviewerProof_v3_PIXELAREA_FIXED.ipynb",
    "24_AllWeather_Independent_Validation_Search_Acquisition_and_Scoring.ipynb",
    "24B_Rerun_Failed_TierA_Search_and_Accuracy_Matrix.ipynb",
    "24C_TierA_Reference_Pairing_and_Sample_Eligibility.ipynb",
    "25_MultiSource_Validation_and_S1_Detection_Bias_Audit.ipynb",
    "26 newly added here: freeze/reproducibility only",
]


def resolve_master_root():
    if MASTER_PROJECT_ROOT_OVERRIDE is not None:
        p = Path(MASTER_PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, *cwd.parents]
    for p in candidates:
        if (
            (p / "data/processed/flood_events/EVENT001/flood_mask.tif").exists()
            and (p / "config/frozen_sar_workflow.json").exists()
            and (p / "data/raw/boundaries/gbm_delta.shp").exists()
        ):
            return p
    raise RuntimeError(
        "Could not auto-detect project root. Set MASTER_PROJECT_ROOT_OVERRIDE."
    )

MASTER_PROJECT_ROOT = resolve_master_root()
os.chdir(MASTER_PROJECT_ROOT)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
V3_AUDIT_DIR = MASTER_PROJECT_ROOT / "reports" / "v3_event3_rebuild_audit"
V3_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

STAGE_LOG = []
def stage_mark(stage, status, detail=""):
    STAGE_LOG.append({
        "utc": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "status": status,
        "detail": detail,
    })
    print(f"[{status}] {stage}: {detail}")

print("MASTER_PROJECT_ROOT:", MASTER_PROJECT_ROOT)
print("RUN_ID:", RUN_ID)
print("EVENT003 last allowed baseline:", EVENT003_LAST_ALLOWED_BASELINE_DATE)
print("Python:", sys.version)

## 1 — Pre-v3 archive and safety snapshot

In [ ]:
from pathlib import Path
import json, shutil

PROJECT_ROOT = MASTER_PROJECT_ROOT
EVENT_ROOT = PROJECT_ROOT / "data/processed/flood_events"
META_ROOT = PROJECT_ROOT / "data/interim/acquisition_metadata"
ARCHIVE_ROOT = PROJECT_ROOT / "archive" / f"v2_before_event3_v3_{RUN_ID}"
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=False)

OLD_EVENT3_DIR = EVENT_ROOT / "EVENT003"
OLD_EVENT3_META_DIR = META_ROOT / "EVENT003"
OLD_EVENT3_META = OLD_EVENT3_DIR / "event_flood_metadata.json"
OLD_EVENT3_PAIR_CANDIDATES = [
    OLD_EVENT3_DIR / "acquisition_pair_plan.csv",
    OLD_EVENT3_META_DIR / "sentinel1_paired_track_plan.csv",
]

if not OLD_EVENT3_DIR.exists():
    raise FileNotFoundError("Existing EVENT003 is required so its event acquisitions can be preserved.")
if not OLD_EVENT3_META.exists():
    raise FileNotFoundError(OLD_EVENT3_META)

old_event3_metadata = json.loads(OLD_EVENT3_META.read_text(encoding="utf-8"))
if old_event3_metadata.get("v3_baseline_revision") and not ALLOW_OVERWRITE_EXISTING_V3_EVENT003:
    raise RuntimeError(
        "EVENT003 already carries v3_baseline_revision=True. "
        "Refusing to overwrite. Set ALLOW_OVERWRITE_EXISTING_V3_EVENT003=True only if intentional."
    )

old_pair_path = next((p for p in OLD_EVENT3_PAIR_CANDIDATES if p.exists()), None)
if old_pair_path is None:
    raise FileNotFoundError("No existing EVENT003 acquisition pair plan was found.")

OLD_EVENT3_PAIR_PLAN = pd.read_csv(old_pair_path)
required_old_cols = {"orbit_pass", "relative_orbit", "pre_date", "event_date"}
if not required_old_cols.issubset(OLD_EVENT3_PAIR_PLAN.columns):
    raise RuntimeError(
        "Existing EVENT003 pair plan lacks required columns: "
        + ", ".join(sorted(required_old_cols - set(OLD_EVENT3_PAIR_PLAN.columns)))
    )

OLD_EVENT3_PAIR_PLAN["pre_date"] = pd.to_datetime(OLD_EVENT3_PAIR_PLAN["pre_date"], errors="raise")
OLD_EVENT3_PAIR_PLAN["event_date"] = pd.to_datetime(OLD_EVENT3_PAIR_PLAN["event_date"], errors="raise")
OLD_EVENT3_PAIR_PLAN = OLD_EVENT3_PAIR_PLAN.sort_values(["relative_orbit", "event_date"]).reset_index(drop=True)

if OLD_EVENT3_PAIR_PLAN.duplicated(["orbit_pass", "relative_orbit"]).any():
    raise RuntimeError("Existing EVENT003 pair plan has duplicate orbit/pass records.")

old_pair_snapshot = OLD_EVENT3_PAIR_PLAN.copy()
old_pair_snapshot.to_csv(V3_AUDIT_DIR / "EVENT003_v2_pair_plan_before_revision.csv", index=False)
(V3_AUDIT_DIR / "EVENT003_v2_metadata_before_revision.json").write_text(
    json.dumps(old_event3_metadata, indent=2, default=str), encoding="utf-8"
)

# Audit EVENT001/2/4 presence before changing EVENT003.
for eid in ["EVENT001", "EVENT002", "EVENT004"]:
    d = EVENT_ROOT / eid
    for fn in ["flood_mask.tif", "valid_mask.tif", "event_flood_metadata.json"]:
        p = d / fn
        if not p.exists():
            raise FileNotFoundError(f"{eid} missing required unchanged input: {p}")

# Archive EVENT003 by MOVE (no duplicate large rasters).
archive_event3 = ARCHIVE_ROOT / "data_processed_flood_events_EVENT003"
shutil.move(str(OLD_EVENT3_DIR), str(archive_event3))

if OLD_EVENT3_META_DIR.exists():
    shutil.move(str(OLD_EVENT3_META_DIR), str(ARCHIVE_ROOT / "data_interim_acquisition_metadata_EVENT003"))

# Archive downstream result directories that MUST be recomputed because EVENT003 changed.
# MOVE preserves v2 without duplicating large files.
recompute_dirs = [
    PROJECT_ROOT / "data/processed/recurrence",
    PROJECT_ROOT / "reports/final_analysis",
    PROJECT_ROOT / "data/processed/final_analysis",
    PROJECT_ROOT / "reports/spatial_block_robustness",
    PROJECT_ROOT / "data/processed/spatial_block_robustness",
]
for p in recompute_dirs:
    if p.exists():
        dest = ARCHIVE_ROOT / p.relative_to(PROJECT_ROOT).as_posix().replace("/", "__")
        shutil.move(str(p), str(dest))
        print("Archived:", p, "->", dest)

# Archive only derived validation outputs. Keep source documents, labels, downloads and user inputs.
validation_output_dirs = [
    PROJECT_ROOT / "validation/outputs",
    PROJECT_ROOT / "validation_allweather/outputs",
    PROJECT_ROOT / "validation_allweather_rerun/outputs",
    PROJECT_ROOT / "validation_allweather_24C/outputs",
    PROJECT_ROOT / "validation_multisource_25/outputs",
]
for p in validation_output_dirs:
    if p.exists():
        dest = ARCHIVE_ROOT / p.relative_to(PROJECT_ROOT).as_posix().replace("/", "__")
        shutil.move(str(p), str(dest))
        print("Archived validation outputs:", p, "->", dest)

# Preserve existing v2 paper outputs; v3 writes to a new directory.
print("V2 archive root:", ARCHIVE_ROOT)
stage_mark("PRE_V3_ARCHIVE", "COMPLETED", str(ARCHIVE_ROOT))

## 2 — EVENT003 v3 rebuild

### 2.1 Scientific change

The old EVENT003 event acquisitions are preserved exactly. Only the pre-event baseline is reselected.

Eligibility rule:

1. same orbit pass and relative orbit as the preserved event acquisition;
2. Sentinel-1 IW with VV and VH;
3. within the declared maximum pre-event search gap;
4. acquisition date **≤ 2020-06-26**;
5. among eligible candidates, retain candidates within 0.25 percentage points of the best common coverage and choose the temporally nearest one.

This cutoff is a documentary timing gate. It must not be described as a complete hydrometeorological proof that every selected baseline pixel was dry.

In [ ]:
from pathlib import Path
from itertools import combinations
import gc, json, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from shapely.geometry import mapping
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import ee
import geemap

PROJECT_ROOT = MASTER_PROJECT_ROOT
EVENT_ID = "EVENT003"
GEE_PROJECT = "ee-tarin1"
AOI_PATH = "data/raw/boundaries/gbm_delta.shp"

S1_SCALE_M = 10
COVERAGE_CHECK_SCALE_M = 1000
MAX_PRE_EVENT_GAP_DAYS = int(EVENT003_MAX_PRE_EVENT_GAP_DAYS)
COVERAGE_EQUIVALENCE_TOLERANCE_PP = float(EVENT003_COVERAGE_EQUIVALENCE_TOLERANCE_PP)
MIN_PLAUSIBLE_FLOOD_FRACTION = 0.001
MAX_PLAUSIBLE_FLOOD_FRACTION = 0.60
DELETE_TEMP_AFTER_SUCCESS = True

EVENTS = {
    "EVENT003": {
        "name": "July 2020 Bangladesh Monsoon Flood",
        "event_start": "2020-07-18",
        "event_end": "2020-07-30",
        "event_reference": "2020-07-24",
        "evidence_source": "Bangladesh FFWC Annual Flood Report 2020",
        "evidence_url": "https://api.ffwc.gov.bd/assets/annual-reports/annual20.pdf",
        "notes": (
            "The 18-30 July interval is retained as the analysis/search window for continuity, "
            "not as the documentary onset. FFWC reports active flood spells before and through "
            "18/19 July; therefore those dates are excluded from the v3 baseline pool."
        ),
    }
}

event_spec = EVENTS[EVENT_ID]
AOI_FILE = PROJECT_ROOT / AOI_PATH
FROZEN_CONFIG = PROJECT_ROOT / "config/frozen_sar_workflow.json"
JRC_WATER = PROJECT_ROOT / "data/interim/water/jrc_permanent_water.tif"
PROCESSED_ROOT = PROJECT_ROOT / "data/processed/flood_events"
META_ROOT = PROJECT_ROOT / "data/interim/acquisition_metadata"
WORK_ROOT = PROJECT_ROOT / "data/interim/_space_efficient_event_work"
MANIFEST_DIR = PROJECT_ROOT / "manifests"

for d in [PROCESSED_ROOT, META_ROOT, WORK_ROOT, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)
for p in [AOI_FILE, FROZEN_CONFIG, JRC_WATER]:
    if not p.exists():
        raise FileNotFoundError(p)

frozen = json.loads(FROZEN_CONFIG.read_text(encoding="utf-8"))
required_frozen = [
    "algorithm", "vv_weight", "vh_weight", "random_seed",
    "max_sample_windows", "max_samples_per_window", "histogram_bins", "config_sha256",
]
missing = [k for k in required_frozen if k not in frozen]
if missing:
    raise RuntimeError("Frozen config missing: " + ", ".join(missing))

print("Frozen SAR config SHA:", frozen["config_sha256"])
print("Old EVENT003 event acquisitions to preserve:")
display(OLD_EVENT3_PAIR_PLAN[["orbit_pass","relative_orbit","pre_date","event_date"]])

In [ ]:
def human_bytes(n):
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:,.2f} {unit}"
        n /= 1024

def path_size(path):
    path = Path(path)
    if not path.exists():
        return 0
    if path.is_file():
        return path.stat().st_size
    total = 0
    for p in path.rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except OSError:
            pass
    return total

def safe_remove(path):
    path = Path(path)
    if not path.exists():
        return 0
    n = path_size(path)
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()
    return n

def disk_report(label):
    u = shutil.disk_usage(PROJECT_ROOT)
    print("\n" + "=" * 72)
    print(label)
    print("=" * 72)
    print("Total:", human_bytes(u.total))
    print("Used :", human_bytes(u.used))
    print("Free :", human_bytes(u.free))

disk_report("CURRENT DISK STATUS")

In [ ]:
def audit_unchanged_event(event_id):
    event_dir = PROCESSED_ROOT / event_id
    flood = event_dir / "flood_mask.tif"
    valid = event_dir / "valid_mask.tif"
    meta = event_dir / "event_flood_metadata.json"
    for p in [flood, valid, meta]:
        if not p.exists():
            raise RuntimeError(f"{event_id} incomplete: missing {p.name}")
    metadata = json.loads(meta.read_text(encoding="utf-8"))
    if metadata.get("frozen_config_sha256") != frozen["config_sha256"]:
        raise RuntimeError(f"{event_id} frozen-config SHA mismatch")
    with rasterio.open(flood) as fsrc, rasterio.open(valid) as vsrc:
        if (fsrc.crs != vsrc.crs or fsrc.transform != vsrc.transform or
            fsrc.width != vsrc.width or fsrc.height != vsrc.height):
            raise RuntimeError(f"{event_id}: flood/valid grid mismatch")
        fvals, vvals, violations = set(), set(), 0
        for _, window in fsrc.block_windows(1):
            f = fsrc.read(1, window=window)
            v = vsrc.read(1, window=window)
            fvals.update(int(x) for x in np.unique(f))
            vvals.update(int(x) for x in np.unique(v))
            violations += int(np.count_nonzero((f > 0) & (v == 0)))
    if not fvals.issubset({0,1}) or not vvals.issubset({0,1}) or violations:
        raise RuntimeError(f"{event_id}: binary/subset QC failed")
    return {
        "event_id": event_id,
        "frozen_sha_match": True,
        "binary_subset_qc": True,
        "otsu_threshold_db": metadata.get("event_otsu_threshold_db"),
        "flood_fraction": metadata.get("flood_fraction"),
    }

unchanged_audit = pd.DataFrame([audit_unchanged_event(e) for e in ["EVENT001","EVENT002","EVENT004"]])
display(unchanged_audit)
print("✅ EVENT001, EVENT002 and EVENT004 will be reused unchanged.")

In [ ]:
EVENT_START = pd.Timestamp(event_spec["event_start"])
EVENT_END = pd.Timestamp(event_spec["event_end"])
EVENT_REFERENCE = pd.Timestamp(event_spec["event_reference"])
BASELINE_CUTOFF = pd.Timestamp(EVENT003_LAST_ALLOWED_BASELINE_DATE)

EVENT_OUT = PROCESSED_ROOT / EVENT_ID
EVENT_META = META_ROOT / EVENT_ID
EVENT_WORK = WORK_ROOT / EVENT_ID
for d in [EVENT_OUT, EVENT_META, EVENT_WORK]:
    d.mkdir(parents=True, exist_ok=True)

PAIR_PLAN_CSV = EVENT_META / "sentinel1_paired_track_plan.csv"
FINAL_PAIR_PLAN = EVENT_OUT / "acquisition_pair_plan.csv"
EVIDENCE_JSON = EVENT_OUT / "event_evidence.json"
FINAL_FLOOD = EVENT_OUT / "flood_mask.tif"
FINAL_VALID = EVENT_OUT / "valid_mask.tif"
FINAL_META = EVENT_OUT / "event_flood_metadata.json"
TMP_STACK = EVENT_WORK / "paired_s1_4band.tif"
BASELINE_QA_CSV = EVENT_META / "EVENT003_v3_baseline_candidate_QA.csv"

# Existing temporary EVENT003 work is stale by definition after baseline revision.
if EVENT_WORK.exists():
    shutil.rmtree(EVENT_WORK)
    EVENT_WORK.mkdir(parents=True, exist_ok=True)

aoi = gpd.read_file(AOI_FILE)
if aoi.empty or aoi.crs is None:
    raise RuntimeError("AOI is empty or has no CRS")
aoi_4326 = aoi.to_crs(4326)
try:
    aoi_geom = aoi_4326.geometry.union_all()
except AttributeError:
    aoi_geom = aoi_4326.geometry.unary_union
if aoi_geom.is_empty:
    raise RuntimeError("AOI union is empty")

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    print("Starting Earth Engine authentication...")
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize(project=GEE_PROJECT)
if ee.String("VRFSZ_EVENT3_V3_CHECK").getInfo() != "VRFSZ_EVENT3_V3_CHECK":
    raise RuntimeError("Earth Engine probe failed")
EE_AOI = ee.Geometry(mapping(aoi_geom))

evidence = {
    "event_id": EVENT_ID,
    "event_name": event_spec["name"],
    "analysis_search_start": str(EVENT_START.date()),
    "analysis_search_end": str(EVENT_END.date()),
    "documented_reference": str(EVENT_REFERENCE.date()),
    "evidence_source": event_spec["evidence_source"],
    "evidence_url": event_spec["evidence_url"],
    "notes": event_spec["notes"],
    "baseline_documentary_cutoff": str(BASELINE_CUTOFF.date()),
    "baseline_selection_scope": (
        "Documentary pre-flood timing gate; not a complete hydrometeorological dry-state proof."
    ),
    "selection_rule": "event acquisitions preserved from v2; only EVENT003 pre-event baseline reselected",
}
EVIDENCE_JSON.write_text(json.dumps(evidence, indent=2), encoding="utf-8")
print("EVENT003 baseline cutoff:", BASELINE_CUTOFF.date())

In [ ]:
def s1_collection(start, end):
    end_exclusive = pd.Timestamp(end) + pd.Timedelta(days=1)
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(EE_AOI)
        .filterDate(
            pd.Timestamp(start).strftime("%Y-%m-%d"),
            end_exclusive.strftime("%Y-%m-%d"),
        )
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    )

def collection_to_groups(collection):
    info = collection.getInfo()
    rows = []
    for feature in info.get("features", []):
        props = feature.get("properties", {})
        ms = props.get("system:time_start")
        orbit = props.get("relativeOrbitNumber_start")
        if ms is None or orbit is None:
            continue
        dt = pd.to_datetime(ms, unit="ms", utc=True)
        rows.append({
            "date": pd.Timestamp(dt.date()),
            "orbit_pass": str(props.get("orbitProperties_pass", "")).upper(),
            "relative_orbit": int(orbit),
            "image_id": feature.get("id"),
        })
    if not rows:
        return pd.DataFrame(columns=["date", "orbit_pass", "relative_orbit", "scene_count"])
    df = pd.DataFrame(rows)
    return (
        df.groupby(["date", "orbit_pass", "relative_orbit"], as_index=False)
        .agg(
            scene_count=("image_id", "size"),
            image_ids=("image_id", lambda s: ";".join(sorted(set(str(x) for x in s if pd.notna(x))))),
        )
    )

def build_group_image(group):
    date = pd.Timestamp(group["date"])
    next_date = date + pd.Timedelta(days=1)
    col = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(EE_AOI)
        .filterDate(date.strftime("%Y-%m-%d"), next_date.strftime("%Y-%m-%d"))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("orbitProperties_pass", str(group["orbit_pass"])))
        .filter(ee.Filter.eq("relativeOrbitNumber_start", int(group["relative_orbit"])))
    )
    count = int(col.size().getInfo())
    if count == 0:
        raise RuntimeError(f"No images for {group}")
    return col.select(["VV", "VH"]).mosaic().clip(EE_AOI).toFloat()

def coverage_from_mask(mask):
    val = (
        mask.rename("valid").unmask(0)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=EE_AOI,
            scale=COVERAGE_CHECK_SCALE_M,
            bestEffort=True,
            maxPixels=1e8,
            tileScale=4,
        )
        .get("valid")
        .getInfo()
    )
    return 0.0 if val is None else 100.0 * float(val)

def image_coverage(image):
    return coverage_from_mask(image.select("VV").mask().gt(0))

def union_coverage(groups):
    masks = [
        build_group_image(g).select("VV").mask().gt(0).unmask(0).rename("valid")
        for g in groups
    ]
    return coverage_from_mask(ee.ImageCollection.fromImages(masks).max())

### 2.2 Preserve EVENT003 event acquisitions; do not reselect them

In [ ]:
if not PRESERVE_EVENT003_EVENT_ACQUISITIONS:
    raise RuntimeError("This v3 notebook is designed to modify the baseline only; event-acquisition reselection is blocked.")

event_tracks = []
for _, old in OLD_EVENT3_PAIR_PLAN.iterrows():
    event_track = {
        "date": pd.Timestamp(old["event_date"]),
        "orbit_pass": str(old["orbit_pass"]).upper(),
        "relative_orbit": int(old["relative_orbit"]),
    }
    img = build_group_image(event_track)
    cov = image_coverage(img)
    event_track["coverage_percent"] = float(cov)
    event_track["event_offset_days"] = int(abs((pd.Timestamp(event_track["date"]) - EVENT_REFERENCE).days))

    # recover exact scene IDs for provenance
    d0 = pd.Timestamp(event_track["date"])
    g = collection_to_groups(s1_collection(d0, d0))
    g = g[(g["orbit_pass"] == event_track["orbit_pass"]) & (g["relative_orbit"] == event_track["relative_orbit"])]
    if g.empty:
        raise RuntimeError(f"Could not recover preserved event group {event_track}")
    event_track["scene_count"] = int(g.iloc[0]["scene_count"])
    event_track["image_ids"] = g.iloc[0].get("image_ids", "")
    event_tracks.append(event_track)

if len(event_tracks) != len(OLD_EVENT3_PAIR_PLAN):
    raise RuntimeError("Preserved EVENT003 event-track count changed")

EVENT_PLAN = {
    "tracks": tuple(event_tracks),
    "track_count": len(event_tracks),
    "coverage_percent": float(union_coverage(event_tracks)),
    "date_spread_days": int((max(pd.Timestamp(t["date"]) for t in event_tracks) - min(pd.Timestamp(t["date"]) for t in event_tracks)).days),
    "temporal_cost": int(sum(t["event_offset_days"] for t in event_tracks)),
    "orbit_pass": ";".join(sorted(set(t["orbit_pass"] for t in event_tracks))),
}

display(pd.DataFrame(event_tracks))
print("Preserved combined event coverage:", EVENT_PLAN["coverage_percent"])
print("✅ EVENT003 event acquisitions frozen from v2; only baselines will change.")

### 2.3 Reselect same-orbit baselines with documentary cutoff

In [ ]:
def discover_pre_groups(event_date, orbit_pass, relative_orbit):
    event_date = pd.Timestamp(event_date)
    start = event_date - pd.Timedelta(days=MAX_PRE_EVENT_GAP_DAYS)
    col = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(EE_AOI)
        .filterDate(start.strftime("%Y-%m-%d"), event_date.strftime("%Y-%m-%d"))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("orbitProperties_pass", str(orbit_pass)))
        .filter(ee.Filter.eq("relativeOrbitNumber_start", int(relative_orbit)))
    )
    return collection_to_groups(col)


def pair_common_coverage(pre_group, event_group):
    pre = build_group_image(pre_group)
    event = build_group_image(event_group)
    common = (
        pre.select("VV").mask().gt(0)
        .And(event.select("VV").mask().gt(0))
        .unmask(0)
        .rename("common")
    )
    return coverage_from_mask(common)

pair_rows = []
qa_rows = []

for pair_id, event_track in enumerate(EVENT_PLAN["tracks"], 1):
    event_track = dict(event_track)
    event_date = pd.Timestamp(event_track["date"])
    orbit_pass = str(event_track["orbit_pass"])
    relative_orbit = int(event_track["relative_orbit"])

    pre_candidates = discover_pre_groups(event_date, orbit_pass, relative_orbit)
    if pre_candidates.empty:
        raise RuntimeError(f"No pre-event candidates for orbit {relative_orbit}")

    evaluated = []
    for _, pre_row in tqdm(
        pre_candidates.iterrows(), total=len(pre_candidates),
        desc=f"EVENT003 pair {pair_id} baseline QA", unit="date"
    ):
        pg = pre_row.to_dict()
        pre_date = pd.Timestamp(pg["date"])
        gap = int((event_date - pre_date).days)
        eligible = bool(pre_date <= BASELINE_CUTOFF)
        cov = pair_common_coverage(pg, event_track)
        rec = {
            "pair_id": pair_id,
            "orbit_pass": orbit_pass,
            "relative_orbit": relative_orbit,
            "pre_date": pre_date,
            "event_date": event_date,
            "pre_event_gap_days": gap,
            "common_coverage_percent": float(cov),
            "event_offset_days": int(event_track["event_offset_days"]),
            "pre_scene_ids": pg.get("image_ids", ""),
            "event_scene_ids": event_track.get("image_ids", ""),
            "eligible_documentary_pre_flood": eligible,
            "decision_reason": (
                "ELIGIBLE: date on/before documentary cutoff"
                if eligible else
                "REJECT: date after 2020-06-26 documentary baseline cutoff"
            ),
        }
        evaluated.append(rec)
        qa_rows.append(rec.copy())

    eval_df = pd.DataFrame(evaluated)
    eligible_df = eval_df[eval_df["eligible_documentary_pre_flood"]].copy()
    if eligible_df.empty:
        raise RuntimeError(
            f"No documentary-eligible baseline for orbit {relative_orbit} within {MAX_PRE_EVENT_GAP_DAYS} days."
        )

    best_cov = float(eligible_df["common_coverage_percent"].max())
    equivalent = eligible_df[
        eligible_df["common_coverage_percent"] >= best_cov - COVERAGE_EQUIVALENCE_TOLERANCE_PP
    ].copy()
    chosen = (
        equivalent.sort_values(
            ["pre_event_gap_days", "common_coverage_percent"],
            ascending=[True, False]
        ).iloc[0].to_dict()
    )
    chosen["selection_status"] = "SELECTED_V3"
    pair_rows.append(chosen)

baseline_qa = pd.DataFrame(qa_rows).sort_values(["relative_orbit", "pre_date"])
pair_plan = pd.DataFrame(pair_rows).sort_values(["relative_orbit"]).reset_index(drop=True)

# Hard gates proving that only the baseline changed.
old_event_key = OLD_EVENT3_PAIR_PLAN[["orbit_pass","relative_orbit","event_date"]].copy()
old_event_key["event_date"] = pd.to_datetime(old_event_key["event_date"]).dt.normalize()
new_event_key = pair_plan[["orbit_pass","relative_orbit","event_date"]].copy()
new_event_key["event_date"] = pd.to_datetime(new_event_key["event_date"]).dt.normalize()
old_event_key = old_event_key.sort_values(["orbit_pass","relative_orbit"]).reset_index(drop=True)
new_event_key = new_event_key.sort_values(["orbit_pass","relative_orbit"]).reset_index(drop=True)
if not old_event_key.equals(new_event_key):
    raise RuntimeError("EVENT003 event acquisition changed. V3 must modify baseline only.")

if (pd.to_datetime(pair_plan["pre_date"]) > BASELINE_CUTOFF).any():
    raise RuntimeError("Selected v3 baseline violates the documentary cutoff")

baseline_qa.to_csv(BASELINE_QA_CSV, index=False)
pair_plan.to_csv(PAIR_PLAN_CSV, index=False)
pair_plan.to_csv(FINAL_PAIR_PLAN, index=False)
pair_plan.to_csv(V3_AUDIT_DIR / "EVENT003_v3_selected_pair_plan.csv", index=False)

display(baseline_qa)
print("SELECTED V3 EVENT003 PAIRS")
display(pair_plan)
print("Saved baseline QA:", BASELINE_QA_CSV)

### 2.4 Build matched Sentinel-1 stack

In [ ]:
pair_images = []

for _, row in pair_plan.iterrows():
    pre_group = {
        "date": pd.Timestamp(row["pre_date"]),
        "orbit_pass": row["orbit_pass"],
        "relative_orbit": int(row["relative_orbit"]),
    }
    event_group = {
        "date": pd.Timestamp(row["event_date"]),
        "orbit_pass": row["orbit_pass"],
        "relative_orbit": int(row["relative_orbit"]),
    }

    pre = build_group_image(pre_group).rename(["pre_VV", "pre_VH"])
    event = build_group_image(event_group).rename(["event_VV", "event_VH"])

    common = (
        pre.select("pre_VV").mask().gt(0)
        .And(event.select("event_VV").mask().gt(0))
    )

    quality = (
        ee.Image.constant(-float(row["event_offset_days"]))
        .rename("quality")
        .toFloat()
    )

    pair_img = (
        pre.addBands(event)
        .addBands(quality)
        .updateMask(common)
        .clip(EE_AOI)
    )
    pair_images.append(pair_img)

matched = (
    ee.ImageCollection.fromImages(pair_images)
    .qualityMosaic("quality")
    .select(["pre_VV", "pre_VH", "event_VV", "event_VH"])
    .clip(EE_AOI)
    .unmask(-9999)
    .toFloat()
)

print("Bands:", matched.bandNames().getInfo())

### 2.5 Export matched stack

In [ ]:
if TMP_STACK.exists():
    print("Existing temporary stack:", TMP_STACK)
    print("Size:", human_bytes(path_size(TMP_STACK)))
else:
    print("Downloading:", TMP_STACK)
    geemap.download_ee_image(
        image=matched,
        filename=str(TMP_STACK),
        region=EE_AOI,
        scale=S1_SCALE_M,
        crs="EPSG:32645",
        overwrite=False,
        num_threads=4,
    )

if not TMP_STACK.exists():
    raise RuntimeError("Temporary S1 stack was not created.")

with rasterio.open(TMP_STACK) as src:
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Shape:", src.width, src.height)
    print("Bands:", src.count)
    if src.count != 4:
        raise RuntimeError("Expected exactly four bands.")

print("Temporary stack size:", human_bytes(path_size(TMP_STACK)))

### 2.6 Frozen Otsu classifier

In [ ]:
def otsu_threshold(values, bins):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if values.size < 1000:
        raise RuntimeError("Too few valid samples for Otsu.")

    lo, hi = np.percentile(values, [1, 99])
    values = values[(values >= lo) & (values <= hi)]

    hist, edges = np.histogram(values, bins=bins)
    hist = hist.astype(np.float64)
    centers = (edges[:-1] + edges[1:]) / 2.0

    w1 = np.cumsum(hist)
    w2 = np.cumsum(hist[::-1])[::-1]

    m1 = np.cumsum(hist * centers) / np.maximum(w1, 1)
    m2 = (
        np.cumsum((hist * centers)[::-1])
        / np.maximum(w2[::-1], 1)
    )[::-1]

    between = w1[:-1] * w2[1:] * (m1[:-1] - m2[1:]) ** 2
    return float(centers[int(np.nanargmax(between))])

rng = np.random.default_rng(int(frozen["random_seed"]))
sample_scores = []

with rasterio.open(TMP_STACK) as s1_src, rasterio.open(JRC_WATER) as jrc_src:
    windows = [w for _, w in s1_src.block_windows(1)]
    max_windows = int(frozen["max_sample_windows"])

    if len(windows) > max_windows:
        idx = np.sort(rng.choice(len(windows), size=max_windows, replace=False))
        windows = [windows[i] for i in idx]

    with WarpedVRT(
        jrc_src,
        crs=s1_src.crs,
        transform=s1_src.transform,
        width=s1_src.width,
        height=s1_src.height,
        resampling=Resampling.nearest,
    ) as jrc_vrt:

        for window in tqdm(windows, desc=f"{EVENT_ID} Otsu sample", unit="block"):
            pre_vv = s1_src.read(1, window=window)
            pre_vh = s1_src.read(2, window=window)
            event_vv = s1_src.read(3, window=window)
            event_vh = s1_src.read(4, window=window)
            permanent = jrc_vrt.read(1, window=window) > 0

            valid = (
                (pre_vv != -9999)
                & (pre_vh != -9999)
                & (event_vv != -9999)
                & (event_vh != -9999)
                & np.isfinite(pre_vv)
                & np.isfinite(pre_vh)
                & np.isfinite(event_vv)
                & np.isfinite(event_vh)
                & (~permanent)
            )

            if not valid.any():
                continue

            score = (
                float(frozen["vv_weight"]) * (event_vv[valid] - pre_vv[valid])
                + float(frozen["vh_weight"]) * (event_vh[valid] - pre_vh[valid])
            )

            cap = int(frozen["max_samples_per_window"])
            if score.size > cap:
                take = rng.choice(score.size, cap, replace=False)
                score = score[take]

            sample_scores.append(score.astype(np.float32))

if not sample_scores:
    raise RuntimeError("No valid SAR samples collected.")

sample_scores = np.concatenate(sample_scores)
event_threshold = otsu_threshold(sample_scores, int(frozen["histogram_bins"]))

print("Sample count:", f"{sample_scores.size:,}")
print("Otsu threshold:", event_threshold, "dB")

### 2.7 Full-resolution classification

In [ ]:
FINAL_FLOOD = EVENT_OUT / "flood_mask.tif"
FINAL_VALID = EVENT_OUT / "valid_mask.tif"
FINAL_META = EVENT_OUT / "event_flood_metadata.json"

with rasterio.open(TMP_STACK) as ref:
    out_profile = ref.profile.copy()

out_profile.update(
    dtype="uint8",
    count=1,
    nodata=0,
    compress="deflate",
)

total_valid = 0
total_flood = 0

with (
    rasterio.open(TMP_STACK) as s1_src,
    rasterio.open(JRC_WATER) as jrc_src,
    rasterio.open(FINAL_FLOOD, "w", **out_profile) as flood_dst,
    rasterio.open(FINAL_VALID, "w", **out_profile) as valid_dst,
):
    windows = [w for _, w in s1_src.block_windows(1)]

    with WarpedVRT(
        jrc_src,
        crs=s1_src.crs,
        transform=s1_src.transform,
        width=s1_src.width,
        height=s1_src.height,
        resampling=Resampling.nearest,
    ) as jrc_vrt:

        for window in tqdm(
            windows,
            desc=f"{EVENT_ID} classify",
            unit="block",
        ):
            pre_vv = s1_src.read(1, window=window)
            pre_vh = s1_src.read(2, window=window)
            event_vv = s1_src.read(3, window=window)
            event_vh = s1_src.read(4, window=window)
            permanent = jrc_vrt.read(1, window=window) > 0

            valid = (
                (pre_vv != -9999)
                & (pre_vh != -9999)
                & (event_vv != -9999)
                & (event_vh != -9999)
                & np.isfinite(pre_vv)
                & np.isfinite(pre_vh)
                & np.isfinite(event_vv)
                & np.isfinite(event_vh)
                & (~permanent)
            )

            score = np.full(pre_vv.shape, np.nan, dtype=np.float32)
            score[valid] = (
                float(frozen["vv_weight"]) * (event_vv[valid] - pre_vv[valid])
                + float(frozen["vh_weight"]) * (event_vh[valid] - pre_vh[valid])
            )

            flood = valid & (score <= event_threshold)

            valid_dst.write(valid.astype("uint8"), 1, window=window)
            flood_dst.write(flood.astype("uint8"), 1, window=window)

            total_valid += int(valid.sum())
            total_flood += int(flood.sum())

flood_fraction = total_flood / max(total_valid, 1)

print("Valid land pixels:", f"{total_valid:,}")
print("Flood pixels:", f"{total_flood:,}")
print("Flood fraction:", f"{100*flood_fraction:.2f}%")

if flood_fraction < MIN_PLAUSIBLE_FLOOD_FRACTION:
    print("⚠️ Very small flood fraction; inspect before cleanup.")
if flood_fraction > MAX_PLAUSIBLE_FLOOD_FRACTION:
    print("⚠️ Very large flood fraction; inspect for non-flood seasonal change.")

### 2.8 Hard raster QC

In [ ]:
def full_qc(flood_path, valid_path):
    flood_values = set()
    valid_values = set()
    violations = 0

    with rasterio.open(flood_path) as fsrc, rasterio.open(valid_path) as vsrc:
        if (
            fsrc.crs != vsrc.crs
            or fsrc.transform != vsrc.transform
            or fsrc.width != vsrc.width
            or fsrc.height != vsrc.height
        ):
            raise RuntimeError("Flood/valid grid mismatch.")

        for _, window in tqdm(
            list(fsrc.block_windows(1)),
            desc=f"{EVENT_ID} final QC",
            unit="block",
        ):
            f = fsrc.read(1, window=window)
            v = vsrc.read(1, window=window)

            flood_values.update(int(x) for x in np.unique(f))
            valid_values.update(int(x) for x in np.unique(v))
            violations += int(np.count_nonzero((f > 0) & (v == 0)))

    return flood_values, valid_values, violations

flood_values, valid_values, violations = full_qc(FINAL_FLOOD, FINAL_VALID)

print("Flood values:", sorted(flood_values))
print("Valid values:", sorted(valid_values))
print("F=1 while A=0:", violations)

if not flood_values.issubset({0, 1}):
    raise RuntimeError("Flood mask is not binary.")
if not valid_values.issubset({0, 1}):
    raise RuntimeError("Valid mask is not binary.")
if violations != 0:
    raise RuntimeError("Flood is not a subset of valid observations.")

print("✅ Raster QC PASS")

### 2.9 Save v3 metadata and version provenance

In [ ]:
serial_plan = pair_plan.copy()
serial_plan["pre_date"] = pd.to_datetime(serial_plan["pre_date"]).dt.strftime("%Y-%m-%d")
serial_plan["event_date"] = pd.to_datetime(serial_plan["event_date"]).dt.strftime("%Y-%m-%d")

metadata = {
    "event_id": EVENT_ID,
    "event_name": event_spec["name"],
    "analysis_search_start": str(EVENT_START.date()),
    "analysis_search_end": str(EVENT_END.date()),
    "documented_event_reference": str(EVENT_REFERENCE.date()),
    "evidence_source": event_spec["evidence_source"],
    "evidence_url": event_spec["evidence_url"],
    "algorithm": frozen["algorithm"],
    "threshold_strategy": frozen.get("threshold_strategy", "event_specific_otsu_same_algorithm"),
    "event_otsu_threshold_db": float(event_threshold),
    "valid_land_pixels": int(total_valid),
    "flood_pixels": int(total_flood),
    "flood_fraction": float(flood_fraction),
    "frozen_config_sha256": frozen["config_sha256"],
    "event_track_combined_coverage_percent": float(EVENT_PLAN["coverage_percent"]),
    "event_track_date_spread_days": int(EVENT_PLAN["date_spread_days"]),
    "pair_plan": serial_plan.to_dict(orient="records"),
    "uses_dem_or_vrfsz": False,
    "permanent_water_excluded": True,
    "flood_mask": str(FINAL_FLOOD.relative_to(PROJECT_ROOT)),
    "valid_mask": str(FINAL_VALID.relative_to(PROJECT_ROOT)),
    "v3_baseline_revision": True,
    "v3_revision_scope": "EVENT003 pre-event baselines only; event acquisitions and frozen classifier preserved",
    "baseline_documentary_cutoff": str(BASELINE_CUTOFF.date()),
    "baseline_condition_scope": "documentary pre-flood timing gate; not a complete hydrometeorological dry-state proof",
    "baseline_candidate_qa_csv": str(BASELINE_QA_CSV.relative_to(PROJECT_ROOT)),
    "v2_archive_root": str(ARCHIVE_ROOT.relative_to(PROJECT_ROOT)),
}
FINAL_META.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")

check = json.loads(FINAL_META.read_text(encoding="utf-8"))
if check["frozen_config_sha256"] != frozen["config_sha256"]:
    raise RuntimeError("Frozen SHA mismatch")
if not check.get("v3_baseline_revision"):
    raise RuntimeError("v3 baseline revision flag missing")

print("✅ V3 EVENT003 metadata saved:", FINAL_META)
display(pd.DataFrame([{
    "event_id": EVENT_ID,
    "threshold_db": event_threshold,
    "flood_fraction": flood_fraction,
    "selected_pre_dates": "; ".join(serial_plan["pre_date"].tolist()),
    "preserved_event_dates": "; ".join(serial_plan["event_date"].tolist()),
    "frozen_sha": frozen["config_sha256"],
}]))

In [ ]:
# Lightweight preview from final flood mask only.
with rasterio.open(FINAL_FLOOD) as src:
    scale = max(src.width / 1400, src.height / 1400, 1)
    width = max(1, int(src.width / scale))
    height = max(1, int(src.height / scale))
    preview = src.read(
        1,
        out_shape=(height, width),
        resampling=Resampling.nearest,
    )
    bounds = src.bounds

extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(preview, extent=extent, origin="upper", interpolation="nearest")
ax.set_title(f"{EVENT_ID} — Frozen-Workflow Flood Mask")
plt.tight_layout()
plt.show()

In [ ]:
final_ready = (
    FINAL_FLOOD.exists()
    and FINAL_VALID.exists()
    and FINAL_META.exists()
    and FINAL_PAIR_PLAN.exists()
)

if not final_ready:
    raise RuntimeError("Final outputs incomplete; cleanup blocked.")

meta_check = json.loads(FINAL_META.read_text(encoding="utf-8"))
if meta_check.get("frozen_config_sha256") != frozen["config_sha256"]:
    raise RuntimeError("Frozen SHA mismatch; cleanup blocked.")

print("Temporary work size:", human_bytes(path_size(EVENT_WORK)))

if DELETE_TEMP_AFTER_SUCCESS:
    removed = safe_remove(EVENT_WORK)
    gc.collect()
    print("✅ Temporary SAR event deleted:", human_bytes(removed))
else:
    print("Temporary data preserved because DELETE_TEMP_AFTER_SUCCESS=False.")

disk_report(f"AFTER {EVENT_ID}")

### 2.10 V2 → V3 EVENT003 change audit

In [ ]:
OLD_EVENT3_ARCHIVE = ARCHIVE_ROOT / "data_processed_flood_events_EVENT003"
OLD_FLOOD = OLD_EVENT3_ARCHIVE / "flood_mask.tif"
OLD_VALID = OLD_EVENT3_ARCHIVE / "valid_mask.tif"
OLD_META = OLD_EVENT3_ARCHIVE / "event_flood_metadata.json"

change_rows = []
old_meta_json = json.loads(OLD_META.read_text(encoding="utf-8")) if OLD_META.exists() else {}

with rasterio.open(FINAL_FLOOD) as nfs, rasterio.open(FINAL_VALID) as nvs:
    new_profile_sig = (str(nfs.crs), tuple(nfs.transform), nfs.width, nfs.height)
    common_valid_n = 0
    changed_n = 0
    old_flood_n = 0
    new_flood_n = 0

    with rasterio.open(OLD_FLOOD) as ofs, rasterio.open(OLD_VALID) as ovs:
        with WarpedVRT(ofs, crs=nfs.crs, transform=nfs.transform, width=nfs.width, height=nfs.height, resampling=Resampling.nearest) as ofv, \
             WarpedVRT(ovs, crs=nfs.crs, transform=nfs.transform, width=nfs.width, height=nfs.height, resampling=Resampling.nearest) as ovv:
            for _, window in nfs.block_windows(1):
                nf = nfs.read(1, window=window) > 0
                nv = nvs.read(1, window=window) > 0
                of = ofv.read(1, window=window) > 0
                ov = ovv.read(1, window=window) > 0
                cv = nv & ov
                common_valid_n += int(cv.sum())
                changed_n += int(np.count_nonzero((nf != of) & cv))
                old_flood_n += int(np.count_nonzero(of & cv))
                new_flood_n += int(np.count_nonzero(nf & cv))

change_summary = {
    "old_otsu_threshold_db": old_meta_json.get("event_otsu_threshold_db"),
    "new_otsu_threshold_db": float(event_threshold),
    "old_flood_fraction_metadata": old_meta_json.get("flood_fraction"),
    "new_flood_fraction_metadata": float(flood_fraction),
    "common_valid_pixels": common_valid_n,
    "changed_class_pixels_common_valid": changed_n,
    "changed_class_fraction_common_valid": changed_n / max(common_valid_n, 1),
    "old_flood_pixels_common_valid": old_flood_n,
    "new_flood_pixels_common_valid": new_flood_n,
}
pd.DataFrame([change_summary]).to_csv(V3_AUDIT_DIR / "EVENT003_v2_vs_v3_mask_change_summary.csv", index=False)
display(pd.DataFrame([change_summary]))
print("✅ EVENT003 v2→v3 change audit saved.")

### 2.11 Harmonize EVENT003 if needed and hard-audit all four events

In [ ]:
def grid_signature(path):
    with rasterio.open(path) as src:
        return (
            str(src.crs),
            src.transform,
            src.width,
            src.height,
        )


def harmonize_binary_mask(source, reference, destination):
    with rasterio.open(reference) as ref, rasterio.open(source) as src:
        profile = ref.profile.copy()
        profile.update(
            dtype="uint8",
            count=1,
            nodata=0,
            compress="deflate",
        )

        with (
            WarpedVRT(
                src,
                crs=ref.crs,
                transform=ref.transform,
                width=ref.width,
                height=ref.height,
                resampling=Resampling.nearest,
            ) as vrt,
            rasterio.open(destination, "w", **profile) as dst,
        ):
            windows = [w for _, w in dst.block_windows(1)]

            for window in tqdm(
                windows,
                desc=f"Harmonize {Path(source).name}",
                unit="block",
                leave=False,
            ):
                arr = (vrt.read(1, window=window) > 0).astype("uint8")
                dst.write(arr, 1, window=window)


canonical_flood = PROCESSED_ROOT / "EVENT001/flood_mask.tif"
canonical_grid = grid_signature(canonical_flood)

for event_id in ["EVENT002", "EVENT003", "EVENT004"]:
    folder = PROCESSED_ROOT / event_id
    flood = folder / "flood_mask.tif"
    valid = folder / "valid_mask.tif"

    if grid_signature(flood) == canonical_grid:
        print(event_id, "already matches EVENT001 canonical grid.")
        continue

    print(event_id, "requires canonical-grid harmonization.")

    flood_tmp = folder / "_flood_mask_canonical.tif"
    valid_tmp = folder / "_valid_mask_canonical.tif"

    harmonize_binary_mask(flood, canonical_flood, flood_tmp)
    harmonize_binary_mask(valid, canonical_flood, valid_tmp)

    # Full subset/binary audit before replacement.
    fvals, vvals, violations = full_qc(flood_tmp, valid_tmp)

    if (
        not fvals.issubset({0, 1})
        or not vvals.issubset({0, 1})
        or violations != 0
    ):
        raise RuntimeError(f"{event_id}: harmonized outputs failed QC.")

    flood_tmp.replace(flood)
    valid_tmp.replace(valid)

    print("✅", event_id, "harmonized to canonical grid.")

In [ ]:
final_rows = []

for event_id in ["EVENT001", "EVENT002", "EVENT003", "EVENT004"]:
    folder = PROCESSED_ROOT / event_id

    flood = folder / "flood_mask.tif"
    valid = folder / "valid_mask.tif"
    meta = folder / "event_flood_metadata.json"

    for p in [flood, valid, meta]:
        if not p.exists():
            raise RuntimeError(f"{event_id}: missing {p.name}")

    metadata = json.loads(meta.read_text(encoding="utf-8"))
    sha_ok = metadata.get("frozen_config_sha256") == frozen["config_sha256"]

    fvals, vvals, violations = full_qc(flood, valid)

    binary_ok = fvals.issubset({0, 1}) and vvals.issubset({0, 1})
    subset_ok = violations == 0
    same_grid = grid_signature(flood) == canonical_grid

    final_rows.append({
        "event_id": event_id,
        "same_canonical_grid": same_grid,
        "frozen_sha_match": sha_ok,
        "binary_ok": binary_ok,
        "flood_subset_valid": subset_ok,
        "otsu_threshold_db": metadata.get("event_otsu_threshold_db"),
        "flood_fraction": metadata.get("flood_fraction"),
        "documented_reference": metadata.get("documented_event_reference"),
    })

final_audit = pd.DataFrame(final_rows)
display(final_audit)

hard_cols = [
    "same_canonical_grid",
    "frozen_sha_match",
    "binary_ok",
    "flood_subset_valid",
]

if not bool(final_audit[hard_cols].all().all()):
    raise RuntimeError("FINAL FOUR-EVENT TECHNICAL GATE FAILED.")

print("\n✅ ALL FOUR EVENTS ARE TECHNICALLY READY FOR RECURRENCE.")
print("Review the four flood fractions/maps before interpreting recurrence.")
print("Then run: 11_to_LAST_Combined_MultiEvent_Recurrence.ipynb")

In [ ]:
disk_report("FINAL DISK STATUS")

print("Temporary work-root size:", human_bytes(path_size(WORK_ROOT)))

for event_id in ["EVENT001", "EVENT002", "EVENT003", "EVENT004"]:
    folder = PROCESSED_ROOT / event_id
    print(
        event_id,
        "final folder size:",
        human_bytes(path_size(folder)),
    )

In [ ]:
stage_mark('EVENT003_V3_REBUILD', 'COMPLETED', 'Baseline revised; event acquisitions preserved; four-event technical gate passed')

# 3 — Notebook 11: rebuild four-event recurrence

Source lineage: `11_to_LAST_Combined_MultiEvent_Recurrence.ipynb`. Code is embedded below so this master notebook does not call the source notebook at runtime.

In [ ]:
if RUN_STAGE_11:
    stage_mark('11', 'STARTED', 'embedded source: 11_to_LAST_Combined_MultiEvent_Recurrence.ipynb')

# Notebook 11 → Last — Combined Multi-Event Flood Recurrence Workflow

This notebook combines the remaining notebooks **after Notebook 10 — Map All Flood Events**.

It contains:

- **Section 11 — Multi-event readiness audit**
- **Section 11A — Build flood stack \(F_{ij}\)**
- **Section 11B — Build valid-observation stack \(A_{ij}\)**
- **Section 11C — Hard consistency checks**
- **Section 12 — Compute event-normalized flood recurrence \(R_i\)**
- **Section 12A — Recurrence QC**
- **Section 12B — Visualization and final observation-branch summary**

---

## What the uploaded outputs show at the current stage

The pilot workflow already produced a deterministic SAR threshold and a full-resolution flood mask.
The frozen workflow was then accepted and written to disk. Notebook 10 successfully applied that
frozen configuration, but the current output shows only **EVENT001** was discovered and processed.

Therefore the project is **not yet ready for a final multi-event recurrence result**.

The scientific paper design requires at least **three independently selected and processed flood
events** before recurrence is treated as a final result.

This notebook enforces that requirement automatically.

---

## Required state before final execution

You must first have:

1. EVENT001 processed by the frozen SAR method;
2. EVENT002 processed by the **same frozen SAR method**;
3. EVENT003 processed by the **same frozen SAR method**;
4. optional EVENT004 may be added;
5. each accepted event folder must contain:

```text
data/processed/flood_events/EVENTxxx/
├── flood_mask.tif
├── valid_mask.tif
└── event_flood_metadata.json
```

The metadata for all events must point to the **same frozen configuration SHA-256**.

---

## Scientific definition

For pixel \(i\) and event \(j\):

\[
F_{ij} =
\begin{cases}
1, & \text{flooded}\\
0, & \text{not flooded}
\end{cases}
\]

and

\[
A_{ij} =
\begin{cases}
1, & \text{valid Sentinel-1 land observation}\\
0, & \text{not valid / not observed}
\end{cases}
\]

The recurrence is then:

\[
R_i =
\frac{\sum_j F_{ij}A_{ij}}
{\sum_j A_{ij}},
\qquad
\sum_j A_{ij} > 0
\]

Missing observations are therefore **never treated as dry**.

---

## Anti-circularity

This notebook does **not** use:

- DeltaDTM;
- MSL-aligned DeltaDTM;
- VRFSZ classes;

to define or modify flood pixels.

The output of this notebook is the independent Sentinel-1 flood-recurrence surface that will later
be compared against the terrain classes.

---
# SECTION 11 — Multi-Event Readiness Audit

This section checks whether Notebook 10 has actually produced enough successful flood-event products
to justify recurrence analysis.

### Why this gate matters

A recurrence surface from one event is not recurrence. It is just a binary event map represented as
0/1.

The notebook therefore refuses final processing with fewer than three successful events unless
development mode is explicitly enabled.

In [ ]:
if RUN_STAGE_11:
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    import rasterio
    from rasterio.vrt import WarpedVRT
    from rasterio.enums import Resampling
    from tqdm.auto import tqdm
    import matplotlib.pyplot as plt

    # ============================================================
    # USER / SCIENTIFIC SETTINGS
    # ============================================================

    MIN_EVENTS_FOR_PAPER = 3

    # Keep False for the real analysis.
    # Set True only if you want to test notebook mechanics with <3 events.
    ALLOW_DEVELOPMENT_RUN = False

    # If True, every event metadata file must contain the same frozen config SHA.
    REQUIRE_IDENTICAL_FROZEN_CONFIG = True

    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

    EVENT_ROOT = PROJECT_ROOT / "data/processed/flood_events"
    REC_DIR = PROJECT_ROOT / "data/processed/recurrence"
    CONFIG_FILE = PROJECT_ROOT / "config/frozen_sar_workflow.json"

    REC_DIR.mkdir(parents=True, exist_ok=True)

    if not EVENT_ROOT.exists():
        raise FileNotFoundError(
            f"Processed flood-event directory not found:\n{EVENT_ROOT}"
        )

    if not CONFIG_FILE.exists():
        raise FileNotFoundError(
            "Frozen SAR configuration not found. "
            "Notebook 09 must be completed before this notebook."
        )

    frozen_config = json.loads(
        CONFIG_FILE.read_text(encoding="utf-8")
    )

    EXPECTED_CONFIG_SHA = frozen_config.get("config_sha256")

    print("Frozen algorithm:", frozen_config.get("algorithm"))
    print("Frozen config SHA:", EXPECTED_CONFIG_SHA)

# STAGE 9 — GFDS-only independent coarse microwave corroboration

**Source provenance:** `28_Targeted_MultiSource_Flood_Evidence_Search_and_Local_Validation.ipynb, GFDS branch only`.

GFDS is never treated as 10 m ground truth.

In [ ]:
# Optional one-time installation if needed:
# %pip install -q numpy pandas geopandas rasterio shapely scipy xarray netCDF4 requests matplotlib scikit-learn cdsapi

from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
import os
import re
import json
import math
import warnings
import calendar

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin, array_bounds
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import from_bounds as window_from_bounds, Window

from shapely.geometry import Point, box, mapping
from scipy import ndimage
from scipy.stats import spearmanr

try:
    import xarray as xr
except Exception:
    xr = None

try:
    from sklearn.metrics import roc_auc_score
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore", category=FutureWarning)

print("Imports OK.")
print("xarray:", xr is not None)
print("scikit-learn:", SKLEARN_AVAILABLE)

In [ ]:

# GFDS-only external corroboration configuration.
# This intentionally excludes GloFAS/CYGNSS/gauge branches from the final quantitative workflow.

PROJECT_ROOT_OVERRIDE = None
AOI_REL = "data/raw/boundaries/gbm_delta.shp"
EVENT_ROOT_REL = "data/processed/flood_events"

GFDS_EVENTS = {
    "2017": {"event_id": "EVENT004", "date": "2017-08-16"},
    "2020": {"event_id": "EVENT003", "date": "2020-07-24"},
    "2024": {"event_id": "EVENT002", "date": "2024-08-23"},
    "2026": {"event_id": "EVENT001", "date": "2026-07-12"},
}

RUN_GFDS = True
GFDS_CANDIDATE_SD = 2.0
GFDS_STRONG_SD = 3.0
GFDS_SEARCH_WINDOWS = [3, 7, 15]
GFDS_DATASET = "ALL"
GFDS_PRODUCT_FOLDER = "AvgMagTiffs"
GFDS_BASE = "https://www.gdacs.org/flooddetection/DATA"
GFDS_ALLOW_FULL_DOWNLOAD_FALLBACK = False
REQUEST_TIMEOUT = 60

def resolve_gfds_project_root():
    if PROJECT_ROOT_OVERRIDE:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (
            (p / AOI_REL).exists()
            and (p / EVENT_ROOT_REL / "EVENT001" / "flood_mask.tif").exists()
        ):
            return p
    raise RuntimeError("Could not locate project root for GFDS stage.")

GFDS_PROJECT_ROOT = resolve_gfds_project_root()
AOI_PATH = GFDS_PROJECT_ROOT / AOI_REL
EVENT_ROOT = GFDS_PROJECT_ROOT / EVENT_ROOT_REL

for year, spec in GFDS_EVENTS.items():
    event_dir = EVENT_ROOT / spec["event_id"]
    spec["flood_mask"] = str(event_dir / "flood_mask.tif")
    spec["valid_mask"] = str(event_dir / "valid_mask.tif")

GFDS_OUT_ROOT = GFDS_PROJECT_ROOT / "validation_targeted_gfds"
OUTPUTS = GFDS_OUT_ROOT / "outputs"
GFDS_CACHE = GFDS_OUT_ROOT / "cache"
OUTPUTS.mkdir(parents=True, exist_ok=True)
GFDS_CACHE.mkdir(parents=True, exist_ok=True)

aoi = gpd.read_file(AOI_PATH)
if aoi.empty or aoi.crs is None:
    raise RuntimeError("AOI is empty or CRS is missing.")
aoi_4326 = aoi.to_crs(4326)
try:
    AOI_GEOM_4326 = aoi_4326.geometry.union_all()
except AttributeError:
    AOI_GEOM_4326 = aoi_4326.geometry.unary_union
WEST, SOUTH, EAST, NORTH = map(float, AOI_GEOM_4326.bounds)
AOI_BBOX = (WEST, SOUTH, EAST, NORTH)

print("GFDS stage ready:", GFDS_PROJECT_ROOT)


In [ ]:
def make_session():
    s = requests.Session()
    retries = Retry(
        total=4,
        connect=4,
        read=4,
        status=4,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "HEAD"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retries)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({
        "User-Agent": "GBM-Targeted-MultiSource-Validation/1.0"
    })
    return s


SESSION = make_session()


def temporal_distance_days(event_date, obs_date):
    return abs(
        (
            pd.Timestamp(event_date).normalize()
            - pd.Timestamp(obs_date).normalize()
        ).days
    )


def read_sar_mask_pair(year):
    spec = GFDS_EVENTS[str(year)]

    with rasterio.open(spec["flood_mask"]) as src:
        flood = src.read(1)
        flood_profile = src.profile.copy()
        flood_transform = src.transform
        flood_crs = src.crs

    with rasterio.open(spec["valid_mask"]) as src:
        valid = src.read(1)
        if src.crs != flood_crs or src.transform != flood_transform:
            raise RuntimeError(
                f"{year}: flood_mask and valid_mask grids differ."
            )

    return (
        (flood == 1).astype("float32"),
        (valid == 1).astype("float32"),
        flood_transform,
        flood_crs,
        flood_profile,
    )


def aggregate_sar_to_reference_grid(
    year,
    dst_shape,
    dst_transform,
    dst_crs,
):
    """
    Return:
      sar_flood_fraction = flooded valid SAR area / valid SAR area
      sar_valid_fraction = fraction of destination cell covered by valid SAR

    This is the correct direction for comparing 10 m SAR with a coarser
    external source. We do not upsample the coarse source to 10 m and pretend
    it contains 10 m reference information.
    """
    flood, valid, src_transform, src_crs, _ = read_sar_mask_pair(year)

    numerator_src = flood * valid

    numerator_dst = np.zeros(dst_shape, dtype="float32")
    valid_dst = np.zeros(dst_shape, dtype="float32")

    reproject(
        numerator_src,
        numerator_dst,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.average,
        src_nodata=None,
        dst_nodata=0,
    )

    reproject(
        valid,
        valid_dst,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.average,
        src_nodata=None,
        dst_nodata=0,
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        frac = np.where(valid_dst > 0, numerator_dst / valid_dst, np.nan)

    return np.clip(frac, 0, 1), np.clip(valid_dst, 0, 1)


def sar_fraction_in_buffers(year, points_gdf, buffer_km):
    """
    Compute event-mask flood fraction inside buffers centered on independently
    selected points. Buffering is performed in a local metric CRS.
    """
    if points_gdf.empty:
        return pd.Series(dtype=float)

    flood, valid, transform, crs, profile = read_sar_mask_pair(year)

    pts = points_gdf.to_crs(crs)

    # Buffer in source raster CRS if projected in metres. If the source CRS is
    # geographic, use EPSG:3857 for buffering, then return to source CRS.
    try:
        is_geographic = crs.is_geographic
    except Exception:
        is_geographic = False

    if is_geographic:
        temp = points_gdf.to_crs(3857)
        geoms = temp.geometry.buffer(buffer_km * 1000).to_crs(crs)
    else:
        geoms = pts.geometry.buffer(buffer_km * 1000)

    results = []

    for geom in geoms:
        mask = rasterize(
            [(mapping(geom), 1)],
            out_shape=flood.shape,
            transform=transform,
            fill=0,
            dtype="uint8",
        ).astype(bool)

        ok = mask & (valid == 1)

        if not ok.any():
            results.append(np.nan)
        else:
            results.append(float(flood[ok].mean()))

    return pd.Series(results, index=points_gdf.index, dtype=float)

# 6. GDACS/JRC Global Flood Detection System (GFDS)

GFDS is one of the most useful additional sources for this project because:

- it is independent of Sentinel-1;
- it uses passive microwave observations;
- it provides daily global flood-magnitude rasters from 1997 onward;
- the merged 4-day product improves sampling over the tropics.

However, it is coarse and contains known multi-sensor calibration artifacts.

Therefore the notebook uses it as:

> **independent coarse microwave flood-hotspot corroboration**

not as 10 m ground truth.

## Hotspot rule

For each event:

1. inspect dates progressively in ±3, then ±7, then ±15 days;
2. crop only the AOI;
3. convert integer magnitude to standard deviations (`raw / 1000`);
4. identify cells ≥2 SD, with ≥3 SD treated as strong;
5. rank dates using temporal proximity + count/strength of anomalous cells;
6. only after the hotspot is selected, aggregate the frozen Sentinel-1 mask to the GFDS grid.

The comparison metric is hotspot enrichment/correlation, not confusion-matrix accuracy.

In [ ]:
def gfds_url(date):
    d = pd.Timestamp(date)
    return (
        f"{GFDS_BASE}/{GFDS_DATASET}/{GFDS_PRODUCT_FOLDER}/"
        f"{d:%Y}/{d:%m}/"
        f"mag_4days_signal_4days_avg_4days_{d:%Y%m%d}.tif"
    )


def gfds_cache_path(date):
    d = pd.Timestamp(date)
    return GFDS_CACHE / f"{d:%Y%m%d}_avgmag.tif"


def read_gfds_aoi_remote(date):
    """
    First try GDAL /vsicurl/ window access. This avoids downloading the
    ~global raster when the server supports HTTP range requests.
    """
    url = gfds_url(date)
    vsi_url = f"/vsicurl/{url}"

    with rasterio.open(vsi_url) as src:
        aoi_bounds_src = transform_bounds(
            "EPSG:4326",
            src.crs,
            *AOI_BBOX,
            densify_pts=21,
        )

        win = window_from_bounds(
            *aoi_bounds_src,
            transform=src.transform,
        ).round_offsets().round_lengths()

        win = win.intersection(Window(0, 0, src.width, src.height))

        arr = src.read(1, window=win)
        transform = src.window_transform(win)

        return arr, transform, src.crs, url, "REMOTE_WINDOW"


def download_gfds_file(date):
    url = gfds_url(date)
    dst = gfds_cache_path(date)

    if dst.exists() and dst.stat().st_size > 0:
        return dst

    dst.parent.mkdir(parents=True, exist_ok=True)

    with SESSION.get(
        url,
        stream=True,
        timeout=REQUEST_TIMEOUT,
    ) as r:
        r.raise_for_status()

        with open(dst, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    return dst


def read_gfds_aoi(date):
    try:
        return read_gfds_aoi_remote(date)

    except Exception as remote_exc:
        if not GFDS_ALLOW_FULL_DOWNLOAD_FALLBACK:
            raise RuntimeError(
                "GFDS remote-window read failed and full-download fallback "
                "is disabled. Set GFDS_ALLOW_FULL_DOWNLOAD_FALLBACK=True "
                "only if you accept downloading the global GeoTIFF. "
                f"Original error: {remote_exc}"
            )

        path = download_gfds_file(date)

        with rasterio.open(path) as src:
            aoi_bounds_src = transform_bounds(
                "EPSG:4326",
                src.crs,
                *AOI_BBOX,
                densify_pts=21,
            )

            win = window_from_bounds(
                *aoi_bounds_src,
                transform=src.transform,
            ).round_offsets().round_lengths()

            win = win.intersection(Window(0, 0, src.width, src.height))

            arr = src.read(1, window=win)
            transform = src.window_transform(win)

            return (
                arr,
                transform,
                src.crs,
                str(path),
                "DOWNLOADED_WINDOW",
            )


def summarize_gfds(date, event_date):
    raw, transform, crs, source, mode = read_gfds_aoi(date)

    magnitude = raw.astype("float32") / 1000.0

    # Remove implausibly large fill values if any.
    magnitude[~np.isfinite(magnitude)] = np.nan
    magnitude[np.abs(magnitude) > 100] = np.nan

    # AOI polygon mask on local GFDS grid.
    aoi_geom = aoi_4326.to_crs(crs)

    try:
        geom = aoi_geom.geometry.union_all()
    except AttributeError:
        geom = aoi_geom.geometry.unary_union

    aoi_mask = rasterize(
        [(mapping(geom), 1)],
        out_shape=magnitude.shape,
        transform=transform,
        fill=0,
        dtype="uint8",
    ).astype(bool)

    valid = aoi_mask & np.isfinite(magnitude)
    candidate = valid & (magnitude >= GFDS_CANDIDATE_SD)
    strong = valid & (magnitude >= GFDS_STRONG_SD)

    return {
        "date": pd.Timestamp(date),
        "temporal_distance_days":
            temporal_distance_days(event_date, date),
        "candidate_cells": int(candidate.sum()),
        "strong_cells": int(strong.sum()),
        "max_magnitude_sd":
            float(np.nanmax(magnitude[valid])) if valid.any() else np.nan,
        "mean_candidate_magnitude_sd":
            float(np.nanmean(magnitude[candidate]))
            if candidate.any() else np.nan,
        "magnitude": magnitude,
        "valid": valid,
        "candidate": candidate,
        "strong": strong,
        "transform": transform,
        "crs": crs,
        "source": source,
        "mode": mode,
    }


gfds_date_rows = []
gfds_objects = {}

if RUN_GFDS:
    for year, spec in EVENTS.items():
        event_date = pd.Timestamp(spec["date"])
        searched = set()
        found_close_hotspot = False

        print("\nGFDS", year, spec["event_id"])

        for half_window in GFDS_SEARCH_WINDOWS:
            dates = pd.date_range(
                event_date - pd.Timedelta(days=half_window),
                event_date + pd.Timedelta(days=half_window),
                freq="D",
            )

            new_dates = [d for d in dates if d.date() not in searched]

            for d in new_dates:
                searched.add(d.date())

                try:
                    info = summarize_gfds(d, event_date)

                    row = {
                        "year": year,
                        "event_id": spec["event_id"],
                        "event_date": spec["date"],
                        "gfds_date": d.date().isoformat(),
                        "temporal_distance_days":
                            info["temporal_distance_days"],
                        "candidate_cells_ge_2sd":
                            info["candidate_cells"],
                        "strong_cells_ge_3sd":
                            info["strong_cells"],
                        "max_magnitude_sd":
                            info["max_magnitude_sd"],
                        "mean_candidate_magnitude_sd":
                            info["mean_candidate_magnitude_sd"],
                        "access_mode": info["mode"],
                        "status": "OK",
                    }

                    gfds_date_rows.append(row)
                    gfds_objects[(year, d.date().isoformat())] = info

                    if info["strong_cells"] > 0:
                        found_close_hotspot = True

                except Exception as exc:
                    gfds_date_rows.append({
                        "year": year,
                        "event_id": spec["event_id"],
                        "event_date": spec["date"],
                        "gfds_date": d.date().isoformat(),
                        "temporal_distance_days":
                            temporal_distance_days(event_date, d),
                        "status": "ERROR",
                        "error": repr(exc),
                    })

            if found_close_hotspot:
                break


gfds_dates = pd.DataFrame(gfds_date_rows)

if len(gfds_dates):
    # Independent-source ranking: temporal proximity first, then anomaly size.
    gfds_dates["rank_score"] = (
        gfds_dates["temporal_distance_days"].fillna(999) * 1000
        - gfds_dates.get("strong_cells_ge_3sd", 0).fillna(0) * 10
        - gfds_dates.get("candidate_cells_ge_2sd", 0).fillna(0)
        - gfds_dates.get("max_magnitude_sd", 0).fillna(0)
    )

    gfds_dates = gfds_dates.sort_values(
        ["year", "rank_score"],
        ascending=[True, True],
    )

gfds_dates.to_csv(
    OUTPUTS / "03_gfds_candidate_dates.csv",
    index=False,
)

display(gfds_dates.head(100) if len(gfds_dates) else gfds_dates)

In [ ]:

gfds_date_rows = []
gfds_objects = {}

if RUN_GFDS:
    for year, spec in GFDS_EVENTS.items():
        event_date = pd.Timestamp(spec["date"])
        searched = set()
        found_close_hotspot = False

        print("\nGFDS", year, spec["event_id"])

        for half_window in GFDS_SEARCH_WINDOWS:
            dates = pd.date_range(
                event_date - pd.Timedelta(days=half_window),
                event_date + pd.Timedelta(days=half_window),
                freq="D",
            )
            new_dates = [d for d in dates if d.date() not in searched]

            for d in new_dates:
                searched.add(d.date())

                try:
                    info = summarize_gfds(d, event_date)
                    gfds_date_rows.append({
                        "year": year,
                        "event_id": spec["event_id"],
                        "event_date": spec["date"],
                        "gfds_date": d.date().isoformat(),
                        "temporal_distance_days": info["temporal_distance_days"],
                        "candidate_cells_ge_2sd": info["candidate_cells"],
                        "strong_cells_ge_3sd": info["strong_cells"],
                        "max_magnitude_sd": info["max_magnitude_sd"],
                        "mean_candidate_magnitude_sd": info["mean_candidate_magnitude_sd"],
                        "access_mode": info["mode"],
                        "status": "OK",
                    })
                    gfds_objects[(year, d.date().isoformat())] = info
                    if info["strong_cells"] > 0:
                        found_close_hotspot = True

                except Exception as exc:
                    gfds_date_rows.append({
                        "year": year,
                        "event_id": spec["event_id"],
                        "event_date": spec["date"],
                        "gfds_date": d.date().isoformat(),
                        "temporal_distance_days": temporal_distance_days(event_date, d),
                        "status": "ERROR",
                        "error": repr(exc),
                    })

            if found_close_hotspot:
                break

gfds_dates = pd.DataFrame(gfds_date_rows)

if len(gfds_dates):
    gfds_dates["rank_score"] = (
        gfds_dates["temporal_distance_days"].fillna(999) * 1000
        - gfds_dates.get("strong_cells_ge_3sd", 0).fillna(0) * 10
        - gfds_dates.get("candidate_cells_ge_2sd", 0).fillna(0)
        - gfds_dates.get("max_magnitude_sd", 0).fillna(0)
    )
    gfds_dates = gfds_dates.sort_values(
        ["year", "rank_score"],
        ascending=[True, True],
    )

gfds_dates.to_csv(
    OUTPUTS / "03_gfds_candidate_dates.csv",
    index=False,
)

display(gfds_dates.head(100) if len(gfds_dates) else gfds_dates)


### 6A. Compare the independently selected GFDS hotspot with Sentinel-1

In [ ]:
gfds_compare_rows = []

if len(gfds_dates):
    for year, g in gfds_dates.groupby("year"):
        usable = g[
            g["status"].eq("OK")
            & (
                g["candidate_cells_ge_2sd"].fillna(0) > 0
            )
        ].copy()

        if usable.empty:
            gfds_compare_rows.append({
                "year": year,
                "event_id": GFDS_EVENTS[year]["event_id"],
                "status": "NO_GFDS_HOTSPOT",
            })
            continue

        best = usable.sort_values("rank_score").iloc[0]
        key = (year, best["gfds_date"])
        obj = gfds_objects[key]

        sar_frac, sar_valid = aggregate_sar_to_reference_grid(
            year,
            obj["magnitude"].shape,
            obj["transform"],
            obj["crs"],
        )

        ok = obj["valid"] & np.isfinite(sar_frac) & (sar_valid >= 0.80)

        hot = ok & (obj["magnitude"] >= GFDS_CANDIDATE_SD)
        strong = ok & (obj["magnitude"] >= GFDS_STRONG_SD)
        background = ok & (obj["magnitude"] < 1.0)

        if ok.sum() >= 5:
            r, p = spearmanr(
                obj["magnitude"][ok],
                sar_frac[ok],
                nan_policy="omit",
            )
        else:
            r, p = np.nan, np.nan

        row = {
            "year": year,
            "event_id": GFDS_EVENTS[year]["event_id"],
            "selected_gfds_date": best["gfds_date"],
            "temporal_distance_days":
                int(best["temporal_distance_days"]),
            "n_valid_joint_cells": int(ok.sum()),
            "n_hot_cells_ge_2sd": int(hot.sum()),
            "n_strong_cells_ge_3sd": int(strong.sum()),
            "median_sar_fraction_hot_ge_2sd":
                float(np.nanmedian(sar_frac[hot])) if hot.any() else np.nan,
            "median_sar_fraction_strong_ge_3sd":
                float(np.nanmedian(sar_frac[strong])) if strong.any() else np.nan,
            "median_sar_fraction_background_lt_1sd":
                float(np.nanmedian(sar_frac[background]))
                if background.any() else np.nan,
            "spearman_gfds_mag_vs_sar_fraction": r,
            "spearman_p": p,
            "evidence_role":
                "INDEPENDENT_COARSE_MICROWAVE_CORROBORATION_ONLY",
            "status": "GFDS_LOCAL_COMPARISON_COMPLETE",
        }

        gfds_compare_rows.append(row)

gfds_compare_df = pd.DataFrame(gfds_compare_rows)

gfds_compare_df.to_csv(
    OUTPUTS / "04_gfds_sar_local_concordance.csv",
    index=False,
)

display(gfds_compare_df)

# STAGE 10 — Final corrected recurrence, inference, robustness, figures, provenance and freeze

**Source provenance:** `correction_FINAL_ALL_IN_ONE.ipynb, scientific sections only`.

Authoritative final analysis state. Manuscript-editing cells are intentionally excluded.

# correction_FINAL_ALL_IN_ONE

## VRFSZ–GBM final correction / reconciliation notebook

This is the single notebook for the remaining local correction work after the corrected EVENT003 rerun. It is deliberately conservative and reproducible.

It performs, in this order:

1. dependency/environment gate;
2. project/input discovery and corrected EVENT003 hard gate;
3. corrected flood-count / valid-observation / recurrence rebuild from the current four event masks;
4. reconstruction of the **original 1-km sufficient-statistic lattice anchored to the Sentinel-1 raster origin**;
5. restoration of the **original active-block rule**: `VRFSZ_obs + StableHigh_obs > 0`;
6. corrected 80-km/original-origin paired block bootstrap;
7. corrected 40/60/80/100/120 km × four-origin robustness;
8. corrected extended semivariogram;
9. old robustness/Figure S2 staleness audit;
10. corrected Figure 4, Figure 5, and Figure S2 generation;
11. SAR provenance patch;
12. conservative DeltaDTM paired-product wording and native-elevation confounding wording;
13. GFDS-only corroboration wording when the completed local GFDS file exists;
14. manuscript-ready numerical text and optional redlined DOCX patch;
15. citation-style audit without inventing numbered citations;
16. GitHub/Zenodo/Data Availability/Code Availability preparation templates;
17. SHA-256 correction freeze and final science gate.

### Locked methodological correction

The earlier closure notebook used a different projected block origin and kept only blocks containing both comparison classes. That is **not** the original implementation. This notebook restores the original design: 1-km lattice anchored to the 10 m Sentinel-1 raster row/column origin and an active block whenever either VRFSZ or Stable High has at least one valid cell-event observation.

### What this notebook does *not* do

It does not redownload Sentinel-1, DeltaDTM, GloFAS, CYGNSS, optical, gauges, or other external data. It does not invent a GitHub URL, release, DOI, CRediT statement, funding statement, conflict declaration, or final journal AI disclosure. Those are external/author actions after the local scientific freeze.

## 0. Dependency bootstrap — run this cell first

In [ ]:
AUTO_INSTALL_MISSING = True

import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'rasterio': 'rasterio',
    'matplotlib': 'matplotlib',
    'tqdm': 'tqdm',
    'geopandas': 'geopandas',
    'pyproj': 'pyproj',
    'shapely': 'shapely',
    'Pillow': 'PIL',
    'python-docx': 'docx',
}

missing = [
    package for package, module in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]

if missing and AUTO_INSTALL_MISSING:
    print('Installing missing packages:', missing)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-U', *missing
    ])

missing_after = [
    package for package, module in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]

if missing_after:
    raise RuntimeError(
        'Missing dependencies after installation attempt: ' + ', '.join(missing_after)
    )

print('Dependency gate PASS.')

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import windows
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from pyproj import Geod
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm.auto import tqdm

from docx import Document
from docx.enum.text import WD_COLOR_INDEX
from docx.shared import Inches
from docx.oxml import OxmlElement

warnings.filterwarnings('ignore', category=FutureWarning)

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('All imports loaded.')

## 1. User controls

In [ ]:
# ============================================================
# USER CONTROLS
# ============================================================
PROJECT_ROOT_OVERRIDE = None
MANUSCRIPT_PATH_OVERRIDE = None
RUN_MANUSCRIPT_PATCH = True
FIG_DPI = 600

EVENTS = {
    'EVENT001': {'year': 2026, 'date': '2026-07-12'},
    'EVENT002': {'year': 2024, 'date': '2024-08-23'},
    'EVENT003': {'year': 2020, 'date': '2020-07-24'},
    'EVENT004': {'year': 2017, 'date': '2017-08-16'},
}

EXPECTED_EVENT003_FRACTION = 0.205340
EVENT003_TOL = 0.001

THRESHOLDS_M = [0.5, 1.0, 2.0]
PRIMARY_THRESHOLD_M = 1.0
EXPECTED_CORRECTED_RR = {0.5: 1.0031, 1.0: 0.9681, 2.0: 0.9003}
RR_TOL = 0.002

# Diagnostic only. Do not tune seed/method to reproduce this interval.
PRIOR_CORRECTED_PRIMARY_CI = (0.8553, 1.0605)
PRIOR_CI_DIAGNOSTIC_TOL = 0.04

# Original block/bootstrap design.
ORIGINAL_BLOCK_KM = 80
PRIMARY_BOOTSTRAP_REPLICATES = 2000
BLOCK_WIDTHS_KM = [40, 60, 80, 100, 120]
OFFSET_FRACTIONS = [
    (0.0, 0.0),
    (0.5, 0.0),
    (0.0, 0.5),
    (0.5, 0.5),
]
ROBUSTNESS_REPLICATES = 5000
MIN_ACTIVE_BLOCKS_FOR_INFERENCE = 20
RANDOM_SEED = 20260830

COARSE_STATS_CELL_M = 1000
ANALYSIS_WINDOW_PIXELS = 1024

# Corrected recurrence semivariogram.
VARIOGRAM_CELL_M = 1000
MAX_VARIogram_HARD_KM = 200
MAX_LAG_FRACTION_SHORT_DIM = 0.45
SILL_FRACTION = 0.95
STABILITY_LAGS = 3
MIN_PAIR_FRACTION_OF_LAG1 = 0.05

# External/publication metadata: leave blank until author fills them.
SUBMISSION_METADATA = {
    'github_repository_url': '',
    'github_release_tag': '',
    'zenodo_doi': '',
    'license': '',
    'funding_statement': '',
    'competing_interests': '',
    'credit_statement': '',
    'ai_disclosure': '',
}

print('Controls loaded.')

## 2. Resolve project root and paths

In [ ]:
AOI_REL = 'data/raw/boundaries/gbm_delta.shp'
TERRAIN_REL = 'data/processed/terrain'
EVENT_ROOT_REL = 'data/processed/flood_events'
FROZEN_SAR_CONFIG_REL = 'config/frozen_sar_workflow.json'


def resolve_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        p = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (
            (p / AOI_REL).exists()
            and (p / TERRAIN_REL / 'vrfsz_classes_T1m.tif').exists()
            and (p / EVENT_ROOT_REL / 'EVENT001' / 'flood_mask.tif').exists()
        ):
            return p

    raise RuntimeError(
        'Could not locate project root. Set PROJECT_ROOT_OVERRIDE in Section 1.'
    )


PROJECT_ROOT = resolve_project_root()
AOI_PATH = PROJECT_ROOT / AOI_REL
TERRAIN_DIR = PROJECT_ROOT / TERRAIN_REL
EVENT_ROOT = PROJECT_ROOT / EVENT_ROOT_REL
FROZEN_SAR_CONFIG = PROJECT_ROOT / FROZEN_SAR_CONFIG_REL

OUT_ROOT = PROJECT_ROOT / 'reports' / 'correction_final'
DATA_OUT = OUT_ROOT / 'data'
FIG_OUT = OUT_ROOT / 'figures'
TEXT_OUT = OUT_ROOT / 'manuscript_ready_text'
PROV_OUT = OUT_ROOT / 'provenance'
SUBMISSION_OUT = OUT_ROOT / 'submission_package'
DOCX_OUT = OUT_ROOT / 'manuscript'
FREEZE_OUT = OUT_ROOT / 'final_freeze'

for p in [DATA_OUT, FIG_OUT, TEXT_OUT, PROV_OUT, SUBMISSION_OUT, DOCX_OUT, FREEZE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

for event_id, spec in EVENTS.items():
    d = EVENT_ROOT / event_id
    spec['flood_mask'] = d / 'flood_mask.tif'
    spec['valid_mask'] = d / 'valid_mask.tif'
    spec['metadata_json'] = d / 'event_flood_metadata.json'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUT_ROOT:', OUT_ROOT)

## 3. Input audit and corrected EVENT003 hard gate

In [ ]:
mandatory = {
    'AOI': AOI_PATH,
    'T1_class': TERRAIN_DIR / 'vrfsz_classes_T1m.tif',
    'frozen_sar_config': FROZEN_SAR_CONFIG,
}

for event_id, spec in EVENTS.items():
    mandatory[f'{event_id}_flood'] = spec['flood_mask']
    mandatory[f'{event_id}_valid'] = spec['valid_mask']

audit = pd.DataFrame([
    {
        'asset': name,
        'path': str(path),
        'exists': path.exists(),
        'size_MB': path.stat().st_size / 1024**2 if path.exists() else np.nan,
    }
    for name, path in mandatory.items()
])
display(audit)
audit.to_csv(DATA_OUT / '00_input_audit.csv', index=False)

if not audit['exists'].all():
    missing = audit.loc[~audit['exists'], 'path'].tolist()
    raise FileNotFoundError('Missing mandatory inputs:\n' + '\n'.join(missing))


def event_fraction(flood_path, valid_path):
    nf = 0
    nv = 0
    with rasterio.open(flood_path) as fs, rasterio.open(valid_path) as vs:
        if (
            fs.crs != vs.crs
            or fs.transform != vs.transform
            or fs.width != vs.width
            or fs.height != vs.height
        ):
            raise RuntimeError(f'Flood/valid grid mismatch: {flood_path.parent.name}')
        for _, w in fs.block_windows(1):
            flood = fs.read(1, window=w)
            valid = vs.read(1, window=w) == 1
            nv += int(valid.sum())
            nf += int(((flood == 1) & valid).sum())
    return nf, nv, nf / nv if nv else np.nan


event_state_rows = []
for event_id, spec in EVENTS.items():
    nf, nv, frac = event_fraction(spec['flood_mask'], spec['valid_mask'])
    event_state_rows.append({
        'event_id': event_id,
        'year': spec['year'],
        'event_date': spec['date'],
        'flood_pixels': nf,
        'valid_pixels': nv,
        'flood_fraction': frac,
    })

event_state = pd.DataFrame(event_state_rows).sort_values('event_id')
display(event_state)
event_state.to_csv(DATA_OUT / '01_current_event_state.csv', index=False)

e3 = float(event_state.loc[event_state['event_id'].eq('EVENT003'), 'flood_fraction'].iloc[0])
if abs(e3 - EXPECTED_EVENT003_FRACTION) > EVENT003_TOL:
    raise RuntimeError(
        'STOP — EVENT003 is not the corrected mask. '
        f'Expected ~{EXPECTED_EVENT003_FRACTION:.6f}; found {e3:.6f}.'
    )

print('PASS corrected EVENT003:', f'{e3:.6f}')

## 4. Resolve terrain-class rasters; regenerate only if missing

In [ ]:
NATIVE = TERRAIN_DIR / 'DeltaDTM_v1_1_native_GBM.tif'
MSL = TERRAIN_DIR / 'DeltaDTM_v1_1_MSL_GBM.tif'
COMMON_VALID = TERRAIN_DIR / 'terrain_common_valid_mask.tif'

CLASS_FILES = {
    0.5: TERRAIN_DIR / 'vrfsz_classes_T0p5m.tif',
    1.0: TERRAIN_DIR / 'vrfsz_classes_T1m.tif',
    2.0: TERRAIN_DIR / 'vrfsz_classes_T2m.tif',
}

CLOSURE29_CLASS_DIR = PROJECT_ROOT / 'reports' / 'reviewer_closure_29' / 'derived_terrain_classes'


def closure29_candidate(T):
    tag = str(T).replace('.', 'p')
    return CLOSURE29_CLASS_DIR / f'vrfsz_classes_T{tag}m_closure29.tif'


for T in THRESHOLDS_M:
    if CLASS_FILES[T].exists():
        continue

    c29 = closure29_candidate(T)
    if c29.exists():
        CLASS_FILES[T] = c29
        continue

    if not (NATIVE.exists() and MSL.exists() and COMMON_VALID.exists()):
        raise FileNotFoundError(
            f'Class raster missing for T={T} and terrain regeneration inputs unavailable.'
        )

    target = DATA_OUT / f"vrfsz_classes_T{str(T).replace('.', 'p')}m_corrected.tif"

    with rasterio.open(NATIVE) as ns, rasterio.open(MSL) as ms, rasterio.open(COMMON_VALID) as cs:
        profile = ns.profile.copy()
        profile.update(dtype='uint8', count=1, nodata=0, compress='deflate', tiled=True)
        with rasterio.open(target, 'w', **profile) as dst:
            for _, w in tqdm(list(ns.block_windows(1)), desc=f'Regenerate class T={T}'):
                zn = ns.read(1, window=w)
                zm = ms.read(1, window=w)
                common = cs.read(1, window=w) == 1
                finite = common & np.isfinite(zn) & np.isfinite(zm)
                c = np.zeros(zn.shape, dtype=np.uint8)
                c[finite & (zn <= T) & (zm <= T)] = 1
                c[finite & (zn > T) & (zm <= T)] = 2
                c[finite & (zn > T) & (zm > T)] = 3
                c[finite & (zn <= T) & (zm > T)] = 4
                dst.write(c, 1, window=w)

    CLASS_FILES[T] = target

for T, p in CLASS_FILES.items():
    print(T, '->', p)

## 5. Rebuild corrected flood-count / observation-count / recurrence

These are rebuilt directly from the **current four event masks**, so no older recurrence raster is trusted merely because it exists.

In [ ]:
CORR_FLOOD_COUNT = DATA_OUT / 'flood_count_corrected.tif'
CORR_OBS_COUNT = DATA_OUT / 'observation_count_corrected.tif'
CORR_RECURRENCE = DATA_OUT / 'flood_recurrence_corrected.tif'

event_ids = list(EVENTS)
with rasterio.open(EVENTS[event_ids[0]]['flood_mask']) as ref:
    ref_profile = ref.profile.copy()

for eid in event_ids[1:]:
    with rasterio.open(EVENTS[eid]['flood_mask']) as src:
        if (
            src.crs != ref_profile['crs']
            or src.transform != ref_profile['transform']
            or src.width != ref_profile['width']
            or src.height != ref_profile['height']
        ):
            raise RuntimeError('All corrected event masks must share the same 10 m grid.')

count_profile = ref_profile.copy()
count_profile.update(dtype='uint8', count=1, nodata=0, compress='deflate', tiled=True)
rec_profile = ref_profile.copy()
rec_profile.update(dtype='float32', count=1, nodata=-9999.0, compress='deflate', tiled=True)

flood_srcs = [rasterio.open(EVENTS[eid]['flood_mask']) for eid in event_ids]
valid_srcs = [rasterio.open(EVENTS[eid]['valid_mask']) for eid in event_ids]

try:
    with rasterio.open(CORR_FLOOD_COUNT, 'w', **count_profile) as f_dst,          rasterio.open(CORR_OBS_COUNT, 'w', **count_profile) as o_dst,          rasterio.open(CORR_RECURRENCE, 'w', **rec_profile) as r_dst:

        for _, w in tqdm(list(flood_srcs[0].block_windows(1)), desc='Rebuild corrected recurrence'):
            shape = (int(w.height), int(w.width))
            fcount = np.zeros(shape, dtype=np.uint8)
            ocount = np.zeros(shape, dtype=np.uint8)

            for fs, vs in zip(flood_srcs, valid_srcs):
                f = fs.read(1, window=w)
                v = vs.read(1, window=w) == 1
                ocount += v.astype(np.uint8)
                fcount += ((f == 1) & v).astype(np.uint8)

            rec = np.full(shape, -9999.0, dtype=np.float32)
            ok = ocount > 0
            rec[ok] = fcount[ok].astype(np.float32) / ocount[ok].astype(np.float32)

            f_dst.write(fcount, 1, window=w)
            o_dst.write(ocount, 1, window=w)
            r_dst.write(rec, 1, window=w)
finally:
    for src in flood_srcs + valid_srcs:
        src.close()

print('Corrected recurrence products written.')

# 6. Restore the original 1-km sufficient-statistic lattice

The 1-km grid is anchored to the row/column origin of the 10 m Sentinel-1 analysis grid. It is a lossless aggregation device for the pooled RR because only exact flood-detection and valid-observation sums are carried forward.

In [ ]:
with rasterio.open(CORR_OBS_COUNT) as ref:
    TARGET_CRS = ref.crs
    TARGET_TRANSFORM = ref.transform
    TARGET_WIDTH = ref.width
    TARGET_HEIGHT = ref.height
    XRES_M = abs(float(ref.transform.a))
    YRES_M = abs(float(ref.transform.e))

coarse_px_x = max(1, int(round(COARSE_STATS_CELL_M / XRES_M)))
coarse_px_y = max(1, int(round(COARSE_STATS_CELL_M / YRES_M)))
COARSE_COLS = math.ceil(TARGET_WIDTH / coarse_px_x)
COARSE_ROWS = math.ceil(TARGET_HEIGHT / coarse_px_y)
COARSE_N = COARSE_ROWS * COARSE_COLS

print('1-km sufficient-statistic grid:', COARSE_COLS, 'x', COARSE_ROWS, '=', COARSE_N)

CLASS_CODES = {'VRFSZ': 2, 'StableHigh': 3}
coarse_flood = {
    T: {name: np.zeros(COARSE_N, dtype=np.float64) for name in CLASS_CODES}
    for T in THRESHOLDS_M
}
coarse_obs = {
    T: {name: np.zeros(COARSE_N, dtype=np.float64) for name in CLASS_CODES}
    for T in THRESHOLDS_M
}


def iter_windows(width, height, size):
    for row_off in range(0, height, size):
        h = min(size, height - row_off)
        for col_off in range(0, width, size):
            w = min(size, width - col_off)
            yield windows.Window(col_off=col_off, row_off=row_off, width=w, height=h)

analysis_windows = list(iter_windows(TARGET_WIDTH, TARGET_HEIGHT, ANALYSIS_WINDOW_PIXELS))

with ExitStack() as stack:
    fsrc = stack.enter_context(rasterio.open(CORR_FLOOD_COUNT))
    osrc = stack.enter_context(rasterio.open(CORR_OBS_COUNT))
    class_vrts = {}

    for T in THRESHOLDS_M:
        csrc = stack.enter_context(rasterio.open(CLASS_FILES[T]))
        class_vrts[T] = stack.enter_context(WarpedVRT(
            csrc,
            crs=TARGET_CRS,
            transform=TARGET_TRANSFORM,
            width=TARGET_WIDTH,
            height=TARGET_HEIGHT,
            resampling=Resampling.nearest,
            src_nodata=0,
            nodata=0,
            dtype='uint8',
        ))

    for win in tqdm(analysis_windows, desc='Build original 1-km sufficient statistics', unit='window'):
        flood = fsrc.read(1, window=win).astype(np.float64)
        obs = osrc.read(1, window=win).astype(np.float64)

        if np.any((flood < 0) | (obs < 0) | (flood > obs)):
            raise RuntimeError('Invalid corrected flood/observation counts.')

        row0 = int(win.row_off)
        col0 = int(win.col_off)
        h = int(win.height)
        w = int(win.width)

        coarse_rows = (np.arange(row0, row0 + h, dtype=np.int32) // coarse_px_y)[:, None]
        coarse_cols = (np.arange(col0, col0 + w, dtype=np.int32) // coarse_px_x)[None, :]
        coarse_ids = (coarse_rows * COARSE_COLS + coarse_cols).astype(np.int32, copy=False)
        observed = obs > 0

        for T in THRESHOLDS_M:
            cls = class_vrts[T].read(1, window=win).astype(np.uint8)
            for name, code_value in CLASS_CODES.items():
                mask = observed & (cls == code_value)
                if not mask.any():
                    continue
                ids = coarse_ids[mask]
                coarse_flood[T][name] += np.bincount(
                    ids, weights=flood[mask], minlength=COARSE_N
                )
                coarse_obs[T][name] += np.bincount(
                    ids, weights=obs[mask], minlength=COARSE_N
                )

print('PASS: original 1-km sufficient-statistic lattice rebuilt.')

## 7. Lossless corrected RR reconstruction gate

In [ ]:
rr_rows = []
for T in THRESHOLDS_M:
    fv = float(coarse_flood[T]['VRFSZ'].sum())
    ov = float(coarse_obs[T]['VRFSZ'].sum())
    fh = float(coarse_flood[T]['StableHigh'].sum())
    oh = float(coarse_obs[T]['StableHigh'].sum())
    rr = (fv / ov) / (fh / oh)
    expected = EXPECTED_CORRECTED_RR[T]

    rr_rows.append({
        'threshold_m': T,
        'VRFSZ_flood': fv,
        'VRFSZ_obs': ov,
        'StableHigh_flood': fh,
        'StableHigh_obs': oh,
        'relative_risk': rr,
        'expected_corrected_RR': expected,
        'absolute_difference': abs(rr - expected),
    })

rr_table = pd.DataFrame(rr_rows)
display(rr_table)
rr_table.to_csv(DATA_OUT / '02_corrected_pooled_RR.csv', index=False)

for _, row in rr_table.iterrows():
    if row['absolute_difference'] > RR_TOL:
        raise RuntimeError(
            f"Corrected RR reconstruction failed at T={row['threshold_m']}: "
            f"{row['relative_risk']:.6f} vs expected {row['expected_corrected_RR']:.6f}."
        )

print('PASS: corrected RR values reproduced losslessly.')

# 8. Restore original block geometry and active-block rule

Original rule: a block is active when `(VRFSZ_obs + StableHigh_obs) > 0`. It does **not** require both classes to be present.

Expected original structural counts are 26 active blocks at every threshold, with 21/22/23 blocks containing both classes at T=0.5/1/2 m. Because EVENT003 correction changes flood detection rather than terrain/valid-observation geometry, these counts should remain unchanged.

In [ ]:
coarse_row_index = np.arange(COARSE_N, dtype=np.int32) // COARSE_COLS
coarse_col_index = np.arange(COARSE_N, dtype=np.int32) % COARSE_COLS


def block_arrays_for_setting(threshold, block_km, offset_x_fraction, offset_y_fraction):
    cells_per_block = max(1, int(round(block_km * 1000 / COARSE_STATS_CELL_M)))
    shift_x = int(round(cells_per_block * offset_x_fraction))
    shift_y = int(round(cells_per_block * offset_y_fraction))

    block_col = np.floor_divide(coarse_col_index + shift_x, cells_per_block)
    block_row = np.floor_divide(coarse_row_index + shift_y, cells_per_block)

    n_cols = int(block_col.max() + 1)
    block_id = (block_row * n_cols + block_col).astype(np.int32)
    n_blocks_possible = int(block_id.max() + 1)

    output = {}
    for name in CLASS_CODES:
        output[f'{name}_flood'] = np.bincount(
            block_id,
            weights=coarse_flood[threshold][name],
            minlength=n_blocks_possible,
        )
        output[f'{name}_obs'] = np.bincount(
            block_id,
            weights=coarse_obs[threshold][name],
            minlength=n_blocks_possible,
        )

    active = (output['VRFSZ_obs'] + output['StableHigh_obs']) > 0
    output['raw_block_id'] = np.flatnonzero(active)

    for key in ['VRFSZ_flood', 'VRFSZ_obs', 'StableHigh_flood', 'StableHigh_obs']:
        output[key] = output[key][active]

    return output


EXPECTED_STRUCTURE = {
    0.5: {'active': 26, 'both': 21},
    1.0: {'active': 26, 'both': 22},
    2.0: {'active': 26, 'both': 23},
}

structure_rows = []
original_setting_arrays = {}

for T in THRESHOLDS_M:
    arr = block_arrays_for_setting(T, ORIGINAL_BLOCK_KM, 0.0, 0.0)
    original_setting_arrays[T] = arr
    active = len(arr['VRFSZ_obs'])
    both = int(((arr['VRFSZ_obs'] > 0) & (arr['StableHigh_obs'] > 0)).sum())

    structure_rows.append({
        'threshold_m': T,
        'active_blocks': active,
        'blocks_with_both_classes': both,
        'expected_active_blocks': EXPECTED_STRUCTURE[T]['active'],
        'expected_both_blocks': EXPECTED_STRUCTURE[T]['both'],
        'active_match': active == EXPECTED_STRUCTURE[T]['active'],
        'both_match': both == EXPECTED_STRUCTURE[T]['both'],
    })

structure_df = pd.DataFrame(structure_rows)
display(structure_df)
structure_df.to_csv(DATA_OUT / '03_original_block_structure_reproduction.csv', index=False)

if not structure_df['active_match'].all():
    raise RuntimeError('STOP — original 80-km active-block structure was not reproduced.')
if not structure_df['both_match'].all():
    raise RuntimeError('STOP — blocks-with-both-classes structure changed unexpectedly.')

print('PASS: original 80-km/original-origin block structure reproduced.')

## 9. Corrected primary 80-km / original-origin paired bootstrap

In [ ]:
def bootstrap_rr(arrays, replicates, seed):
    fv = arrays['VRFSZ_flood'].astype(np.float64)
    ov = arrays['VRFSZ_obs'].astype(np.float64)
    fh = arrays['StableHigh_flood'].astype(np.float64)
    oh = arrays['StableHigh_obs'].astype(np.float64)

    n_blocks = len(ov)
    point_rr = (fv.sum() / ov.sum()) / (fh.sum() / oh.sum())
    rng = np.random.default_rng(seed)
    rr = np.full(replicates, np.nan, dtype=np.float64)

    for b in range(replicates):
        idx = rng.integers(0, n_blocks, size=n_blocks)
        FV = float(fv[idx].sum())
        OV = float(ov[idx].sum())
        FH = float(fh[idx].sum())
        OH = float(oh[idx].sum())

        if OV <= 0 or OH <= 0 or FH <= 0:
            continue

        risk_v = FV / OV
        risk_h = FH / OH
        if risk_h > 0:
            rr[b] = risk_v / risk_h

    rr = rr[np.isfinite(rr)]
    if rr.size < 0.95 * replicates:
        raise RuntimeError(f'Too few valid bootstrap replicates: {rr.size}/{replicates}')

    return {
        'point_rr': point_rr,
        'n_blocks': n_blocks,
        'valid_replicates': int(rr.size),
        'bootstrap_mean': float(rr.mean()),
        'bootstrap_median': float(np.median(rr)),
        'bootstrap_se': float(rr.std(ddof=1)),
        'ci_low': float(np.quantile(rr, 0.025)),
        'ci_high': float(np.quantile(rr, 0.975)),
        'prob_rr_gt_1': float(np.mean(rr > 1)),
        'tail_rr_le_1': float(np.mean(rr <= 1)),
    }


primary_rows = []
for T in THRESHOLDS_M:
    # Use the exact 19B setting seed formula for the original 80-km/origin setting.
    # We do not tune the seed to reproduce a previously reported CI.
    seed = RANDOM_SEED + int(T * 100) + ORIGINAL_BLOCK_KM * 1000 + 0
    result = bootstrap_rr(
        original_setting_arrays[T],
        replicates=PRIMARY_BOOTSTRAP_REPLICATES,
        seed=seed,
    )
    primary_rows.append({
        'threshold_m': T,
        'seed': seed,
        'bootstrap_replicates': PRIMARY_BOOTSTRAP_REPLICATES,
        **result,
    })

primary_bootstrap = pd.DataFrame(primary_rows)
display(primary_bootstrap)
primary_bootstrap.to_csv(DATA_OUT / '04_corrected_primary_80km_bootstrap.csv', index=False)

primary_t1 = primary_bootstrap.loc[primary_bootstrap['threshold_m'].eq(1.0)].iloc[0]
prior_diff = max(
    abs(float(primary_t1['ci_low']) - PRIOR_CORRECTED_PRIMARY_CI[0]),
    abs(float(primary_t1['ci_high']) - PRIOR_CORRECTED_PRIMARY_CI[1]),
)

print(
    'Primary T=1 corrected result:',
    f"RR={primary_t1['point_rr']:.6f}, 95% CI={primary_t1['ci_low']:.6f}–{primary_t1['ci_high']:.6f}, "
    f"active blocks={int(primary_t1['n_blocks'])}"
)

if prior_diff > PRIOR_CI_DIAGNOSTIC_TOL:
    print(
        'WARNING: this auditable original-method CI differs from the previously recorded corrected CI. '
        'Do not mix the intervals; use the method/seed recorded in this notebook for the final freeze.'
    )
else:
    print('Diagnostic PASS: interval is consistent with the previously recorded corrected range.')

# 10. Corrected 40–120 km × four-origin robustness

This reproduces the original 19B robustness design exactly: 40/60/80/100/120 km, four fixed origins, 5000 paired block-bootstrap replicates per setting, minimum 20 active blocks for inferential interpretation, and the original deterministic setting seed rule.

In [ ]:
robust_rows = []
total_runs = len(THRESHOLDS_M) * len(BLOCK_WIDTHS_KM) * len(OFFSET_FRACTIONS)
bar = tqdm(total=total_runs, desc='Corrected block robustness', unit='setting')

try:
    for T in THRESHOLDS_M:
        corrected_rr = float(rr_table.loc[rr_table['threshold_m'].eq(T), 'relative_risk'].iloc[0])

        for block_km in BLOCK_WIDTHS_KM:
            for offset_index, (off_x, off_y) in enumerate(OFFSET_FRACTIONS):
                arr = block_arrays_for_setting(T, block_km, off_x, off_y)
                seed = RANDOM_SEED + int(T * 100) + block_km * 1000 + offset_index
                result = bootstrap_rr(arr, ROBUSTNESS_REPLICATES, seed)

                usable = (
                    result['n_blocks'] >= MIN_ACTIVE_BLOCKS_FOR_INFERENCE
                    and np.isfinite(result['ci_low'])
                    and np.isfinite(result['ci_high'])
                )

                if usable:
                    relation = (
                        'above_1' if result['ci_low'] > 1
                        else 'below_1' if result['ci_high'] < 1
                        else 'includes_1'
                    )
                else:
                    relation = 'insufficient_blocks'

                robust_rows.append({
                    'threshold_m': T,
                    'block_km': block_km,
                    'offset_index': offset_index,
                    'offset_x_fraction': off_x,
                    'offset_y_fraction': off_y,
                    'is_original_setting': (
                        block_km == ORIGINAL_BLOCK_KM and off_x == 0.0 and off_y == 0.0
                    ),
                    'active_blocks': result['n_blocks'],
                    'inferentially_usable': usable,
                    'valid_bootstrap_replicates': result['valid_replicates'],
                    'corrected_point_RR': corrected_rr,
                    'reconstructed_point_RR': result['point_rr'],
                    'CI_low': result['ci_low'],
                    'CI_high': result['ci_high'],
                    'bootstrap_mean_RR': result['bootstrap_mean'],
                    'bootstrap_probability_RR_gt_1': result['prob_rr_gt_1'],
                    'CI_relation_to_1': relation,
                    'seed': seed,
                })
                bar.update(1)
finally:
    bar.close()

robustness_df = pd.DataFrame(robust_rows)
display(robustness_df)
robustness_df.to_csv(DATA_OUT / '05_corrected_block_size_and_origin_robustness.csv', index=False)

In [ ]:
summary_rows = []
for T in THRESHOLDS_M:
    for block_km in BLOCK_WIDTHS_KM:
        sub_all = robustness_df.loc[
            robustness_df['threshold_m'].eq(T)
            & robustness_df['block_km'].eq(block_km)
        ]
        sub = sub_all.loc[sub_all['inferentially_usable']]

        if sub.empty:
            summary_rows.append({
                'threshold_m': T,
                'block_km': block_km,
                'usable_origins': 0,
                'min_active_blocks': int(sub_all['active_blocks'].min()),
                'max_active_blocks': int(sub_all['active_blocks'].max()),
                'CI_low_min': np.nan,
                'CI_low_max': np.nan,
                'CI_high_min': np.nan,
                'CI_high_max': np.nan,
                'all_usable_CIs_include_1': np.nan,
            })
            continue

        summary_rows.append({
            'threshold_m': T,
            'block_km': block_km,
            'usable_origins': len(sub),
            'min_active_blocks': int(sub['active_blocks'].min()),
            'max_active_blocks': int(sub['active_blocks'].max()),
            'CI_low_min': float(sub['CI_low'].min()),
            'CI_low_max': float(sub['CI_low'].max()),
            'CI_high_min': float(sub['CI_high'].min()),
            'CI_high_max': float(sub['CI_high'].max()),
            'all_usable_CIs_include_1': bool((sub['CI_relation_to_1'] == 'includes_1').all()),
        })

robust_summary = pd.DataFrame(summary_rows)
display(robust_summary)
robust_summary.to_csv(DATA_OUT / '06_corrected_block_width_robustness_summary.csv', index=False)

## 11. Corrected extended semivariogram

In [ ]:
with rasterio.open(CORR_RECURRENCE) as src:
    xres_m = abs(float(src.transform.a))
    yres_m = abs(float(src.transform.e))
    width_m = src.width * xres_m
    height_m = src.height * yres_m
    short_dim_m = min(width_m, height_m)

    factor_x = max(1, int(round(VARIOGRAM_CELL_M / xres_m)))
    factor_y = max(1, int(round(VARIOGRAM_CELL_M / yres_m)))
    coarse_width = math.ceil(src.width / factor_x)
    coarse_height = math.ceil(src.height / factor_y)

    coarse_rec = src.read(
        1,
        out_shape=(coarse_height, coarse_width),
        masked=True,
        resampling=Resampling.average,
    ).filled(np.nan)

    coarse_cell_x_m = width_m / coarse_width
    coarse_cell_y_m = height_m / coarse_height
    variogram_cell_m = float(np.mean([coarse_cell_x_m, coarse_cell_y_m]))

geometry_max_lag_km = MAX_LAG_FRACTION_SHORT_DIM * short_dim_m / 1000
max_lag_km = min(MAX_VARIogram_HARD_KM, geometry_max_lag_km)
max_lag_cells = min(
    int(math.floor(max_lag_km * 1000 / variogram_cell_m)),
    coarse_rec.shape[0] - 1,
    coarse_rec.shape[1] - 1,
)

sill = float(np.nanvar(coarse_rec))
if not np.isfinite(sill) or sill <= 0:
    raise RuntimeError('Corrected recurrence surface has no usable variance.')


def directional_semivariance(arr, lag):
    sums = 0.0
    n_pairs = 0
    counts = {}

    a, b = arr[:, :-lag], arr[:, lag:]
    valid = np.isfinite(a) & np.isfinite(b)
    n = int(valid.sum())
    if n:
        d = a[valid] - b[valid]
        sums += float(np.square(d).sum())
        n_pairs += n
    counts['EW'] = n

    a, b = arr[:-lag, :], arr[lag:, :]
    valid = np.isfinite(a) & np.isfinite(b)
    n = int(valid.sum())
    if n:
        d = a[valid] - b[valid]
        sums += float(np.square(d).sum())
        n_pairs += n
    counts['NS'] = n

    a, b = arr[:-lag, :-lag], arr[lag:, lag:]
    valid = np.isfinite(a) & np.isfinite(b)
    n = int(valid.sum())
    if n:
        d = a[valid] - b[valid]
        sums += float(np.square(d).sum())
        n_pairs += n
    counts['NW_SE'] = n

    a, b = arr[:-lag, lag:], arr[lag:, :-lag]
    valid = np.isfinite(a) & np.isfinite(b)
    n = int(valid.sum())
    if n:
        d = a[valid] - b[valid]
        sums += float(np.square(d).sum())
        n_pairs += n
    counts['NE_SW'] = n

    gamma = 0.5 * sums / n_pairs if n_pairs > 0 else np.nan
    return gamma, n_pairs, counts


vario_rows = []
for lag in tqdm(range(1, max_lag_cells + 1), desc='Corrected extended semivariogram'):
    gamma, n_pairs, counts = directional_semivariance(coarse_rec, lag)
    vario_rows.append({
        'lag_cells': lag,
        'distance_km': lag * variogram_cell_m / 1000,
        'semivariance': gamma,
        'sill_variance': sill,
        'fraction_of_sill': gamma / sill if np.isfinite(gamma) else np.nan,
        'pairs_total': n_pairs,
        'pairs_EW': counts['EW'],
        'pairs_NS': counts['NS'],
        'pairs_NW_SE': counts['NW_SE'],
        'pairs_NE_SW': counts['NE_SW'],
    })

extended_variogram = pd.DataFrame(vario_rows)
lag1_pairs = int(extended_variogram.loc[0, 'pairs_total'])
pair_threshold = int(max(1, lag1_pairs * MIN_PAIR_FRACTION_OF_LAG1))
extended_variogram['pair_support_ok'] = extended_variogram['pairs_total'] >= pair_threshold

threshold_gamma = SILL_FRACTION * sill
supported = extended_variogram.loc[extended_variogram['pair_support_ok']].reset_index(drop=True)
gamma_values = supported['semivariance'].to_numpy(float)
dist_values = supported['distance_km'].to_numpy(float)
empirical_range_km = None

for i in range(0, max(0, len(gamma_values) - STABILITY_LAGS + 1)):
    run = gamma_values[i:i + STABILITY_LAGS]
    if np.isfinite(run).all() and np.all(run >= threshold_gamma):
        empirical_range_km = float(dist_values[i])
        break

range_resolved = empirical_range_km is not None
extended_variogram.to_csv(DATA_OUT / '07_corrected_extended_empirical_variogram.csv', index=False)

range_diag = {
    'sill_variance': sill,
    'sill_fraction': SILL_FRACTION,
    'stability_lags': STABILITY_LAGS,
    'tested_max_lag_km': float(extended_variogram['distance_km'].max()),
    'minimum_pair_support': pair_threshold,
    'empirical_range_resolved': range_resolved,
    'empirical_95pct_sill_range_km': empirical_range_km,
}
(DATA_OUT / '08_corrected_range_diagnostic.json').write_text(
    json.dumps(range_diag, indent=2), encoding='utf-8'
)
print(json.dumps(range_diag, indent=2))

## 12. Event-level descriptive RR table

In [ ]:
event_rows = []

for T in THRESHOLDS_M:
    class_path = CLASS_FILES[T]
    with rasterio.open(class_path) as csrc:
        for eid, spec in EVENTS.items():
            vv = vf = sv = sf = 0

            with rasterio.open(spec['flood_mask']) as fs,                  rasterio.open(spec['valid_mask']) as vs,                  WarpedVRT(
                     csrc,
                     crs=fs.crs,
                     transform=fs.transform,
                     width=fs.width,
                     height=fs.height,
                     resampling=Resampling.nearest,
                     src_nodata=0,
                     nodata=0,
                 ) as cv:

                for _, w in fs.block_windows(1):
                    flood = fs.read(1, window=w)
                    valid = vs.read(1, window=w) == 1
                    cls = cv.read(1, window=w)
                    m_v = valid & (cls == 2)
                    m_s = valid & (cls == 3)
                    vv += int(m_v.sum())
                    vf += int(((flood == 1) & m_v).sum())
                    sv += int(m_s.sum())
                    sf += int(((flood == 1) & m_s).sum())

            pv = vf / vv if vv else np.nan
            ps = sf / sv if sv else np.nan
            rr = pv / ps if ps > 0 else np.nan

            event_rows.append({
                'threshold_m': T,
                'event_id': eid,
                'year': spec['year'],
                'event_date': spec['date'],
                'VRFSZ_obs': vv,
                'VRFSZ_flood': vf,
                'StableHigh_obs': sv,
                'StableHigh_flood': sf,
                'p_VRFSZ': pv,
                'p_StableHigh': ps,
                'event_RR_descriptive': rr,
            })

event_rr = pd.DataFrame(event_rows)
display(event_rr)
event_rr.to_csv(DATA_OUT / '09_corrected_event_level_RR.csv', index=False)

## 13. Old robustness / Figure S2 staleness audit

In [ ]:
old_robust_candidates = list((PROJECT_ROOT / 'reports').rglob('block_size_and_origin_robustness.csv'))
stale_rows = []

for p in old_robust_candidates:
    if DATA_OUT in p.parents:
        continue
    try:
        old = pd.read_csv(p)
        if not {'threshold_m', 'reconstructed_point_RR'}.issubset(old.columns):
            continue
        old_t1 = old.loc[old['threshold_m'].eq(1.0)]
        if old_t1.empty:
            continue
        old_rr = float(old_t1['reconstructed_point_RR'].dropna().iloc[0])
        new_rr = float(rr_table.loc[rr_table['threshold_m'].eq(1.0), 'relative_risk'].iloc[0])
        stale = abs(old_rr - new_rr) > 1e-6
        stale_rows.append({
            'path': str(p),
            'old_T1_RR': old_rr,
            'current_T1_RR': new_rr,
            'stale_for_corrected_EVENT003': stale,
        })
    except Exception as exc:
        stale_rows.append({
            'path': str(p),
            'error': repr(exc),
            'stale_for_corrected_EVENT003': True,
        })

stale_audit = pd.DataFrame(stale_rows)
display(stale_audit)
stale_audit.to_csv(DATA_OUT / '10_old_robustness_staleness_audit.csv', index=False)

if len(stale_audit) and stale_audit['stale_for_corrected_EVENT003'].fillna(True).any():
    print('DECISION: old robustness/Figure S2 support is stale for corrected EVENT003. Use the new Figure S2.')
else:
    print('No stale old robustness table detected; still use the new corrected outputs for one final freeze.')

# 14. Regenerate Figure 4 — corrected event masks, recurrence, coverage

In [ ]:
aoi = gpd.read_file(AOI_PATH)
MAX_PLOT_DIM = 1800

with rasterio.open(EVENTS['EVENT001']['flood_mask']) as ref:
    scale = max(ref.width / MAX_PLOT_DIM, ref.height / MAX_PLOT_DIM, 1)
    plot_w = max(1, int(round(ref.width / scale)))
    plot_h = max(1, int(round(ref.height / scale)))


def raster_extent(src):
    return [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

fig, axes = plt.subplots(2, 3, figsize=(12.5, 9.0))
ordered = ['EVENT001', 'EVENT002', 'EVENT003', 'EVENT004']
letters = ['A', 'B', 'C', 'D']

mask_cmap = ListedColormap(['#f2f2f2', '#1764ab'])
mask_cmap.set_bad('#bdbdbd')
mask_norm = BoundaryNorm([-0.5, 0.5, 1.5], mask_cmap.N)

for i, eid in enumerate(ordered):
    ax = axes.flat[i]
    spec = EVENTS[eid]
    with rasterio.open(spec['flood_mask']) as fs, rasterio.open(spec['valid_mask']) as vs:
        flood = fs.read(1, out_shape=(plot_h, plot_w), resampling=Resampling.nearest)
        valid = vs.read(1, out_shape=(plot_h, plot_w), resampling=Resampling.nearest)
        arr = np.ma.array(flood, mask=(valid != 1))
        ax.imshow(
            arr,
            extent=raster_extent(fs),
            origin='upper',
            cmap=mask_cmap,
            norm=mask_norm,
            interpolation='nearest',
        )
        aoi.to_crs(fs.crs).boundary.plot(ax=ax, linewidth=0.45, edgecolor='black')

    frac = float(event_state.loc[event_state['event_id'].eq(eid), 'flood_fraction'].iloc[0])
    ax.set_title(f"{letters[i]}. {eid} | {spec['date']}\nFlood fraction = {frac:.3f}", fontsize=9)
    ax.tick_params(labelsize=7)

ax = axes.flat[4]
with rasterio.open(CORR_RECURRENCE) as src:
    rec = src.read(1, out_shape=(plot_h, plot_w), masked=True, resampling=Resampling.average)
    im = ax.imshow(rec, extent=raster_extent(src), origin='upper', vmin=0, vmax=1, interpolation='nearest')
    aoi.to_crs(src.crs).boundary.plot(ax=ax, linewidth=0.45, edgecolor='black')
ax.set_title('E. Valid-observation-normalized recurrence', fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

ax = axes.flat[5]
with rasterio.open(CORR_OBS_COUNT) as src:
    obs = src.read(1, out_shape=(plot_h, plot_w), masked=True, resampling=Resampling.nearest)
    obs = np.ma.masked_where(obs == 0, obs)
    im2 = ax.imshow(obs, extent=raster_extent(src), origin='upper', vmin=1, vmax=4, interpolation='nearest')
    aoi.to_crs(src.crs).boundary.plot(ax=ax, linewidth=0.45, edgecolor='black')
ax.set_title('F. Valid event-observation count', fontsize=9)
fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.03, ticks=[1, 2, 3, 4])

for ax in axes.flat:
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle('Corrected Sentinel-1 event masks, recurrence, and observation coverage', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97])

FIG4_PNG = FIG_OUT / 'Figure4_CORRECTED.png'
FIG4_PDF = FIG_OUT / 'Figure4_CORRECTED.pdf'
fig.savefig(FIG4_PNG, dpi=FIG_DPI, bbox_inches='tight')
fig.savefig(FIG4_PDF, bbox_inches='tight')
plt.show()
print('Saved:', FIG4_PNG)

# 15. Regenerate Figure 5 — corrected statistical summary

In [ ]:
t1_rr = rr_table.loc[rr_table['threshold_m'].eq(1.0)].iloc[0]
p_v = t1_rr['VRFSZ_flood'] / t1_rr['VRFSZ_obs']
p_s = t1_rr['StableHigh_flood'] / t1_rr['StableHigh_obs']
thr = primary_bootstrap.sort_values('threshold_m').copy()
evt = event_rr.loc[event_rr['threshold_m'].eq(1.0)].sort_values('year').copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))

axes[0].bar(['VRFSZ', 'Stable High'], [p_v, p_s])
axes[0].set_ylabel('P(SAR flood detection | valid observation)')
axes[0].set_title('A. Primary-threshold class contrast')
axes[0].set_ylim(0, max(p_v, p_s) * 1.25)

x = np.arange(len(thr))
y = thr['point_rr'].to_numpy()
lo = y - thr['ci_low'].to_numpy()
hi = thr['ci_high'].to_numpy() - y
axes[1].errorbar(x, y, yerr=[lo, hi], fmt='o', capsize=4)
axes[1].axhline(1.0, linestyle='--', linewidth=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"T={t:g} m" for t in thr['threshold_m']])
axes[1].set_ylabel('Relative risk')
axes[1].set_title('B. Corrected threshold sensitivity')

x2 = np.arange(len(evt))
axes[2].scatter(x2, evt['event_RR_descriptive'], s=55)
axes[2].axhline(1.0, linestyle='--', linewidth=1)
axes[2].set_xticks(x2)
axes[2].set_xticklabels([str(y) for y in evt['year']])
axes[2].set_ylabel('Descriptive event-level RR')
axes[2].set_title('C. Event heterogeneity at T=1 m')

fig.suptitle('Corrected VRFSZ–SAR flood-detection statistical summary', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.95])

FIG5_PNG = FIG_OUT / 'Figure5_CORRECTED.png'
FIG5_PDF = FIG_OUT / 'Figure5_CORRECTED.pdf'
fig.savefig(FIG5_PNG, dpi=FIG_DPI, bbox_inches='tight')
fig.savefig(FIG5_PDF, bbox_inches='tight')
plt.show()
print('Saved:', FIG5_PNG)

# 16. Regenerate Figure S2 — corrected semivariogram + block robustness

In [ ]:
t1_robust = robustness_df.loc[robustness_df['threshold_m'].eq(1.0)].copy()
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0))

axes[0].plot(extended_variogram['distance_km'], extended_variogram['semivariance'])
axes[0].axhline(SILL_FRACTION * sill, linestyle='--', label=f'{100*SILL_FRACTION:.0f}% of variance')
axes[0].axvline(ORIGINAL_BLOCK_KM, linestyle=':', label='Original 80 km block')
if range_resolved:
    axes[0].axvline(
        empirical_range_km,
        linestyle='-.',
        label=f'Corrected empirical range ~{empirical_range_km:.1f} km'
    )
axes[0].set_xlabel('Lag distance (km)')
axes[0].set_ylabel('Empirical semivariance')
axes[0].set_title('A. Corrected recurrence semivariogram')
axes[0].legend(fontsize=8)

offset_labels = {
    (0.0, 0.0): 'origin',
    (0.5, 0.0): 'half-x',
    (0.0, 0.5): 'half-y',
    (0.5, 0.5): 'half-x+y',
}

for (ox, oy), g in t1_robust.groupby(['offset_x_fraction', 'offset_y_fraction']):
    g = g.sort_values('block_km')
    y = g['reconstructed_point_RR'].to_numpy()
    low = g['CI_low'].to_numpy()
    high = g['CI_high'].to_numpy()
    axes[1].errorbar(
        g['block_km'], y, yerr=[y - low, high - y], marker='o', capsize=3,
        label=offset_labels[(ox, oy)]
    )

axes[1].axhline(1.0, linestyle='--', linewidth=1)
axes[1].set_xlabel('Block width (km)')
axes[1].set_ylabel('Corrected T=1 relative risk')
axes[1].set_title('B. Block-width / origin robustness')
axes[1].legend(fontsize=8)

fig.tight_layout()
FIGS2_PNG = FIG_OUT / 'FigureS2_CORRECTED.png'
FIGS2_PDF = FIG_OUT / 'FigureS2_CORRECTED.pdf'
fig.savefig(FIGS2_PNG, dpi=FIG_DPI, bbox_inches='tight')
fig.savefig(FIGS2_PDF, bbox_inches='tight')
plt.show()
print('Saved:', FIGS2_PNG)

# 17. Patch the SAR provenance record

This records the previously missing operational fields without rerunning SAR classification.

Important nodata distinction: provider/source SAR rasters in this project have not always used one universal sentinel value. Therefore the provenance record states the actual validity rule — finite/non-nodata inputs are required — and notes that values such as `-9999` are excluded **where present**, rather than falsely asserting that every source raster uses `-9999`.

In [ ]:
frozen_cfg = {}
if FROZEN_SAR_CONFIG.exists():
    frozen_cfg = json.loads(FROZEN_SAR_CONFIG.read_text(encoding='utf-8'))

with rasterio.open(EVENTS['EVENT001']['flood_mask']) as ref:
    sar_grid = {
        'crs': str(ref.crs),
        'resolution_x_m': abs(float(ref.transform.a)),
        'resolution_y_m': abs(float(ref.transform.e)),
        'width': ref.width,
        'height': ref.height,
        'final_mask_dtype': ref.dtypes[0],
        'final_mask_nodata': ref.nodata,
    }

sar_provenance_patch = {
    'algorithm': frozen_cfg.get('algorithm', 'dual_polarization_db_change_otsu'),
    'vv_weight': frozen_cfg.get('vv_weight', 0.5),
    'vh_weight': frozen_cfg.get('vh_weight', 0.5),
    'threshold_strategy': frozen_cfg.get('threshold_strategy', 'event_specific_otsu_same_algorithm'),
    'histogram_bins': frozen_cfg.get('histogram_bins', 512),
    'permanent_water_rule': frozen_cfg.get('permanent_water_rule', 'JRC permanent water excluded before thresholding'),
    'uses_dem_or_vrfsz': frozen_cfg.get('uses_dem_or_vrfsz', False),
    'explicit_score_equation': 'score = 0.5*(VV_event_dB - VV_pre_dB) + 0.5*(VH_event_dB - VH_pre_dB)',
    'explicit_threshold_direction': 'flood = valid_land & (score <= event_specific_Otsu_threshold)',
    'input_validity_rule': 'finite pre/event VV and VH required; provider/export nodata (including -9999 where present) and other non-finite invalid values are excluded; permanent water is excluded',
    'JRC_transfer_resampling': 'nearest-neighbor categorical transfer',
    'terrain_class_transfer_resampling': 'nearest-neighbor categorical transfer',
    'final_flood_mask_encoding': 'uint8; 1=flood, 0=not-flood/outside-invalid; nodata=0 where encoded in the final mask product',
    'final_valid_mask_encoding': 'uint8; 1=valid observation, 0=invalid/no observation; nodata=0 where encoded in the final mask product',
    'analysis_grid': sar_grid,
    'config_sha256': frozen_cfg.get('config_sha256'),
}

(PROV_OUT / 'SAR_PROVENANCE_PATCH.json').write_text(
    json.dumps(sar_provenance_patch, indent=2, default=str), encoding='utf-8'
)

prov_table = pd.DataFrame([
    {'field': k, 'value': v}
    for k, v in sar_provenance_patch.items()
    if not isinstance(v, dict)
])
prov_table.to_csv(PROV_OUT / 'SAR_PROVENANCE_PATCH.csv', index=False)
display(prov_table)
print('SAR provenance patch saved.')

# 18. Conservative DeltaDTM interpretation branch

In [ ]:
deltadtm_text = (
    'Native-product provenance and interpretation boundary.\n\n'
    'The paired terrain comparison uses a native DeltaDTM representation obtained from a '
    'third-party Earth Engine mirror and an MSL/MDT-aligned DeltaDTM representation on an '
    'exactly matched source grid. Exact grid correspondence, common-valid footprint, and '
    'cellwise differencing were verified. However, those checks do not by themselves prove '
    'that every mirrored native cell value is identical to the official DeltaDTM v1.1 release. '
    'Accordingly, this study treats the terrain analysis as a paired-product comparison and '
    'does not claim that the observed elevation difference isolates only the vertical-reference '
    'transformation. The screening result therefore quantifies the practical classification '
    'difference between the two paired terrain representations used here.\n\n'
    'Native-elevation confounding.\n\n'
    'VRFSZ and Stable High are defined relative to the screening threshold and are therefore '
    'not balanced in their native-elevation distributions. The relative-risk statistic is an '
    'observational contrast in SAR flood-detection occurrence conditional on the selected '
    'events and terrain classes; it is not interpreted as a causal effect of vertical-reference '
    'treatment on flooding.'
)

(TEXT_OUT / 'DeltaDTM_and_confounding_required_wording.txt').write_text(
    deltadtm_text, encoding='utf-8'
)
print(deltadtm_text)

# 19. GFDS-only corroboration wording — only if the completed local GFDS table exists

In [ ]:
GFDS_CANDIDATES = [
    PROJECT_ROOT / 'validation_targeted_multisource_28' / 'outputs' / '04_gfds_sar_local_concordance.csv',
    PROJECT_ROOT / 'reports' / 'validation_targeted_multisource_28' / 'outputs' / '04_gfds_sar_local_concordance.csv',
]

gfds_path = next((p for p in GFDS_CANDIDATES if p.exists()), None)

if gfds_path is not None:
    gfds_df = pd.read_csv(gfds_path)
    display(gfds_df)
    gfds_wording = (
        'Independent external corroboration was restricted to the GDACS/JRC Global Flood '
        'Detection System (GFDS), which provides coarse passive-microwave flood-magnitude '
        'anomalies independent of Sentinel-1. GFDS hotspots were identified before comparison '
        'with the SAR-derived flooded fraction. Because GFDS is substantially coarser than the '
        '10 m Sentinel-1 masks and represents microwave flood magnitude rather than a '
        'high-resolution inundation boundary, it was used only as coarse independent '
        'corroboration. No precision, recall, F1, IoU, or 10 m confusion-matrix accuracy was '
        'derived from GFDS.'
    )
else:
    gfds_df = pd.DataFrame()
    gfds_wording = (
        'No external dataset was promoted to pixel-level ground truth in the final analysis. '
        'Candidate external products that did not yield analysis-ready local observations were '
        'excluded rather than converted into synthetic validation data. The manuscript therefore '
        'reports no conventional sensor-independent 10 m confusion matrix.'
    )

(TEXT_OUT / 'GFDS_validation_wording.txt').write_text(gfds_wording, encoding='utf-8')
print(gfds_wording)

# 20. Generate manuscript-ready corrected numerical text

In [ ]:
def fmt_ci(row):
    return f"{row['ci_low']:.4f}–{row['ci_high']:.4f}"

primary_lines = []
for _, r in primary_bootstrap.sort_values('threshold_m').iterrows():
    primary_lines.append(
        f"T={r['threshold_m']:.1f} m: RR={r['point_rr']:.4f}, "
        f"95% spatial block-bootstrap CI={fmt_ci(r)}, active blocks={int(r['n_blocks'])}."
    )

evt_t1 = event_rr.loc[event_rr['threshold_m'].eq(1.0)].sort_values('event_id')
event_lines = []
for _, r in evt_t1.iterrows():
    event_lines.append(
        f"{r['event_id']} ({int(r['year'])}): VRFSZ {int(r['VRFSZ_flood']):,}/{int(r['VRFSZ_obs']):,}; "
        f"Stable High {int(r['StableHigh_flood']):,}/{int(r['StableHigh_obs']):,}; "
        f"descriptive RR={r['event_RR_descriptive']:.4f}."
    )

rob_t1 = robust_summary.loc[robust_summary['threshold_m'].eq(1.0)].copy()
usable_widths = rob_t1.loc[rob_t1['usable_origins'] > 0, 'block_km'].astype(int).tolist()
all_include = bool(
    rob_t1.loc[rob_t1['usable_origins'] > 0, 'all_usable_CIs_include_1'].fillna(False).all()
)

range_sentence = (
    f'The corrected empirical semivariogram resolved a practical range of approximately {empirical_range_km:.2f} km.'
    if range_resolved
    else 'The corrected empirical semivariogram did not resolve a finite practical range.'
)

final_stats_text = (
    'Corrected relative-risk results.\n\n'
    + '\n'.join(primary_lines)
    + '\n\nAt the primary T=1 m threshold, event-level contrasts were heterogeneous:\n'
    + '\n'.join(event_lines)
    + '\n\nThe corrected pooled point estimate at T=1 m is below unity and its 95% spatial '
      'block-bootstrap interval includes 1. The four-event evidence therefore provides '
      'NO SUPPORT for the predeclared positive VRFSZ–SAR flood-detection association. '
      'This is not interpreted as proof of no association or as evidence of a causal negative effect.\n\n'
      'Spatial robustness.\n\n'
      f"The original 1-km lattice and active-block rule were restored. The corrected 80-km/original-origin "
      f"implementation retained {int(primary_t1['n_blocks'])} active blocks. Extended robustness was evaluated "
      f"at block widths {BLOCK_WIDTHS_KM} km across four fixed origins with {ROBUSTNESS_REPLICATES:,} bootstrap "
      f"replicates per configuration. Inferentially usable widths at T=1 m were {usable_widths}. "
      + ('All inferentially usable corrected T=1 intervals included 1. ' if all_include else 'The usable corrected T=1 intervals were not qualitatively uniform. ')
      + range_sentence
)

(TEXT_OUT / 'Corrected_results_and_robustness_text.txt').write_text(
    final_stats_text, encoding='utf-8'
)
print(final_stats_text)

# 24. Final SHA-256 correction freeze

In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


freeze_files = []
for eid, spec in EVENTS.items():
    freeze_files.extend([spec['flood_mask'], spec['valid_mask']])
freeze_files.extend([FROZEN_SAR_CONFIG, *CLASS_FILES.values()])

for root in [DATA_OUT, FIG_OUT, TEXT_OUT, PROV_OUT, SUBMISSION_OUT, DOCX_OUT]:
    if root.exists():
        freeze_files.extend([p for p in root.rglob('*') if p.is_file()])

rows = []
for p in tqdm(sorted(set(Path(x).resolve() for x in freeze_files if Path(x).exists())), desc='Final correction SHA-256 freeze'):
    rows.append({
        'path': str(p),
        'size_MB': p.stat().st_size / 1024**2,
        'sha256': sha256_file(p),
        'modified': datetime.fromtimestamp(p.stat().st_mtime).isoformat(),
    })

freeze_manifest = pd.DataFrame(rows)
freeze_manifest.to_csv(FREEZE_OUT / 'CORRECTION_FINAL_SHA256_MANIFEST.csv', index=False)

manifest = {
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(PROJECT_ROOT),
    'corrected_EVENT003_fraction': e3,
    'corrected_RR': rr_table.to_dict(orient='records'),
    'corrected_primary_bootstrap': primary_bootstrap.to_dict(orient='records'),
    'original_block_structure': structure_df.to_dict(orient='records'),
    'corrected_range_diagnostic': range_diag,
    'manuscript_source': str(MANUSCRIPT_PATH) if MANUSCRIPT_PATH else None,
    'manuscript_output': str(output_docx) if output_docx else None,
    'submission_metadata': SUBMISSION_METADATA,
    'external_actions_still_required': [
        'Review redlined DOCX and patch log',
        'Convert author-year citations to journal numbered style using the final reference manager',
        'Create/confirm GitHub repository and release',
        'Create Zenodo DOI if desired',
        'Fill repository URL / DOI / license',
        'Confirm CRediT / funding / competing interests',
        'Use journal-compliant AI disclosure wording',
    ],
}
(FREEZE_OUT / 'CORRECTION_FINAL_MANIFEST.json').write_text(
    json.dumps(manifest, indent=2, default=str), encoding='utf-8'
)

display(freeze_manifest.head(100))
print('Correction freeze complete.')

# 25. Final science gate

The local correction passes only if the corrected EVENT003 state, corrected pooled RR, original block structure, corrected primary bootstrap, corrected robustness, regenerated figures, provenance patch, and final hash manifest all exist. Submission readiness is kept separate because GitHub/Zenodo/declarations require author/external actions.

In [ ]:
science_gate = {
    'corrected_EVENT003': abs(e3 - EXPECTED_EVENT003_FRACTION) <= EVENT003_TOL,
    'RR_reconstruction': bool((rr_table['absolute_difference'] <= RR_TOL).all()),
    'original_active_blocks_26': bool((structure_df['active_blocks'] == 26).all()),
    'original_structure_exact': bool(structure_df['active_match'].all() and structure_df['both_match'].all()),
    'primary_bootstrap_created': (DATA_OUT / '04_corrected_primary_80km_bootstrap.csv').exists(),
    'robustness_created': (DATA_OUT / '05_corrected_block_size_and_origin_robustness.csv').exists(),
    'Figure4_created': FIG4_PNG.exists(),
    'Figure5_created': FIG5_PNG.exists(),
    'FigureS2_created': FIGS2_PNG.exists(),
    'SAR_provenance_patch_created': (PROV_OUT / 'SAR_PROVENANCE_PATCH.json').exists(),
    'hash_manifest_created': (FREEZE_OUT / 'CORRECTION_FINAL_SHA256_MANIFEST.csv').exists(),
}

science_gate_df = pd.DataFrame([{'check': k, 'pass': bool(v)} for k, v in science_gate.items()])
display(science_gate_df)
science_gate_df.to_csv(DATA_OUT / '12_FINAL_SCIENCE_GATE.csv', index=False)

if not science_gate_df['pass'].all():
    raise RuntimeError('FINAL CORRECTION SCIENCE GATE FAILED. Inspect 12_FINAL_SCIENCE_GATE.csv.')

submission_complete = all(bool(str(v).strip()) for v in [
    SUBMISSION_METADATA['github_repository_url'],
    SUBMISSION_METADATA['license'],
    SUBMISSION_METADATA['credit_statement'],
    SUBMISSION_METADATA['ai_disclosure'],
])

print('✅ LOCAL SCIENTIFIC CORRECTION: PASS')
if submission_complete:
    print('✅ AUTHOR-SUPPLIED SUBMISSION METADATA: PRESENT')
else:
    print('⚠️ SUBMISSION EXTERNAL/AUTHOR ACTIONS REMAIN — expected until repository/declaration fields are filled.')

print(
    '\nFinal scientific interpretation:\n'
    'The corrected four-event evidence provides NO SUPPORT for the predeclared positive '
    'VRFSZ–SAR flood-detection association. The manuscript must retain observational, '
    'paired-product, and validation limitations.'
)

# END — Repository scientific freeze

A successful final science gate means the repository reproduces the corrected local
scientific state. It does not create a remote DOI and does not prove official native-mirror
cell-value equivalence.